# 🔥 SAFE-FIRE MASTER

دفتر العمل الرئيسي لبناء نظام كشف حريق خفيف وموثوق، من إعداد البيانات حتى التصدير.

**مسار التنفيذ:** `S → A → B → C → D → E → F → X → التحقق الخارجي → G → Export`


## S — إعداد الجلسة

تهيئة البيئة والمسارات والدوال المشتركة. تُشغَّل `S1` في بداية كل جلسة.


In [ ]:
import os, sys, json, time, shutil, zipfile, random, hashlib, subprocess, math
from pathlib import Path
from datetime import datetime

from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics==8.4.90", "pandas", "pyyaml", "pillow"], check=False)
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)
    import ultralytics
import numpy as np
import pandas as pd
import yaml as _yaml
import torch
from ultralytics import YOLO

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


ROOT       = Path("/content/drive/MyDrive/SAFE_FIRE")
D_DATA     = ROOT/"datasets"
D_SUITE    = ROOT/"eval_suite_v2"
D_TRAINNEG = ROOT/"train_neg_v2"
D_WEIGHTS  = ROOT/"weights"
D_RESULTS  = ROOT/"results"
D_SCORES   = ROOT/"results/scores_v2"
D_SWEEPS   = ROOT/"results/sweeps_v2"
D_RUNS     = ROOT/"runs"
D_SM       = ROOT/"safemine_v2"
D_FIGS     = ROOT/"figures_v2"
D_LOGS     = ROOT/"logs"
for d in (ROOT, D_DATA, D_SUITE, D_TRAINNEG, D_WEIGHTS, D_RESULTS, D_SCORES,
          D_SWEEPS, D_RUNS, D_SM, D_SM/"mining", D_SM/"manifests", D_SM/"figures",
          D_FIGS, D_LOGS):
    d.mkdir(parents=True, exist_ok=True)

MASTER_CSV = D_RESULTS/"runs_master_v2.csv"
DECISIONS  = D_RESULTS/"decisions.json"

LOCAL   = Path("/content/sf")
L_DS    = LOCAL/"datasets"
L_EVAL  = LOCAL/"eval"
L_RUNS  = LOCAL/"runs"
L_TMP   = LOCAL/"tmp"
L_HITS  = LOCAL/"hits"
L_CORP  = LOCAL/"corpus"
for d in (LOCAL, L_DS, L_EVAL, L_RUNS, L_TMP, L_HITS, L_CORP):
    d.mkdir(parents=True, exist_ok=True)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
SEEDS   = (42, 123, 2024)
THRS    = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]


RECIPE = dict(
    imgsz=640, batch=32, epochs=300, patience=50,
    optimizer="AdamW", lr0=0.001, lrf=0.01, weight_decay=0.05,
    warmup_epochs=3.0, cos_lr=True,
    mosaic=1.0, mixup=0.0, copy_paste=0.1, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.0, erasing=0.4,
    device=0, workers=8, amp=True, cache="ram",
)

MASTER_COLS = ["run_id", "group", "name", "model", "seed", "data", "lam", "focal",
    "imgsz", "timestamp", "epochs", "best_epoch", "train_min", "params_M", "GFLOPs",
    "mAP50", "mAP5095", "precision", "recall", "f1",
    "flame_AP50", "smoke_AP50", "flame_AP5095", "smoke_AP5095",
    "FPR25_hn", "FPR50_hn", "FPR25_novel", "FPR50_novel",
    "FPR25_dfire", "FPR50_dfire", "FPR25_mined", "FPR50_mined",
    "recall_at_fpr1", "weights", "notes"]


def header(t, c="="):
    print(); print(c*70); print("  " + t); print(c*70)

def robust_extract(zip_path, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    entries = [p for p in dest.iterdir() if not p.name.startswith(".")]
    if len(entries) == 1 and entries[0].is_dir():
        return entries[0]
    return dest

def count_imgs(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in IMG_EXT)

def imgs_in(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)

def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def zip_flat(paths, zpath, arc_prefix=""):

    zpath = Path(zpath)
    tmp = L_TMP/(zpath.name + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for p in paths:
            z.write(p, arc_prefix + Path(p).name)
    shutil.move(str(tmp), str(zpath))
    return zpath

def zip_count(zpath):
    with zipfile.ZipFile(zpath) as z:
        return sum(1 for n in z.namelist() if n.lower().endswith(IMG_EXT))

def save_decision(key, value):
    d = json.loads(DECISIONS.read_text()) if DECISIONS.exists() else {}
    d[key] = value
    d["_updated"] = datetime.now().isoformat(timespec="seconds")
    DECISIONS.write_text(json.dumps(d, indent=2, ensure_ascii=False))
    print("  📌 قرار محفوظ:", key, "=", value)

def get_decision(key, default=None):
    if not DECISIONS.exists():
        return default
    return json.loads(DECISIONS.read_text()).get(key, default)

def ensure_home_fire_local():
    hf = L_DS/"home_fire"
    if not (hf/"train"/"images").exists():
        zp = D_DATA/"home_fire.zip"
        assert zp.exists(), "home_fire.zip غير موجود — شغّلي A1 أولاً"
        print("  فك home_fire.zip ...")
        root = robust_extract(zp, hf)
        if root != hf:
            for item in list(root.iterdir()):
                shutil.move(str(item), str(hf/item.name))
            root.rmdir()
        assert (hf/"train"/"images").exists(), "بنية home_fire غير متوقعة!"
    dy = hf/"data.yaml"
    dy.write_text(
        "path: " + str(hf) + "\n" +
        "train: train/images\nval: val/images\ntest: test/images\n" +
        "names:\n  0: flame\n  1: smoke\nnc: 2\n")
    return str(dy)

NEG_ZIPS = {
    "hn_holdout": D_SUITE/"hn_holdout_500.zip",
    "novel":      D_SUITE/"novel_firelike.zip",
    "dfire_neg":  D_SUITE/"dfire_neg_1500.zip",
    "mined_dev":  D_SUITE/"mined_dev.zip",
}
CORE_NEG = ["hn_holdout", "novel", "dfire_neg"]

def ensure_suite_local(strict=True):

    out = {}
    for name, zp in NEG_ZIPS.items():
        if not zp.exists():
            if strict and name in CORE_NEG:
                raise AssertionError(name + " غير مبني — شغّلي المرحلة A أولاً (" + str(zp) + ")")
            continue
        d = L_EVAL/name
        if count_imgs(d) == 0:
            robust_extract(zp, d)
        out[name] = d
    return out

def scan_max_conf(model, folder, out_csv):
    out_csv = Path(out_csv)
    if out_csv.exists():
        return pd.read_csv(out_csv)["max_conf"].to_numpy()
    imgs = imgs_in(folder)
    rows, B = [], 64
    for i in range(0, len(imgs), B):
        chunk = [str(p) for p in imgs[i:i+B]]
        res = model.predict(chunk, conf=0.01, imgsz=640, verbose=False, save=False)
        for src, r in zip(chunk, res):
            mc = float(r.boxes.conf.max().item()) if len(r.boxes) else 0.0
            rows.append((Path(src).name, mc))
    pd.DataFrame(rows, columns=["image", "max_conf"]).to_csv(out_csv, index=False)
    return np.array([r[1] for r in rows])

def append_master(row):
    df_new = pd.DataFrame([{c: row.get(c, "") for c in MASTER_COLS}])
    if MASTER_CSV.exists():
        df = pd.concat([pd.read_csv(MASTER_CSV), df_new], ignore_index=True)
    else:
        df = df_new
    tmp = MASTER_CSV.with_suffix(".csv.tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(MASTER_CSV)

def already_done(run_id):
    if not MASTER_CSV.exists():
        return False
    df = pd.read_csv(MASTER_CSV)
    if df.empty or "run_id" not in df.columns:
        return False
    hit = df[df["run_id"] == run_id]
    if hit.empty:
        return False
    v = hit["mAP50"].iloc[-1]
    return pd.notna(v) and str(v).strip() != ""

def master_df(group=None):
    assert MASTER_CSV.exists(), "لا توجد نتائج بعد"
    df = pd.read_csv(MASTER_CSV).drop_duplicates(subset="run_id", keep="last")
    if group:
        df = df[df["group"] == group].copy()
    for c in ["mAP50", "mAP5095", "precision", "recall", "f1", "recall_at_fpr1",
              "FPR25_hn", "FPR50_hn", "FPR50_novel", "FPR50_dfire", "FPR50_mined",
              "params_M", "GFLOPs"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def build_neg_dataset(tag, neg_items):


    hf_yaml = ensure_home_fire_local()
    hf = Path(hf_yaml).parent
    root = L_DS/("ds_" + tag)
    if root.exists():
        shutil.rmtree(root)
    (root/"images"/"extra").mkdir(parents=True)
    (root/"labels"/"extra").mkdir(parents=True)
    n = 0
    for src, lbl in neg_items:
        src = Path(src)
        if not src.exists():
            continue
        shutil.copy2(src, root/"images"/"extra"/src.name)
        (root/"labels"/"extra"/(src.stem + ".txt")).write_text(lbl)
        n += 1
    dy = dict(train=[str(hf/"train"/"images"), str(root/"images"/"extra")],
              val=str(hf/"val"/"images"), test=str(hf/"test"/"images"),
              names={0: "flame", 1: "smoke"}, nc=2)
    yp = root/"data.yaml"
    yp.write_text(_yaml.safe_dump(dy, sort_keys=False))
    print("  ✅ داتاسيت %s: %d صورة إضافية → %s" % (tag, n, yp))
    return str(yp)

header("S1 جاهزة ✅")
print("  ultralytics :", ultralytics.__version__, "| torch:", torch.__version__)
print("  GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ فعّلي GPU!")
print("  ROOT        :", ROOT)
print("  runs_master :", "موجود (%d صف)" % len(pd.read_csv(MASTER_CSV)) if MASTER_CSV.exists() else "جديد")
print("  decisions   :", json.loads(DECISIONS.read_text()) if DECISIONS.exists() else "{}")


Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

  S1 جاهزة ✅
  ultralytics : 8.4.90 | torch: 2.11.0+cu128
  GPU         : NVIDIA L4
  ROOT        : /content/drive/MyDrive/SAFE_FIRE
  runs_master : موجود (48 صف)
  decisions   : {'winner': 'yolo26n', '_updated': '2026-07-19T05:03:04', 'teacher_model': 'yolo26s', 'ratio_counts': {'hn15': 688, 'hn30': 1638, 'hn45': 1638, 'hn30rand': 1638}, 'best_ratio': 'hn30', 'neg_budget': 1638, 'kmeans_K': 16, 'lbg_sanity_ok': True, 'best_sm': 'sm_top', 'best_res': 640, 'hero_neg_source': 'hn30', 'hero': {'group': 'hero', 'name': 'combo', 'model': 'yolo26n', 'imgsz': 640}, 'cascade_ok': True, 'cascade_gate': [0.05, 0.7], 'cascade_fast': '/content/drive/MyDrive/SAFE_FIR

### YOLOv5s from-scratch ببروتوكول YOLO-HF

In [ ]:
import os, sys, json, time, shutil, zipfile, subprocess
from pathlib import Path

try:
    ROOT, D_DATA, D_WEIGHTS, D_RESULTS, L_DS, header, ensure_home_fire_local, robust_extract
except NameError:
    raise RuntimeError("شغّلي S1 قبل هذه الخلية")

header("E1.8 — استنساخ بروتوكول YOLO-HF ثم إضافة السوالب")

E18       = ROOT / "e1_8"
E18.mkdir(parents=True, exist_ok=True)
YOLOV5    = Path("/content/yolov5")
SEED      = 42
EPOCHS    = 300
PATIENCE  = 100
BATCH     = 32
IMGSZ     = 640
RESULTS   = E18 / "e1_8_results.csv"


# ------------------------------------------------------------------ ١. المستودع
if not (YOLOV5 / "train.py").exists():
    print("استنساخ YOLOv5 v7.0 (وسم مثبَّت — لا فرع متحرك) ...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "v7.0",
                    "https://github.com/ultralytics/yolov5", str(YOLOV5)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(YOLOV5 / "requirements.txt")], check=False)


# ------------------------------------------------------------- ١.٥ طبقة التوافق
#  YOLOv5 v7.0 صدر نوفمبر 2022. بيئة Colab اليوم Python 3.12 و numpy 2 و
#  torch ≥ 2.6، وثلاثة أشياء تغيّرت تحته:
#
#    • torch.load صار weights_only=True افتراضياً، و check_amp() في v7.0
#      يحمّل yolov5n.pt — فيموت قبل أن يبدأ التدريب أصلاً.
#    • numpy 2 حذف np.trapz، ويستعمله utils/metrics.py عند حساب mAP.
#    • np.bool / np.int / np.float محذوفة منذ numpy 1.24.
#
#  تُعالَج كلها في sitecustomize.py، الذي يستورده بايثون تلقائياً عند الإقلاع.
#
#  ⚠️ لكنه يُستورد من مسارات sys.path المهيّأة **قبل** تشغيل السكربت، ومجلد
#     yolov5 يُضاف بعدها. لذلك لا بد من تمرير PYTHONPATH صراحةً في env أدناه.
#     بدون ذلك يبقى الملف على القرص ولا يُنفَّذ أبداً — وهذا بالضبط سبب فشل
#     المحاولة الأولى.
(YOLOV5 / "sitecustomize.py").write_text(
    "import functools\n"
    "import numpy as np\n"
    "import torch\n"
    "_orig = torch.load\n"
    "@functools.wraps(_orig)\n"
    "def _load(*a, **k):\n"
    "    k.setdefault('weights_only', False)\n"
    "    return _orig(*a, **k)\n"
    "torch.load = _load\n"
    "torch.load._e18_patched = True\n"
    "if not hasattr(np, 'trapz'):  np.trapz = np.trapezoid\n"
    "if not hasattr(np, 'bool'):   np.bool = bool\n"
    "if not hasattr(np, 'int'):    np.int = int\n"
    "if not hasattr(np, 'float'):  np.float = float\n")

ENV = dict(os.environ)
ENV["PYTHONPATH"] = str(YOLOV5) + os.pathsep + ENV.get("PYTHONPATH", "")

_chk = subprocess.run(
    [sys.executable, "-c",
     "import torch, numpy as np;"
     "print('torch.load patched :', getattr(torch.load, '_e18_patched', False));"
     "print('np.trapz available :', hasattr(np, 'trapz'));"
     "print('numpy', np.__version__, '| torch', torch.__version__)"],
    capture_output=True, text=True, cwd=str(YOLOV5), env=ENV)
print(_chk.stdout.strip() or _chk.stderr.strip()[-500:])
if "True" not in _chk.stdout:
    raise RuntimeError("طبقة التوافق لم تُحمَّل")


# --------------------------------------------- ٢. تحقّق من المعمارية قبل أي تدريب
def check_params():
    r = subprocess.run(
        [sys.executable, "-c",
         "from models.yolo import Model;"
         "m = Model('models/yolov5s.yaml', ch=3, nc=2);"
         "print('PARAMS', sum(p.numel() for p in m.parameters()))"],
        capture_output=True, text=True, cwd=str(YOLOV5), env=ENV)
    n = None
    for ln in r.stdout.splitlines():
        if ln.startswith("PARAMS"):
            n = int(ln.split()[1])
    if n is None:
        print(r.stdout[-900:]); print(r.stderr[-900:])
        raise RuntimeError("فشل بناء النموذج")
    print("  YOLOv5s @ nc=2 : %.3f مليون معامل )" % (n / 1e6))
    if abs(n / 1e6 - 7.02) > 0.15:
        print("  ⚠️ ")
    return n


check_params()


# ------------------------------------------------- ٣. المجموعتان: بلا سوالب / بها
hf_yaml = ensure_home_fire_local()
hf = Path(hf_yaml).parent
print("  home_fire :", hf)

# (أ) إيجابيات فقط — مجموعة الورقة كما هي، بلا أي تدخّل
ds_pos = E18 / "ds_pos"
ds_pos.mkdir(exist_ok=True)
(ds_pos / "data.yaml").write_text(
    "train: %s\nval: %s\ntest: %s\nnc: 2\nnames: [flame, smoke]\n"
    % (hf / "train" / "images", hf / "val" / "images", hf / "test" / "images"))

# (ب) نفسها + سوالب SAFE-Mine المنتقاة في الجولة صفر
sel = json.loads((D_DATA / "sm_manifest__round0_selected.json").read_text())
blk = set(json.loads((D_DATA / "sm_manifest__fire_block.json").read_text()))
keep = [s for s in sel if s not in blk]
print("  سوالب مختارة: %d ، محجوبة كنار مسرَّبة: %d ، صافي: %d"
      % (len(sel), len(sel) - len(keep), len(keep)))

ds_neg = E18 / "ds_combo"
neg_img = ds_neg / "images" / "extra"
neg_lbl = ds_neg / "labels" / "extra"
if not neg_img.exists() or len(list(neg_img.glob("*.jpg"))) < len(keep) * 0.9:
    shutil.rmtree(ds_neg, ignore_errors=True)
    neg_img.mkdir(parents=True); neg_lbl.mkdir(parents=True)
    hits = D_DATA / "sm_round0_hits.zip"
    assert hits.exists(), "sm_round0_hits.zip مفقود: " + str(hits)
    want = set(keep)
    n = 0
    with zipfile.ZipFile(hits) as z:
        for m in z.namelist():
            b = os.path.basename(m)
            if b in want:
                with z.open(m) as fsrc, open(neg_img / b, "wb") as fdst:
                    shutil.copyfileobj(fsrc, fdst)
                (neg_lbl / (Path(b).stem + ".txt")).write_text("")   # خلفية = تسمية فارغة
                n += 1
    print("  استُخرجت %d صورة سالبة" % n)
(ds_neg / "data.yaml").write_text(
    "train:\n  - %s\n  - %s\nval: %s\ntest: %s\nnc: 2\nnames: [flame, smoke]\n"
    % (hf / "train" / "images", neg_img,
       hf / "val" / "images", hf / "test" / "images"))


# ------------------------------------------------------------------ ٤. التدريب
def train(tag, data_yaml, epochs, patience, sanity=False):
    name = ("sanity_" if sanity else "") + tag
    out = YOLOV5 / "runs" / "train" / name / "weights" / "best.pt"
    keep_to = D_WEIGHTS / ("e18__%s__s%d__best.pt" % (tag, SEED))
    if (not sanity) and keep_to.exists():
        print("  ⏭ %s موجود — تخطّي التدريب" % keep_to.name)
        return keep_to
    cmd = [sys.executable, str(YOLOV5 / "train.py"),
           "--img", str(IMGSZ), "--batch", str(BATCH), "--epochs", str(epochs),
           "--data", str(data_yaml),
           "--cfg", str(YOLOV5 / "models" / "yolov5s.yaml"),
           "--weights", "",                       # فارغ = من الصفر، كما الورقة
           "--hyp", str(YOLOV5 / "data" / "hyps" / "hyp.scratch-low.yaml"),
           "--patience", str(patience), "--seed", str(SEED),
           "--name", name, "--exist-ok", "--device", "0"]
    print("\n  $ " + " ".join(cmd[1:]) + "\n")
    t0 = time.time()


    r = subprocess.run(cmd, cwd=str(YOLOV5), env=ENV,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n" + "!" * 72)
        print("  فشل train.py ")
        print("!" * 72)
        for ln in (r.stdout + "\n" + r.stderr).strip().splitlines()[-40:]:
            print("   ", ln)
        print("!" * 72)
        raise RuntimeError("train.py رجع " % r.returncode)

    for ln in r.stdout.strip().splitlines()[-6:]:
        print("   ", ln)
    print("  ⏱ %.1f دقيقة" % ((time.time() - t0) / 60))
    assert out.exists(), "best.pt مفقود: " + str(out)
    if not sanity:
        shutil.copy2(out, keep_to)
        return keep_to
    return out

train("pos", ds_pos / "data.yaml", 2, 2, sanity=True)
train("combo", ds_neg / "data.yaml", 2, 2, sanity=True)
print("\n✅ .\n")

w_pos = train("pos", ds_pos / "data.yaml", EPOCHS, PATIENCE)
w_neg = train("combo", ds_neg / "data.yaml", EPOCHS, PATIENCE)


# ---------------------------------------------------------- ٥. التقييم: mAP
def eval_map(weights, data_yaml, tag):
    r = subprocess.run(
        [sys.executable, str(YOLOV5 / "val.py"), "--weights", str(weights),
         "--data", str(data_yaml), "--img", str(IMGSZ), "--task", "test",
         "--name", "val_" + tag, "--exist-ok", "--device", "0"],
        capture_output=True, text=True, cwd=str(YOLOV5), env=ENV)
    if r.returncode != 0:
        print("  ⚠️ val.py رجع %d:" % r.returncode)
        for ln in (r.stdout + "\n" + r.stderr).strip().splitlines()[-25:]:
            print("   ", ln)
        return {}
    print(r.stdout[-1600:])
    out = {}
    for ln in r.stdout.splitlines():
        p = ln.split()
        if len(p) >= 7 and p[0] == "all":
            out = dict(P=float(p[3]), R=float(p[4]),
                       mAP50=float(p[5]), mAP5095=float(p[6]))
    return out


# --------------------------------------- ٦. التقييم: FPR على المجموعات المجمّدة
def scan_fpr(weights, tag):
    import numpy as np, torch, cv2
    sys.path.insert(0, str(YOLOV5))
    from models.common import DetectMultiBackend
    from utils.augmentations import letterbox
    from utils.general import non_max_suppression

    dev = torch.device("cuda:0")
    model = DetectMultiBackend(str(weights), device=dev, dnn=False, fp16=False)
    model.warmup(imgsz=(1, 3, IMGSZ, IMGSZ))

    out = {}
    for zname, key in (("hn_holdout_500.zip", "hn_holdout"),
                       ("novel_firelike.zip", "novel"),
                       ("mined_dev.zip", "mined_dev"),
                       ("dfire_neg_1500.zip", "dfire_neg")):
        zp = ROOT / "eval_suite_v2" / zname
        if not zp.exists():
            print("  ⚠️", zp); continue
        z = zipfile.ZipFile(zp)
        names = [n for n in z.namelist() if n.lower().endswith((".jpg", ".png", ".jpeg"))]
        mx = []
        for i, n in enumerate(names):
            buf = np.frombuffer(z.read(n), np.uint8)
            im0 = cv2.imdecode(buf, cv2.IMREAD_COLOR)
            if im0 is None:
                continue
            im = letterbox(im0, IMGSZ, stride=model.stride, auto=False)[0]
            im = im.transpose((2, 0, 1))[::-1]
            im = torch.from_numpy(np.ascontiguousarray(im)).to(dev).float() / 255.0
            with torch.no_grad():
                pred = model(im[None])
            pred = non_max_suppression(pred, conf_thres=0.001, iou_thres=0.45)[0]
            mx.append(float(pred[:, 4].max()) if len(pred) else 0.0)
            if (i + 1) % 250 == 0:
                print("    %s %d/%d" % (key, i + 1, len(names)))
        a = np.array(mx)
        out["FPR25_" + key] = round(100.0 * float((a >= 0.25).mean()), 2)
        out["FPR50_" + key] = round(100.0 * float((a >= 0.50).mean()), 2)
        np.savetxt(str(E18 / ("scores_%s__%s.csv" % (tag, key))), a,
                   fmt="%.6f", header="max_conf", comments="")
        print("  %-12s %-10s FPR@0.50 = %5.2f %%" % (tag, key, out["FPR50_" + key]))
    return out


import pandas as pd

rows = []
for tag, w, dy in (("pos_only", w_pos, ds_pos / "data.yaml"),
                   ("plus_safemine", w_neg, ds_neg / "data.yaml")):
    header("تقييم: " + tag, "-")
    r = {"run": tag, "protocol": "YOLO-HF (scratch, SGD, 300ep)",
         "model": "yolov5s", "seed": SEED}
    r.update(eval_map(w, dy, tag))
    r.update(scan_fpr(w, tag))
    rows.append(r)

df = pd.DataFrame(rows)
df.to_csv(RESULTS, index=False)

print("\n" + "=" * 78)
print("  E1.8 — النتيجة")
print("=" * 78)
print(df.to_string(index=False))
print()
print("  حُفظ في:", RESULTS)


  E1.8 — استنساخ بروتوكول YOLO-HF ثم إضافة السوالب
torch.load patched : True
np.trapz available : True
numpy 2.0.2 | torch 2.11.0+cu128
  YOLOv5s @ nc=2 : 7.025 مليون معامل )
  home_fire : /content/sf/datasets/home_fire
  سوالب مختارة: 1671 ، محجوبة كنار مسرَّبة: 0 ، صافي: 1671

  $ /content/yolov5/train.py --img 640 --batch 32 --epochs 2 --data /content/drive/MyDrive/SAFE_FIRE/e1_8/ds_pos/data.yaml --cfg /content/yolov5/models/yolov5s.yaml --weights  --hyp /content/yolov5/data/hyps/hyp.scratch-low.yaml --patience 2 --seed 42 --name sanity_pos --exist-ok --device 0

    Command 'git rev-list HEAD..origin/master --count' returned non-zero exit status 128.
  ⏱ 2.1 دقيقة

  $ /content/yolov5/train.py --img 640 --batch 32 --epochs 2 --data /content/drive/MyDrive/SAFE_FIRE/e1_8/ds_combo/data.yaml --cfg /content/yolov5/models/yolov5s.yaml --weights  --hyp /content/yolov5/data/hyps/hyp.scratch-low.yaml --patience 2 --seed 42 --name sanity_combo --exist-ok --device 0

    Command 'git rev-lis

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL models.yolo.DetectionModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([models.yolo.DetectionModel])` or the `torch.serialization.safe_globals([models.yolo.DetectionModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more abou

In [ ]:
# ==========================================================================
#  E1.8 — الخلية الثانية: التقييم فقط
# ==========================================================================
#  التدريبان انتهيا والأوزان على الدرايف. هذه الخلية لا تدرّب شيئاً.
#
#  لماذا فشلت الخلية الأولى هنا:
#      sitecustomize.py يُصلح torch.load في **العمليات الفرعية** فقط، عبر
#      PYTHONPATH. لكن scan_fpr يعمل داخل عملية النوتبوك نفسها، وهي لم
#      تُرقَّع قط. فاستدعى DetectMultiBackend نسخة torch.load الأصلية،
#      و torch >= 2.6 يرفض تحميل DetectionModel تحت weights_only=True.
#      val.py نجا لأنه عملية فرعية. scan_fpr لا.
#
#  الإصلاح: ترقيع torch.load **في هذه العملية** قبل استيراد أي شيء من
#  المستودع. الملف من تدريبك أنتِ على هذا الجهاز، فلا مسألة ثقة هنا.
# ==========================================================================

import os, sys, json, zipfile, subprocess
from pathlib import Path

try:
    ROOT, D_WEIGHTS, header
except NameError:
    raise RuntimeError("شغّلي S1 قبل هذه الخلية")

E18    = ROOT / "e1_8"
YOLOV5 = Path("/content/yolov5")
IMGSZ  = 640
SEED   = 42
RESULTS = E18 / "e1_8_results.csv"

W_POS = D_WEIGHTS / ("e18__pos__s%d__best.pt" % SEED)
W_NEG = D_WEIGHTS / ("e18__combo__s%d__best.pt" % SEED)
for w in (W_POS, W_NEG):
    assert w.exists(), "الأوزان مفقودة: %s" % w
print("الأوزان موجودة:")
for w in (W_POS, W_NEG):
    print("  %-34s %6.1f MB" % (w.name, w.stat().st_size / 1e6))


# ------------------------------------------- الإصلاح: ترقيع torch.load محلياً
import functools
import numpy as np
import torch

if not getattr(torch.load, "_e18_patched", False):
    _orig_load = torch.load

    @functools.wraps(_orig_load)
    def _patched_load(*a, **k):
        k.setdefault("weights_only", False)
        return _orig_load(*a, **k)

    _patched_load._e18_patched = True
    torch.load = _patched_load
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

print("torch.load patched in-process :", getattr(torch.load, "_e18_patched", False))
print("np.trapz available            :", hasattr(np, "trapz"))

ENV = dict(os.environ)
ENV["PYTHONPATH"] = str(YOLOV5) + os.pathsep + ENV.get("PYTHONPATH", "")
if str(YOLOV5) not in sys.path:
    sys.path.insert(0, str(YOLOV5))


# ------------------------------------------------------------------ mAP عبر val.py
def eval_map(weights, data_yaml, tag):
    r = subprocess.run(
        [sys.executable, str(YOLOV5 / "val.py"), "--weights", str(weights),
         "--data", str(data_yaml), "--img", str(IMGSZ), "--task", "test",
         "--name", "val_" + tag, "--exist-ok", "--device", "0"],
        capture_output=True, text=True, cwd=str(YOLOV5), env=ENV)
    if r.returncode != 0:
        print("  ⚠️ val.py رجع %d — آخر ٣٠ سطراً:" % r.returncode)
        for ln in (r.stdout + "\n" + r.stderr).strip().splitlines()[-30:]:
            print("   ", ln)
        return {}
    out = {}
    for ln in r.stdout.splitlines():
        p = ln.split()
        # سطر الملخّص:  all  <images>  <labels>  P  R  mAP50  mAP50-95
        if len(p) >= 7 and p[0] == "all":
            try:
                out = dict(P=float(p[3]), R=float(p[4]),
                           mAP50=float(p[5]), mAP5095=float(p[6]))
            except ValueError:
                pass
        if p and p[0] in ("flame", "smoke") and len(p) >= 7:
            try:
                out["AP50_" + p[0]] = float(p[5])
            except ValueError:
                pass
    if not out:
        print("  ⚠️ لم أستطع قراءة سطر النتائج من val.py — المخرجات الخام:")
        print(r.stdout[-1200:])
    return out


# -------------------------------------------- FPR على المجموعات المجمّدة
def scan_fpr(weights, tag):
    """أعلى ثقة لكل صورة سالبة، بمسار استدلال المستودع نفسه.

    لا يُستخدم ultralytics: أوزان مستودع YOLOv5 صنف مختلف ولا تُحمَّل به،
    وخلط مسارَي استدلال يجعل الأرقام غير قابلة للمقارنة بأرقامك المسجَّلة.
    """
    import cv2
    from models.common import DetectMultiBackend
    from utils.augmentations import letterbox
    from utils.general import non_max_suppression

    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = DetectMultiBackend(str(weights), device=dev, dnn=False, fp16=False)
    model.warmup(imgsz=(1, 3, IMGSZ, IMGSZ))

    out = {}
    for zname, key in (("hn_holdout_500.zip", "hn_holdout"),
                       ("novel_firelike.zip", "novel"),
                       ("mined_dev.zip", "mined_dev"),
                       ("dfire_neg_1500.zip", "dfire_neg")):
        zp = ROOT / "eval_suite_v2" / zname
        if not zp.exists():
            print("  ⚠️ مفقود:", zp); continue
        z = zipfile.ZipFile(zp)
        names = [n for n in z.namelist()
                 if n.lower().endswith((".jpg", ".png", ".jpeg"))]
        mx = []
        for i, n in enumerate(names):
            im0 = cv2.imdecode(np.frombuffer(z.read(n), np.uint8), cv2.IMREAD_COLOR)
            if im0 is None:
                continue
            im = letterbox(im0, IMGSZ, stride=model.stride, auto=False)[0]
            im = im.transpose((2, 0, 1))[::-1]
            im = torch.from_numpy(np.ascontiguousarray(im)).to(dev).float() / 255.0
            with torch.no_grad():
                pred = model(im[None])
            pred = non_max_suppression(pred, conf_thres=0.001, iou_thres=0.45)[0]
            mx.append(float(pred[:, 4].max()) if len(pred) else 0.0)
            if (i + 1) % 250 == 0:
                print("    %s %d/%d" % (key, i + 1, len(names)))
        a = np.array(mx)
        out["FPR25_" + key] = round(100.0 * float((a >= 0.25).mean()), 2)
        out["FPR50_" + key] = round(100.0 * float((a >= 0.50).mean()), 2)
        np.savetxt(str(E18 / ("scores_%s__%s.csv" % (tag, key))), a,
                   fmt="%.6f", header="max_conf", comments="")
        print("  %-14s %-11s FPR@0.50 = %5.2f %%   FPR@0.25 = %5.2f %%"
              % (tag, key, out["FPR50_" + key], out["FPR25_" + key]))
    return out


# ------------------------------------------------------------------- التشغيل
import pandas as pd

rows = []
for tag, w, dy in (("pos_only",      W_POS, E18 / "ds_pos"   / "data.yaml"),
                   ("plus_safemine", W_NEG, E18 / "ds_combo" / "data.yaml")):
    header("تقييم: " + tag, "-")
    r = {"run": tag, "protocol": "YOLO-HF (scratch, SGD, 300ep)",
         "model": "yolov5s", "seed": SEED}
    r.update(eval_map(w, dy, tag))
    r.update(scan_fpr(w, tag))
    rows.append(r)

df = pd.DataFrame(rows)
df.to_csv(RESULTS, index=False)

print("\n" + "=" * 82)
print("  E1.8 — النتيجة")
print("=" * 82)
with pd.option_context("display.width", 200, "display.max_columns", 40):
    print(df.to_string(index=False))
print()
print("  حُفظ في:", RESULTS)
print("  ودرجات كل صورة في:", E18)

الأوزان موجودة:
  e18__pos__s42__best.pt               14.4 MB
  e18__combo__s42__best.pt             14.4 MB
torch.load patched in-process : True
np.trapz available            : True

----------------------------------------------------------------------
  تقييم: pos_only
----------------------------------------------------------------------


Fusing layers... 


  ⚠️ لم أستطع قراءة سطر النتائج من val.py — المخرجات الخام:



YOLOv5s summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs


    hn_holdout 250/500
    hn_holdout 500/500
  pos_only       hn_holdout  FPR@0.50 = 39.40 %   FPR@0.25 = 47.80 %
  pos_only       novel       FPR@0.50 = 38.78 %   FPR@0.25 = 45.92 %
    mined_dev 250/400
  pos_only       mined_dev   FPR@0.50 = 33.25 %   FPR@0.25 = 43.50 %
    dfire_neg 250/1500
    dfire_neg 500/1500
    dfire_neg 750/1500
    dfire_neg 1000/1500
    dfire_neg 1250/1500
    dfire_neg 1500/1500
  pos_only       dfire_neg   FPR@0.50 = 13.33 %   FPR@0.25 = 23.13 %

----------------------------------------------------------------------
  تقييم: plus_safemine
----------------------------------------------------------------------


Fusing layers... 


  ⚠️ لم أستطع قراءة سطر النتائج من val.py — المخرجات الخام:



YOLOv5s summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs


    hn_holdout 250/500
    hn_holdout 500/500
  plus_safemine  hn_holdout  FPR@0.50 = 10.40 %   FPR@0.25 = 15.60 %
  plus_safemine  novel       FPR@0.50 = 12.24 %   FPR@0.25 = 18.88 %
    mined_dev 250/400
  plus_safemine  mined_dev   FPR@0.50 =  5.50 %   FPR@0.25 = 12.00 %
    dfire_neg 250/1500
    dfire_neg 500/1500
    dfire_neg 750/1500
    dfire_neg 1000/1500
    dfire_neg 1250/1500
    dfire_neg 1500/1500
  plus_safemine  dfire_neg   FPR@0.50 =  6.07 %   FPR@0.25 = 12.87 %

  E1.8 — النتيجة
          run                      protocol   model  seed  FPR25_hn_holdout  FPR50_hn_holdout  FPR25_novel  FPR50_novel  FPR25_mined_dev  FPR50_mined_dev  FPR25_dfire_neg  FPR50_dfire_neg
     pos_only YOLO-HF (scratch, SGD, 300ep) yolov5s    42              47.8              39.4        45.92        38.78             43.5            33.25            23.13            13.33
plus_safemine YOLO-HF (scratch, SGD, 300ep) yolov5s    42              15.6              10.4        18.88        12.24  

In [ ]:
# ==========================================================================
#  E1.8 — الخلية الثالثة: mAP فقط، ودمجها مع نتائج FPR
# ==========================================================================
#  جانب FPR اكتمل في الخلية الثانية. الناقص هو mAP، وقد فشل لسبب في طريقتي:
#
#      كنت أحلّل مخرجات val.py **نصياً**، وجدول النتائج يُطبع عبر LOGGER
#      الذي يكتب في مجرى غير الذي كنت أقرأه، فلم أجد سطر "all" أبداً.
#
#  الحل ليس تخمين المجرى الصحيح ولا قراءة الاثنين — بل إلغاء تحليل النص:
#  val.py يعرّف run() التي **تُرجع الأرقام كقيَم**:
#
#      (mp, mr, map50, map5095, *losses), maps, times = val.run(...)
#
#  فتُستدعى داخل هذه العملية وتؤخذ النتيجة مباشرة. لا نص، لا مجارٍ، لا هشاشة.
#  torch.load مرقَّع أصلاً في هذه العملية من الخلية الثانية — وإن لم تكن
#  شُغّلت، تُعاد الترقيع هنا.
# ==========================================================================

import os, sys, importlib
from pathlib import Path

try:
    ROOT, D_WEIGHTS, header
except NameError:
    raise RuntimeError("شغّلي S1 قبل هذه الخلية")

E18     = ROOT / "e1_8"
YOLOV5  = Path("/content/yolov5")
IMGSZ   = 640
SEED    = 42
RESULTS = E18 / "e1_8_results.csv"

W_POS = D_WEIGHTS / ("e18__pos__s%d__best.pt" % SEED)
W_NEG = D_WEIGHTS / ("e18__combo__s%d__best.pt" % SEED)
for w in (W_POS, W_NEG):
    assert w.exists(), "الأوزان مفقودة: %s" % w

# ------------------------------------------------- الترقيع (إن لم يكن مطبَّقاً)
import functools
import numpy as np
import torch

if not getattr(torch.load, "_e18_patched", False):
    _orig = torch.load

    @functools.wraps(_orig)
    def _patched(*a, **k):
        k.setdefault("weights_only", False)
        return _orig(*a, **k)

    _patched._e18_patched = True
    torch.load = _patched
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid
print("torch.load patched :", getattr(torch.load, "_e18_patched", False))

if str(YOLOV5) not in sys.path:
    sys.path.insert(0, str(YOLOV5))

_cwd = os.getcwd()
os.chdir(str(YOLOV5))            # val.py يحلّ مسارات نسبية من جذر المستودع
try:
    v5val = importlib.import_module("val")
finally:
    os.chdir(_cwd)


def eval_map(weights, data_yaml, tag):
    """mAP بقيَم مُرجَعة، لا بتحليل نص."""
    _here = os.getcwd()
    os.chdir(str(YOLOV5))
    try:
        res, maps, _t = v5val.run(
            data=str(data_yaml), weights=str(weights), batch_size=16,
            imgsz=IMGSZ, task="test", device=0, plots=False, verbose=True,
            project=str(YOLOV5 / "runs" / "val"), name="v_" + tag, exist_ok=True)
    finally:
        os.chdir(_here)
    mp, mr, map50, map5095 = float(res[0]), float(res[1]), float(res[2]), float(res[3])
    out = {"P": round(mp, 4), "R": round(mr, 4),
           "mAP50": round(map50, 4), "mAP5095": round(map5095, 4)}
    # maps = mAP50-95 لكل صنف، بترتيب names
    for i, nm in enumerate(("flame", "smoke")):
        if i < len(maps):
            out["AP5095_" + nm] = round(float(maps[i]), 4)
    print("  %-14s P %.4f  R %.4f  mAP50 %.4f  mAP50-95 %.4f"
          % (tag, mp, mr, map50, map5095))
    return out


import pandas as pd

df = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()

new = {}
for tag, w, dy in (("pos_only",      W_POS, E18 / "ds_pos"   / "data.yaml"),
                   ("plus_safemine", W_NEG, E18 / "ds_combo" / "data.yaml")):
    header("mAP: " + tag, "-")
    new[tag] = eval_map(w, dy, tag)

for tag, vals in new.items():
    for k, v in vals.items():
        df.loc[df["run"] == tag, k] = v
df.to_csv(RESULTS, index=False)

# ----------------------------------------------------------------- العرض
cols_map = [c for c in ("run", "P", "R", "mAP50", "mAP5095",
                        "AP5095_flame", "AP5095_smoke") if c in df.columns]
cols_fpr = ["run"] + [c for c in df.columns if c.startswith("FPR50_")]

print("\n" + "=" * 88)
print("  E1.8 — النتيجة الكاملة   (YOLOv5s، من الصفر، بروتوكول YOLO-HF)")
print("=" * 88)
print("\n  الدقة")
print(df[cols_map].to_string(index=False))
print("\n  الإنذارات الكاذبة عند conf 0.50  (٪)")
print(df[cols_fpr].to_string(index=False))

if len(df) == 2 and "mAP50" in df.columns:
    a = df[df["run"] == "pos_only"].iloc[0]
    b = df[df["run"] == "plus_safemine"].iloc[0]
    print("\n  " + "-" * 84)
    print("  الاستنساخ : mAP50 %.4f مقابل %.3f المُبلَّغ في الورقة  (فرق %+.4f)"
          % (a["mAP50"], 0.904, a["mAP50"] - 0.904))
    print("  ثمن السوالب: mAP50 %.4f -> %.4f  (%+.4f)"
          % (a["mAP50"], b["mAP50"], b["mAP50"] - a["mAP50"]))
    for s in ("hn_holdout", "novel", "mined_dev", "dfire_neg"):
        k = "FPR50_" + s
        if k in df.columns and a[k] > 0:
            print("  %-11s : %.2f %% -> %.2f %%   (%.2fx أقل)"
                  % (s, a[k], b[k], a[k] / b[k] if b[k] > 0 else float("inf")))
    print("  " + "-" * 84)
print("\n  حُفظ في:", RESULTS)

YOLOv5 🚀 v7.0-0-g915bbf2 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
YOLOv5s summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs


torch.load patched : True

----------------------------------------------------------------------
  mAP: pos_only
----------------------------------------------------------------------


test: Scanning /content/sf/datasets/home_fire/test/labels.cache... 1300 images, 15 backgrounds, 0 corrupt: 100%|██████████| 1300/1300 00:00
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100%|██████████| 82/82 00:08
                   all       1300       1586      0.902      0.844      0.886      0.552
                 flame       1300        897       0.91       0.87      0.913      0.583
                 smoke       1300        689      0.894      0.819      0.859       0.52
Speed: 0.1ms pre-process, 1.2ms inference, 1.1ms NMS per image at shape (16, 3, 640, 640)
Results saved to /content/yolov5/runs/val/v_pos_only
YOLOv5 🚀 v7.0-0-g915bbf2 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
YOLOv5s summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs


  pos_only       P 0.9017  R 0.8445  mAP50 0.8860  mAP50-95 0.5517

----------------------------------------------------------------------
  mAP: plus_safemine
----------------------------------------------------------------------


test: Scanning /content/sf/datasets/home_fire/test/labels.cache... 1300 images, 15 backgrounds, 0 corrupt: 100%|██████████| 1300/1300 00:00
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100%|██████████| 82/82 00:07
                   all       1300       1586      0.931      0.835      0.905      0.562
                 flame       1300        897      0.954      0.847      0.939      0.602
                 smoke       1300        689      0.908      0.823      0.871      0.522
Speed: 0.1ms pre-process, 0.7ms inference, 1.1ms NMS per image at shape (16, 3, 640, 640)
Results saved to /content/yolov5/runs/val/v_plus_safemine


  plus_safemine  P 0.9308  R 0.8352  mAP50 0.9050  mAP50-95 0.5625

  E1.8 — النتيجة الكاملة   (YOLOv5s، من الصفر، بروتوكول YOLO-HF)

  الدقة
          run      P      R  mAP50  mAP5095  AP5095_flame  AP5095_smoke
     pos_only 0.9017 0.8445  0.886   0.5517        0.5832        0.5202
plus_safemine 0.9308 0.8352  0.905   0.5625        0.6024        0.5225

  الإنذارات الكاذبة عند conf 0.50  (٪)
          run  FPR50_hn_holdout  FPR50_novel  FPR50_mined_dev  FPR50_dfire_neg
     pos_only              39.4        38.78            33.25            13.33
plus_safemine              10.4        12.24             5.50             6.07

  ------------------------------------------------------------------------------------
  الاستنساخ : mAP50 0.8860 مقابل 0.904 المُبلَّغ في الورقة  (فرق -0.0180)
  ثمن السوالب: mAP50 0.8860 -> 0.9050  (+0.0190)
  hn_holdout  : 39.40 % -> 10.40 %   (3.79x أقل)
  novel       : 38.78 % -> 12.24 %   (3.17x أقل)
  mined_dev   : 33.25 % -> 5.50 %   (6.05x أقل)
  dfire_

## A — إعداد البيانات

---



بناء بيانات التدريب والسوالب وتجميد `Eval Suite v2`.



### A1 — بيانات Home Fire

تنزيل المجموعة من Kaggle واستخراج الإحصاءات الوصفية.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"

os.environ.setdefault("KAGGLE_USERNAME", "ayahadi")
os.environ.setdefault("KAGGLE_KEY", "PUT_YOUR_KAGGLE_KEY_HERE")

def kaggle_ready():
    if not os.environ.get("KAGGLE_KEY", "").startswith("PUT_"):
        return True
    return (Path.home()/".kaggle"/"kaggle.json").exists()

header("A1) home_fire")
if not (D_DATA/"home_fire.zip").exists():
    assert kaggle_ready(), "أحتاج مفتاح Kaggle لتنزيل home_fire"
    pipq("kaggle")
    dl = L_TMP/"hf_dl"; dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "pengbo00/home-fire-dataset",
                    "-p", str(dl), "--unzip"], check=True)
    src_root = dl if (dl/"train").exists() else next(p for p in dl.iterdir() if p.is_dir())
    arch = shutil.make_archive(str(L_TMP/"home_fire"), "zip", root_dir=str(src_root))
    shutil.copy2(arch, D_DATA/"home_fire.zip")

else:
    print("  ✅ كاش موجود:", D_DATA/"home_fire.zip")

data_yaml = ensure_home_fire_local()
hf = Path(data_yaml).parent


stats = {}
for sp in ["train", "val", "test"]:
    imgs = count_imgs(hf/sp/"images")
    flame = smoke = empty = 0
    for lf in (hf/sp/"labels").glob("*.txt"):
        t = lf.read_text().strip()
        if not t:
            empty += 1
            continue
        for ln in t.splitlines():
            c = ln.split()[0]
            if c == "0":
                flame += 1
            elif c == "1":
                smoke += 1
    stats[sp] = dict(images=imgs, flame_boxes=flame, smoke_boxes=smoke, empty_labels=empty)
    print("  %-6s images=%-6d flame=%-6d smoke=%-6d empty=%d" % (sp, imgs, flame, smoke, empty))
(D_LOGS/"home_fire_stats.json").write_text(json.dumps(stats, indent=2))
assert stats["test"]["images"] >= 1200, "test غير متوقع!"
print("  ✅ A1 تمّت — الإحصاءات محفوظة logs/home_fire_stats.json")


  A1) home_fire
  ✅ كاش موجود: /content/drive/MyDrive/SAFE_FIRE/datasets/home_fire.zip
  train  images=3900   flame=2978   smoke=1834   empty=87
  val    images=1300   flame=963    smoke=617    empty=33
  test   images=1300   flame=897    smoke=689    empty=15
  ✅ A1 تمّت — الإحصاءات محفوظة logs/home_fire_stats.json


### A2 — مجموعة السوالب الصعبة

بناء فئات Lamp وLight bulb وCandle مع فصل holdout عن train pool. ***Open Image.***


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("A2) HN pool: Lamp / Light bulb / Candle")

HN_HOLD_ZIP = NEG_ZIPS["hn_holdout"]
HN_POOL_ZIP = D_TRAINNEG/"hn_train_pool.zip"

if HN_HOLD_ZIP.exists() and HN_POOL_ZIP.exists():
    pass
else:
    pipq("fiftyone")
    import fiftyone as fo
    import fiftyone.zoo as foz
    from PIL import Image

    HN_CLASSES = ["Lamp", "Light bulb", "Candle"]
    PER = 1500
    outd = L_TMP/"hn_raw"; outd.mkdir(parents=True, exist_ok=True)
    seen_oids = set()
    comp = {}
    for cls in HN_CLASSES:
        kept = 0
        try:

            dsn = "hn_" + cls.replace(" ", "_") + "_" + datetime.now().strftime("%H%M%S")
            ds = foz.load_zoo_dataset("open-images-v7", split="train",
                                      label_types=["classifications"], classes=[cls],
                                      max_samples=PER, shuffle=True, seed=42,
                                      dataset_name=dsn)
            for s in ds.select_fields(["filepath", "positive_labels"]):
                labs = set()
                try:
                    pl = s["positive_labels"]
                    if pl is not None:
                        labs = {c.label for c in pl.classifications}
                except Exception:
                    pass
                if cls not in labs:
                    continue
                p = Path(s.filepath)
                if (p.stem in seen_oids) or (not p.exists()):
                    continue
                try:
                    Image.open(p).verify()
                    w, h = Image.open(p).size
                    if w >= 200 and h >= 200 and max(w, h)/min(w, h) <= 4:
                        dst = outd/(cls.replace(" ", "_") + "_" + p.stem + ".jpg")
                        shutil.copy2(p, dst)
                        seen_oids.add(p.stem)
                        kept += 1
                except Exception:
                    pass
        except Exception as e:
            pass
        comp[cls] = kept
        print("  %-12s -> %d صورة صالحة" % (cls, kept))

    allhn = sorted(outd.glob("*.jpg"))
    rng = random.Random(42)
    rng.shuffle(allhn)
    assert len(allhn) >= 1200, "عدد HN قليل (%d) — ارفعي PER" % len(allhn)
    holdout, pool = allhn[:500], allhn[500:]
    zip_flat(holdout, HN_HOLD_ZIP)
    zip_flat(pool, HN_POOL_ZIP)
    (D_LOGS/"hn_composition.json").write_text(json.dumps(
        {"classes": comp, "total": len(allhn), "holdout": len(holdout),
         "train_pool": len(pool), "seed": 42}, indent=2))
    print("  ✅ holdout=%d → %s" % (len(holdout), HN_HOLD_ZIP.name))
    print("  ✅ train pool=%d → %s (منفصل تماماً عن الـ holdout)" % (len(pool), HN_POOL_ZIP.name))


  A2) HN pool: Lamp / Light bulb / Candle


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


Necessary images already downloaded


INFO:fiftyone.utils.openimages:Necessary images already downloaded


Existing download of split 'train' is sufficient


...

 100% |███████████████| 1500/1500 [6.7s elapsed, 0s remaining, 203.2 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 1500/1500 [6.7s elapsed, 0s remaining, 203.2 samples/s]      


Dataset 'hn_Candle_175125' created


INFO:fiftyone.zoo.datasets:Dataset 'hn_Candle_175125' created


  Candle       -> 670 صورة صالحة
  ✅ holdout=500 → hn_holdout_500.zip
  ✅ train pool=1638 → hn_train_pool.zip (منفصل تماماً عن الـ holdout)


### A3 — مجموعة Novel Fire-like

فئات مضيئة غير مستخدمة في التدريب لقياس التعميم.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("A3) novel_firelike")

if NEG_ZIPS["novel"].exists():
    pass
else:
    pipq("fiftyone")
    import fiftyone as fo
    import fiftyone.zoo as foz
    from PIL import Image


    used_ids = set()
    for zp in [D_TRAINNEG/"hn_train_pool.zip", NEG_ZIPS["hn_holdout"]]:
        assert zp.exists(), "شغّلي A2 أولاً"
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                used_ids.add(Path(n).stem.split("_")[-1])

    NOVEL_CLASSES = ["Lantern", "Torch", "Street light", "Traffic light", "Flashlight", "Chandelier"]
    outd = L_TMP/"novel_raw"; outd.mkdir(parents=True, exist_ok=True)
    comp = {}
    for cls in NOVEL_CLASSES:
        kept = 0
        try:

            dsn = "nv_" + cls.replace(" ", "_") + "_" + datetime.now().strftime("%H%M%S")
            ds = foz.load_zoo_dataset("open-images-v7", split="train",
                                      label_types=["classifications"], classes=[cls],
                                      max_samples=130, shuffle=True, seed=42,
                                      dataset_name=dsn)
            for s in ds.select_fields(["filepath", "positive_labels"]):
                labs = set()
                try:
                    pl = s["positive_labels"]
                    if pl is not None:
                        labs = {c.label for c in pl.classifications}
                except Exception:
                    pass
                if cls not in labs:
                    continue
                p = Path(s.filepath)
                if (p.stem in used_ids) or (not p.exists()):
                    continue
                try:
                    Image.open(p).verify()
                    w, h = Image.open(p).size
                    if w >= 200 and h >= 200 and max(w, h)/min(w, h) <= 4:
                        dst = outd/("nv_" + cls.replace(" ", "_") + "_" + p.stem + ".jpg")
                        shutil.copy2(p, dst)
                        used_ids.add(p.stem)
                        kept += 1
                except Exception:
                    pass
        except Exception as e:
            pass
        comp[cls] = kept
        print("  %-16s -> %d" % (cls, kept))
    allnovel = sorted(outd.glob("*.jpg"))
    rng = random.Random(42); rng.shuffle(allnovel)
    allnovel = allnovel[:400]
    assert len(allnovel) >= 150, "novel قليلة جداً (%d)" % len(allnovel)
    zip_flat(allnovel, NEG_ZIPS["novel"])
    (D_LOGS/"novel_composition.json").write_text(json.dumps(comp, indent=2))
    print("  ✅ novel_firelike.zip:", len(allnovel), "صورة")


  A3) novel_firelike


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


Found 36 images, downloading the remaining 94


INFO:fiftyone.utils.openimages:Found 36 images, downloading the remaining 94


 100% |█████████████████████| 94/94 [13.3s elapsed, 0s remaining, 7.9 files/s]      


...

 100% |█████████████████| 130/130 [175.7ms elapsed, 0s remaining, 774.5 samples/s] 


INFO:eta.core.utils: 100% |█████████████████| 130/130 [175.7ms elapsed, 0s remaining, 774.5 samples/s] 


Dataset 'nv_Chandelier_180653' created


INFO:fiftyone.zoo.datasets:Dataset 'nv_Chandelier_180653' created


  Chandelier       -> 0
  ✅ novel_firelike.zip: 196 صورة


### A4 — سوالب D-Fire

تخزين المصدر واستخراج 1,500 صورة سالبة للتقييم عبر البيانات.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("A4) dfire_neg_1500")

if NEG_ZIPS["dfire_neg"].exists():
    pass
else:
    dfl = L_DS/"dfire"
    raw_zip = D_DATA/"dfire_raw.zip"
    if count_imgs(dfl) < 1000:
        if raw_zip.exists():
            print("  فك dfire_raw.zip من الكاش ...")
            robust_extract(raw_zip, dfl)
        else:
            assert kaggle_ready(), "أحتاج مفتاح Kaggle لتنزيل D-Fire"
            pipq("kaggle")
            dl = L_TMP/"df_dl"; dl.mkdir(parents=True, exist_ok=True)
            subprocess.run(["kaggle", "datasets", "download", "-d",
                            "sayedgamal99/smoke-fire-detection-yolo", "-p", str(dl)], check=True)
            zps = list(dl.glob("*.zip")); assert zps, "تنزيل D-Fire فشل"
            shutil.copy2(zps[0], raw_zip)
            robust_extract(zps[0], dfl)
    test_split = None
    for d in dfl.rglob("images"):
        if d.is_dir() and d.parent.name.lower() in ("test", "val", "valid"):
            test_split = d.parent
            if d.parent.name.lower() == "test":
                break
    assert test_split is not None, "ما لگيت test split بـ D-Fire"
    img_d, lbl_d = test_split/"images", test_split/"labels"
    negs = []
    for ip in img_d.iterdir():
        if ip.suffix.lower() not in IMG_EXT:
            continue
        lf = lbl_d/(ip.stem + ".txt")
        if (not lf.exists()) or (not lf.read_text().strip()):
            negs.append(ip)
    print("  سوالب متاحة:", len(negs))
    rng = random.Random(42); rng.shuffle(negs)
    zip_flat(negs[:1500], NEG_ZIPS["dfire_neg"], arc_prefix="dfneg_")
    print("  ✅ dfire_neg_1500.zip:", min(1500, len(negs)), "صورة")



  A4) dfire_neg_1500
  فك dfire_raw.zip من الكاش ...
  سوالب متاحة: 2005
  ✅ dfire_neg_1500.zip: 1500 صورة


### A5 — سوالب COCO العشوائية

مجموعة ضابطة لمقارنة السوالب العشوائية بالسوالب الصعبة.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("A5) randbg pool (COCO)")

RBG_ZIP = D_TRAINNEG/"randbg_pool.zip"
if RBG_ZIP.exists():
    pass
else:
    pipq("fiftyone")
    import fiftyone as fo
    import fiftyone.zoo as foz
    from PIL import Image
    outd = L_TMP/"rbg_raw"; outd.mkdir(parents=True, exist_ok=True)
    dsn = "rbg_coco_" + datetime.now().strftime("%H%M%S")
    ds = foz.load_zoo_dataset("coco-2017", split="validation",
                              max_samples=2000, shuffle=True, seed=42, dataset_name=dsn)
    kept = 0
    for s in ds:
        p = Path(s.filepath)
        try:
            Image.open(p).verify()
            dst = outd/("rbg_" + p.stem + ".jpg")
            if not dst.exists():
                shutil.copy2(p, dst); kept += 1
        except Exception:
            pass
    allr = sorted(outd.glob("*.jpg"))
    zip_flat(allr, RBG_ZIP)
    print("  ✅ randbg_pool.zip:", len(allr), "صورة")


  A5) randbg pool (COCO)


INFO:fiftyone.zoo.datasets:Downloading split 'validation' to '/root/fiftyone/coco-2017/validation' if necessary


INFO:fiftyone.utils.coco:Downloading annotations to '/root/fiftyone/coco-2017/tmp-download/annotations_trainval2017.zip'


 100% |██████|    1.9Gb/1.9Gb [5.4s elapsed, 0s remaining, 382.2Mb/s]       


INFO:eta.core.utils: 100% |██████|    1.9Gb/1.9Gb [5.4s elapsed, 0s remaining, 382.2Mb/s]       


...

 100% |███████████████| 2000/2000 [18.4s elapsed, 0s remaining, 164.3 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 2000/2000 [18.4s elapsed, 0s remaining, 164.3 samples/s]      


Dataset 'rbg_coco_181116' created


INFO:fiftyone.zoo.datasets:Dataset 'rbg_coco_181116' created


  ✅ randbg_pool.zip: 2000 صورة


### A6 — تجميد Eval Suite v2

إنشاء حزمة التقييم وحفظ manifest مع بصمات MD5.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("A6) تجميد Eval Suite v2")

data_yaml = ensure_home_fire_local()
n_test = count_imgs(Path(data_yaml).parent/"test"/"images")
man = {"created": datetime.now().isoformat(timespec="seconds"), "seed": 42,
       "positives": {"dataset": "home_fire test split", "n": int(n_test)},
       "negatives": {}}
for name in ["hn_holdout", "novel", "dfire_neg", "mined_dev"]:
    zp = NEG_ZIPS[name]
    if not zp.exists():
        if name == "mined_dev":
            print("  (mined_dev يُضاف لاحقاً بالمرحلة D — طبيعي)")
            continue
        raise AssertionError(name + " ناقص — شغّلي خليته أولاً")
    man["negatives"][name] = {"zip": zp.name, "n": zip_count(zp),
                              "size_MB": round(zp.stat().st_size/1e6, 1),
                              "md5": file_md5(zp)}
(D_SUITE/"manifest.json").write_text(json.dumps(man, indent=2, ensure_ascii=False))
print(json.dumps(man, indent=2, ensure_ascii=False))
print()
print("✅ Eval Suite v2 مجمّدة — ممنوع تعديل أي ملف بداخلها من الآن")
print("   الجاية: جلسة تدريب جديدة ← S1 ← B1 ← B2")



  A6) تجميد Eval Suite v2
  (mined_dev يُضاف لاحقاً بالمرحلة D — طبيعي)
{
  "created": "2026-07-13T18:20:12",
  "seed": 42,
  "positives": {
    "dataset": "home_fire test split",
    "n": 1300
  },
  "negatives": {
    "hn_holdout": {
      "zip": "hn_holdout_500.zip",
      "n": 500,
      "size_MB": 129.8,
      "md5": "327e4faad2ef6669679375c9421ba706"
    },
    "novel": {
      "zip": "novel_firelike.zip",
      "n": 196,
      "size_MB": 56.9,
      "md5": "ad0d42bd2ea4e0dcc594250bd0b358d6"
    },
    "dfire_neg": {
      "zip": "dfire_neg_1500.zip",
      "n": 1500,
      "size_MB": 251.9,
      "md5": "2caae5f6cca07426f9ff76e832f85f03"
    }
  }
}

✅ Eval Suite v2 مجمّدة — ممنوع تعديل أي ملف بداخلها من الآن
   الجاية: جلسة تدريب جديدة ← S1 ← B1 ← B2


## B — التدريب واختيار النموذج

تعريف المحرك الموحد، تشغيل بطولة النماذج، ثم اعتماد أفضل نموذج خفيف.


### B1 — محرك التدريب والتقييم

تعريف التدريب الموحد، التقييم، التسجيل وخسارة كبح الخلفية.


In [ ]:
assert "RECIPE" in globals(), "شغّلي S1 أولاً"
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.models.yolo.detect import DetectionTrainer

class FocalBCE(nn.Module):
    def __init__(self, gamma=1.5, alpha=0.25):
        super().__init__()
        self.g, self.a = gamma, alpha
    def forward(self, pred, true):
        loss = F.binary_cross_entropy_with_logits(pred, true, reduction="none")
        p = pred.sigmoid()
        pt = true * p + (1 - true) * (1 - p)
        loss = loss * (1 - pt) ** self.g
        if self.a > 0:
            loss = loss * (true * self.a + (1 - true) * (1 - self.a))
        return loss

def _inner_det(crit):
    for c in (crit, getattr(crit, "one2many", None), getattr(crit, "one2one", None)):
        if c is not None and hasattr(c, "nc") and hasattr(c, "no"):
            return c
    raise RuntimeError("بنية خسارة غير معروفة: " + type(crit).__name__)

def _all_tensors(obj, out=None, depth=0):
    if out is None:
        out = []
    if depth > 8:
        return out
    if torch.is_tensor(obj):
        out.append(obj)
    elif isinstance(obj, dict):
        for v in obj.values():
            _all_tensors(v, out, depth + 1)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            _all_tensors(v, out, depth + 1)
    return out

class BGSuppress:

    def __init__(self, base, lam=1.0, gamma=2.0, use_focal=False):
        self.base, self.lam, self.gamma = base, lam, gamma
        self.inner = _inner_det(base)
        if use_focal:
            for c in (base, getattr(base, "one2many", None), getattr(base, "one2one", None)):
                if c is not None and hasattr(c, "bce"):
                    c.bce = FocalBCE()
        self.trace = []
    def __getattr__(self, k):
        base = self.__dict__.get("base")
        if base is None:
            raise AttributeError(k)
        return getattr(base, k)
    def __deepcopy__(self, memo):
        import copy as _copy
        return _copy.deepcopy(self.__dict__["base"], memo)
    def __reduce__(self):
        return self.__dict__["base"].__reduce_ex__(2)
    @staticmethod
    def _prob(t):
        t = t.reshape(t.shape[0], t.shape[1], -1)
        return t.sigmoid() if (t.min() < 0 or t.max() > 1) else t
    def __call__(self, preds, batch):
        total, items = self.base(preds, batch)
        if self.lam > 0:
            no, nc = self.inner.no, self.inner.nc
            tens = _all_tensors(preds)
            smax = None
            cls_maps = [t for t in tens if t.dim() == 3 and t.shape[1] == nc]
            if cls_maps:
                for t in cls_maps:
                    s = self._prob(t).amax(dim=(1, 2))
                    smax = s if smax is None else torch.maximum(smax, s)
            else:
                for t in tens:
                    if t.dim() >= 3 and t.shape[1] == no and no > nc:
                        bs = t.shape[0]
                        s = self._prob(t.reshape(bs, no, -1)[:, no - nc:, :]).amax(dim=(1, 2))
                        smax = s if smax is None else torch.maximum(smax, s)
            if smax is None:
                raise RuntimeError("[L_bg] ما لگيت درجات أصناف بالـ preds")
            bs = smax.shape[0]
            bidx = batch["batch_idx"].view(-1).long()
            has_gt = torch.zeros(bs, dtype=torch.bool, device=smax.device)
            if bidx.numel():
                has_gt[bidx.unique()] = True
            neg = ~has_gt
            if neg.any():
                lbg = (smax[neg] ** self.gamma).mean()
                total = total + self.lam * bs * lbg
                self.trace.append(float(lbg.detach()))
        return total, items

LAM, GAM, FOCAL = 0.0, 2.0, False

class SMTrainer(DetectionTrainer):
    def _setup_train(self, *args, **kwargs):
        super()._setup_train(*args, **kwargs)
        m = self.model.module if hasattr(self.model, "module") else self.model
        if getattr(m, "args", None) is None:
            m.args = self.args
        m.criterion = BGSuppress(m.init_criterion(), lam=LAM, gamma=GAM, use_focal=FOCAL)
        print("[L_bg] criterion مضبوط: lam=%s gamma=%s focal=%s" % (LAM, GAM, FOCAL))


KEYMAP = {"hn_holdout": "hn", "novel": "novel", "dfire_neg": "dfire", "mined_dev": "mined"}

def scan_max_conf(model, folder, out_csv, imgsz=640):

    out_csv = Path(out_csv)
    if out_csv.exists():
        return pd.read_csv(out_csv)["max_conf"].to_numpy()
    imgs = imgs_in(folder)
    rows, B = [], 64
    for i in range(0, len(imgs), B):
        chunk = [str(p) for p in imgs[i:i+B]]
        res = model.predict(chunk, conf=0.01, imgsz=imgsz, verbose=False, save=False)
        for src, r in zip(chunk, res):
            mc = float(r.boxes.conf.max().item()) if len(r.boxes) else 0.0
            rows.append((Path(src).name, mc))
    pd.DataFrame(rows, columns=["image", "max_conf"]).to_csv(out_csv, index=False)
    return np.array([r[1] for r in rows])

def evaluate_full(weights_path, run_id, imgsz=640):
    m = YOLO(str(weights_path))
    data_yaml = ensure_home_fire_local()
    suite = ensure_suite_local(strict=True)
    v = m.val(data=data_yaml, split="test", imgsz=imgsz, verbose=False, plots=False,
              project=str(L_TMP), name="v", exist_ok=True)
    p, r = float(v.box.mp), float(v.box.mr)
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    ap50, ap = v.box.ap50, v.box.ap
    res = {"mAP50": round(float(v.box.map50), 4), "mAP5095": round(float(v.box.map), 4),
           "precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4),
           "flame_AP50":   round(float(ap50[0]), 4) if len(ap50) > 0 else "",
           "smoke_AP50":   round(float(ap50[1]), 4) if len(ap50) > 1 else "",
           "flame_AP5095": round(float(ap[0]), 4)   if len(ap) > 0 else "",
           "smoke_AP5095": round(float(ap[1]), 4)   if len(ap) > 1 else ""}
    sweep_csv = D_SWEEPS/(run_id + "__test_sweep.csv")
    if sweep_csv.exists() and len(pd.read_csv(sweep_csv)) >= len(THRS):
        sw = pd.read_csv(sweep_csv)
    else:
        rows = []
        for t in THRS:
            vv = m.val(data=data_yaml, split="test", imgsz=imgsz, conf=t,
                       verbose=False, plots=False, project=str(L_TMP), name="v", exist_ok=True)
            rows.append({"conf": t, "recall": round(float(vv.box.mr), 4),
                         "precision": round(float(vv.box.mp), 4),
                         "mAP50": round(float(vv.box.map50), 4)})
        sw = pd.DataFrame(rows)
        sw.to_csv(sweep_csv, index=False)
    scores = {}
    for sname, sdir in suite.items():
        sc_csv = D_SCORES/(run_id + "__" + sname + ".csv")
        print("    مسح سوالب:", sname)
        mx = scan_max_conf(m, sdir, sc_csv, imgsz=imgsz)
        scores[sname] = mx
        res["FPR25_" + KEYMAP[sname]] = round(100*float((mx >= 0.25).mean()), 2)
        res["FPR50_" + KEYMAP[sname]] = round(100*float((mx >= 0.50).mean()), 2)
    hn = scores["hn_holdout"]
    best_rec = 0.0
    for _, rw in sw.iterrows():
        t = float(rw["conf"])
        if 100*float((hn >= t).mean()) <= 1.0:
            best_rec = max(best_rec, float(rw["recall"]))
    res["recall_at_fpr1"] = round(best_rec, 4)
    try:
        res["params_M"] = round(sum(x.numel() for x in m.model.parameters())/1e6, 3)
    except Exception:
        res["params_M"] = ""
    try:
        from ultralytics.utils.torch_utils import get_flops
        res["GFLOPs"] = round(float(get_flops(m.model, imgsz)), 2)
    except Exception:
        res["GFLOPs"] = ""
    return res


def train_and_eval(name, model_name, seed, group, data_yaml=None, epochs=None,
                   imgsz=None, batch=None, lam=0.0, gamma=2.0, focal=False,
                   extra_args=None, sanity=False, notes=""):
    global LAM, GAM, FOCAL
    run_id = group + "__" + name + "__s" + str(seed)
    if (not sanity) and already_done(run_id):
        print("✅", run_id, "منجز — تخطّي")
        return None
    header(("SANITY: " if sanity else "RUN: ") + run_id, "#")
    dy = data_yaml if data_yaml else ensure_home_fire_local()
    best_drive = D_WEIGHTS/(run_id + "__best.pt")
    train_min, best_epoch = "", ""
    need_train = sanity or (not best_drive.exists())
    if not need_train:
        print("  ⏭ الأوزان موجودة — تقييم فقط")
    else:
        args = dict(RECIPE)
        if epochs is not None: args["epochs"] = epochs
        if imgsz  is not None: args["imgsz"]  = imgsz
        if batch  is not None: args["batch"]  = batch
        if sanity:
            args["epochs"], args["patience"] = 2, 2
        if extra_args:
            args.update(extra_args)
        LAM, GAM, FOCAL = float(lam), float(gamma), bool(focal)
        need_custom = (LAM > 0) or FOCAL
        t0 = time.time()
        model = YOLO(model_name + ".pt")
        model.train(trainer=SMTrainer if need_custom else None,
                    data=dy, seed=seed,
                    project=str(L_RUNS), name=run_id, exist_ok=True,
                    plots=(not sanity), save=True, verbose=True, **args)
        train_min = round((time.time() - t0)/60, 1)
        if LAM > 0:
            crit = getattr(model.trainer.model, "criterion", None)
            tr = getattr(crit, "trace", [])
            assert len(tr) > 0, "[L_bg] ما انحسبت ولا مرة — تأكدي الداتاسيت فيه سوالب!"
            print("[L_bg] steps=%d | متوسط كلي=%.4f | آخر 50=%.4f (المفروض ينزل)"
                  % (len(tr), float(np.mean(tr)), float(np.mean(tr[-50:]))))
        rundir = L_RUNS/run_id
        best = rundir/"weights"/"best.pt"
        assert best.exists(), "best.pt مفقود!"
        if sanity:
            print("🧪 SANITY تمّت (%s دقيقة) — الربط سليم" % train_min)
            return str(best)
        shutil.copy2(best, best_drive)
        adir = D_RUNS/run_id; adir.mkdir(parents=True, exist_ok=True)
        for f_ in rundir.glob("*.*"):
            if f_.suffix.lower() in (".png", ".jpg", ".csv", ".yaml"):
                try: shutil.copy2(f_, adir/f_.name)
                except Exception: pass
        try:
            rc = pd.read_csv(rundir/"results.csv")
            rc.columns = [c.strip() for c in rc.columns]
            col = next((c for c in rc.columns if "mAP50-95" in c), None)
            if col is not None:
                idx = rc[col].idxmax()
                best_epoch = int(rc.loc[idx, "epoch"]) if "epoch" in rc.columns else int(idx)
        except Exception:
            pass
        print("  ✅ التدريب خلص بـ", train_min, "دقيقة")
    print("  📊 التقييم الكامل ...")
    res = evaluate_full(best_drive, run_id, imgsz=(imgsz or 640))
    row = {"run_id": run_id, "group": group, "name": name, "model": model_name,
           "seed": seed, "data": Path(dy).parent.name, "lam": lam, "focal": focal,
           "imgsz": (imgsz or RECIPE["imgsz"]),
           "timestamp": datetime.now().isoformat(timespec="seconds"),
           "epochs": (epochs if epochs is not None else RECIPE["epochs"]),
           "best_epoch": best_epoch, "train_min": train_min,
           "weights": str(best_drive), "notes": notes}
    row.update(res)
    append_master(row)
    print("  ✅ %s | mAP50=%s | mAP50-95=%s | R@FPR≤1%%=%s | FPR50_hn=%s%%"
          % (run_id, res["mAP50"], res["mAP5095"], res["recall_at_fpr1"], res.get("FPR50_hn", "")))
    return row

print("✅ B1 جاهز — train_and_eval / evaluate_full / BGSuppress مُعرَّفة")


✅ B1 جاهز — train_and_eval / evaluate_full / BGSuppress مُعرَّفة


### B2 — بطولة اختيار النموذج

مقارنة ستة نماذج عبر ثلاث بذور مع إعادة استخدام الأوزان الموجودة.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"

T_MODELS = ["yolov8n", "yolo11n", "yolo26n", "yolov8s", "yolo11s", "yolo26s"]
todo = [(m, s) for s in SEEDS for m in T_MODELS]
remaining = [(m, s) for (m, s) in todo if not already_done("tournament__" + m + "__s" + str(s))]
print("إجمالي:", len(todo), "| منجز:", len(todo) - len(remaining), "| متبقي:", len(remaining))

for i, (mname, seed) in enumerate(remaining, 1):
    print(); print("#"*70)
    print("#  البطولة %d/%d : %s (seed=%d)" % (i, len(remaining), mname, seed))
    print("#"*70)
    train_and_eval(mname, mname, seed, group="tournament")

left = [(m, s) for (m, s) in todo if not already_done("tournament__" + m + "__s" + str(s))]
print()
print("🏁 البطولة مكتملة ✅ — شغّلي B3" if not left else "⏳ باقي %d رنّة — أعيدي B2 بالجلسة الجاية" % len(left))

إجمالي: 18 | منجز: 0 | متبقي: 18

######################################################################
#  البطولة 1/18 : yolov8n (seed=42)
######################################################################

######################################################################
  RUN: tournament__yolov8n__s42
######################################################################
  فك home_fire.zip ...
  ⏭ الأوزان موجودة — تقييم فقط
  📊 التقييم الكامل ...
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3323.1±771.8 MB/s, size: 284.4 KB)
val: Scanning /content/sf/datasets/home_fire/test/labels... 1300 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1300/1300 1.5Kit/s 0.9s
val: New cache created: /content/sf/datasets/home_fire/test/labels.cache
                 Class     Images  Instances      Box(P          R      

### B3 — تقرير البطولة واختيار الفائز

اعتماد أعلى متوسط mAP50-95 بين النماذج الخفيفة.


In [ ]:
assert "MASTER_CSV" in globals(), "شغّلي S1 أولاً"
NANO_MODELS = {"yolov8n", "yolo11n", "yolo26n"}

df = master_df("tournament")
print("رنّات منجزة:", len(df), "من", 6*len(SEEDS))
assert len(df) > 0, "شغّلي B2 أولاً"

missing = df[df["params_M"].isna()]["model"].unique().tolist()
if missing:
    print("إصلاح params/GFLOPs الناقصة:", missing)
    full = pd.read_csv(MASTER_CSV).drop_duplicates(subset="run_id", keep="last")
    for mn in missing:
        rows = df[df["model"] == mn]
        w = next((Path(str(x)) for x in rows["weights"] if Path(str(x)).exists()), None)
        if w is None:

            continue
        mm = YOLO(str(w))
        p_ = round(sum(x.numel() for x in mm.model.parameters())/1e6, 3)
        try:
            from ultralytics.utils.torch_utils import get_flops
            g_ = round(float(get_flops(mm.model, 640)), 2)
        except Exception:
            g_ = np.nan
        df.loc[df["model"] == mn, "params_M"] = p_
        df.loc[df["model"] == mn, "GFLOPs"] = g_
        full.loc[(full["group"] == "tournament") & (full["model"] == mn), "params_M"] = p_
        full.loc[(full["group"] == "tournament") & (full["model"] == mn), "GFLOPs"] = g_
        print("  ✅ %s: %.3fM params | GFLOPs=%s" % (mn, p_, g_))
    tmp = MASTER_CSV.with_suffix(".csv.tmp")
    full.to_csv(tmp, index=False)
    tmp.replace(MASTER_CSV)
    print("  💾 runs_master_v2.csv رُقّع")

g = df.groupby("model")
summary = pd.DataFrame({
    "runs": g.size(),
    "mAP5095": g["mAP5095"].mean().round(4), "mAP5095_std": g["mAP5095"].std().round(4),
    "mAP50": g["mAP50"].mean().round(4), "recall": g["recall"].mean().round(4),
    "f1": g["f1"].mean().round(4), "R_at_FPR1": g["recall_at_fpr1"].mean().round(4),
    "FPR50_hn": g["FPR50_hn"].mean().round(2),
    "params_M": g["params_M"].mean().round(2), "GFLOPs": g["GFLOPs"].mean().round(1),
}).sort_values("mAP5095", ascending=False)

header("ترتيب البطولة (متوسط عبر الـ seeds)")
print(summary.to_string())

nano = summary[summary.index.isin(NANO_MODELS)]
big  = summary[~summary.index.isin(NANO_MODELS)]
assert not nano.empty, "ماكو موديلات نانو منجزة بعد"
win = str(nano.index[0])
header("🏆 الفائز: " + win)
print("  القاعدة: أعلى mAP50-95 (متوسط الـ seeds) بين الموديلات الخفيفة")
if len(nano) > 1:
    print("  الفارق عن الوصيف (%s): %.4f" % (nano.index[1],
          float(nano.iloc[0]["mAP5095"]) - float(nano.iloc[1]["mAP5095"])))
save_decision("winner", win)
if not big.empty:
    s_best = str(big.index[0])
    save_decision("teacher_model", s_best)
    header("مرشح المعلّم (KD بالمرحلة F): " + s_best)
    print(big.to_string())
summary.to_csv(D_RESULTS/"tournament_results.csv")
print(); print("💾", D_RESULTS/"tournament_results.csv")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for mname, rw in summary.iterrows():
        if pd.isna(rw["GFLOPs"]):
            continue
        yerr = rw["mAP5095_std"] if pd.notna(rw["mAP5095_std"]) else 0.0
        ax.errorbar(rw["GFLOPs"], rw["mAP5095"], yerr=yerr, marker="o", capsize=4)
        ax.annotate(str(mname), (rw["GFLOPs"], rw["mAP5095"]),
                    xytext=(6, 5), textcoords="offset points", fontsize=9)
    ax.set_xlabel("GFLOPs"); ax.set_ylabel("mAP50-95 (test, mean±std)")
    ax.set_title("Model Selection Tournament"); ax.grid(alpha=.3)
    fig.savefig(D_FIGS/"tournament_scatter.png", dpi=200, bbox_inches="tight")
    print("🖼", D_FIGS/"tournament_scatter.png")
except Exception as e:
    pass

رنّات منجزة: 18 من 18
إصلاح params/GFLOPs الناقصة: ['yolov8n', 'yolo11n', 'yolo26n', 'yolov8s', 'yolo11s', 'yolo26s']
  ✅ yolov8n: 3.011M params | GFLOPs=8.2
  ✅ yolo11n: 2.590M params | GFLOPs=6.44
  ✅ yolo26n: 2.505M params | GFLOPs=5.77
  ✅ yolov8s: 11.136M params | GFLOPs=28.65
  ✅ yolo11s: 9.429M params | GFLOPs=21.55
  ✅ yolo26s: 9.949M params | GFLOPs=22.5
  💾 runs_master_v2.csv رُقّع

  ترتيب البطولة (متوسط عبر الـ seeds)
         runs  mAP5095  mAP5095_std   mAP50  recall      f1  R_at_FPR1  FPR50_hn  params_M  GFLOPs
model                                                                                             
yolo26s     3   0.6012       0.0095  0.9215  0.8782  0.9050     0.0000     26.40      9.95    22.5
yolov8s     3   0.5893       0.0053  0.9145  0.8650  0.8925     0.1230     30.53     11.14    28.6
yolo11s     3   0.5891       0.0100  0.9056  0.8605  0.8919     0.0272     33.73      9.43    21.6
yolo26n     3   0.5843       0.0073  0.9013  0.8481  0.8815     0.0000 

## C — معايرة السوالب الصعبة

اختيار نسبة السوالب التي تحقق أفضل `Recall@FPR≤1%` ثم تثبيت النتيجة عبر بذور إضافية.


### C1 — بناء مجموعات النِسَب

تجهيز نسب السوالب الصعبة والضابط العشوائي.


In [ ]:
assert "build_neg_dataset" in globals() or "ROOT" in globals(), "شغّلي S1 أولاً"
header("C1) بناء داتاسيتات النسب")

hn_pool_dir = L_DS/"hn_pool"
if count_imgs(hn_pool_dir) == 0:
    zp = D_TRAINNEG/"hn_train_pool.zip"
    assert zp.exists(), "شغّلي A2 أولاً"
    robust_extract(zp, hn_pool_dir)
rbg_dir = L_DS/"rbg_pool"
if count_imgs(rbg_dir) == 0:
    zp = D_TRAINNEG/"randbg_pool.zip"
    assert zp.exists(), "شغّلي A5 أولاً"
    robust_extract(zp, rbg_dir)

pool = imgs_in(hn_pool_dir)
rng = random.Random(42); rng.shuffle(pool)
rbg = imgs_in(rbg_dir)
rng2 = random.Random(42); rng2.shuffle(rbg)

n_pos = count_imgs(Path(ensure_home_fire_local()).parent/"train"/"images")
n_for = lambda pct: round(pct/(100.0-pct)*n_pos)
print("  إيجابيات التدريب:", n_pos, "| HN pool:", len(pool), "| randbg pool:", len(rbg))

RATIO_DEFS = {"hn15": 15, "hn30": 30, "hn45": 45}
DATA_YAMLS = {}
counts = {}
for tag, pct in RATIO_DEFS.items():
    n = min(n_for(pct), len(pool))
    sel = pool[:n]
    DATA_YAMLS[tag] = build_neg_dataset(tag, [(p, "") for p in sel])
    counts[tag] = n

    (D_SM/"manifests"/("ratio_" + tag + ".json")).write_text(
        json.dumps([p.name for p in sel]))
n30 = counts["hn30"]
sel_r = rbg[:min(n30, len(rbg))]
DATA_YAMLS["hn30rand"] = build_neg_dataset("hn30rand", [(p, "") for p in sel_r])
counts["hn30rand"] = len(sel_r)
(D_SM/"manifests"/"ratio_hn30rand.json").write_text(json.dumps([p.name for p in sel_r]))
save_decision("ratio_counts", counts)
print("  ✅ الداتاسيتات جاهزة:", {k: counts[k] for k in counts})


  C1) بناء داتاسيتات النسب
  فك home_fire.zip ...
  إيجابيات التدريب: 3900 | HN pool: 1638 | randbg pool: 2000
  ✅ داتاسيت hn15: 688 صورة إضافية → /content/sf/datasets/ds_hn15/data.yaml
  ✅ داتاسيت hn30: 1638 صورة إضافية → /content/sf/datasets/ds_hn30/data.yaml
  ✅ داتاسيت hn45: 1638 صورة إضافية → /content/sf/datasets/ds_hn45/data.yaml
  ✅ داتاسيت hn30rand: 1638 صورة إضافية → /content/sf/datasets/ds_hn30rand/data.yaml
  📌 قرار محفوظ: ratio_counts = {'hn15': 688, 'hn30': 1638, 'hn45': 1638, 'hn30rand': 1638}
  ✅ الداتاسيتات جاهزة: {'hn15': 688, 'hn30': 1638, 'hn45': 1638, 'hn30rand': 1638}


### C2 — تدريب النِسَب والضابط

تقييم النسب باستخدام النموذج الفائز وseed 42.


In [ ]:
assert "DATA_YAMLS" in globals(), "شغّلي C1 أولاً بنفس الجلسة"
WINNER = get_decision("winner")
assert WINNER, "شغّلي B3 أولاً (يحفظ الفائز)"
print("الفائز:", WINNER, "| نقطة الصفر = tournament__" + WINNER + "__s42")

for tag in ["hn15", "hn30", "hn45", "hn30rand"]:
    train_and_eval(tag, WINNER, 42, group="hnratio", data_yaml=DATA_YAMLS[tag],
                   notes="HN-ratio study; negatives=" + str(get_decision("ratio_counts", {}).get(tag)))
print("✅ C2 تمّت — شغّلي C4 لاختيار النسبة، وبعدها C3 للـ seeds الإضافية")

Streaming output truncated to the last 5000 lines.
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     76/300      6.34G      1.189     0.6277   0.007067         24        640: 100% ━━━━━━━━━━━━ 144/144 4.3it/s 33.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 5.7it/s 3.7s
                   all       1300       1580      0.912      0.893      0.933       0.62

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     77/300      6.34G      1.182     0.6257    0.00709         19        640: 100% ━━━━━━━━━━━━ 144/144 4.3it/s 33.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 5.6it/s 3.7s
                   all       1300       1580      0.921      0.897      0.934       0.61

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     78/300      6.34G      1.182     0.631

### C4 — اختيار النسبة الفائزة

الاختيار حسب Recall@FPR≤1% ثم mAP50 لكسر التعادل.


In [ ]:
assert "MASTER_CSV" in globals(), "شغّلي S1 أولاً"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"

dfh = master_df("hnratio")
dft = master_df("tournament")
base = dft[(dft["model"] == WINNER) & (dft["seed"] == 42)]
assert not base.empty, "رنّة البطولة الفائزة (s42) مو موجودة"
rows = [dict(tag="hn00", pct=0, **{c: base.iloc[0][c] for c in
        ["mAP50", "mAP5095", "recall", "f1", "recall_at_fpr1", "FPR25_hn", "FPR50_hn",
         "FPR50_novel", "FPR50_dfire"]})]
PCTS = {"hn15": 15, "hn30": 30, "hn45": 45, "hn30rand": 30}
for tag, pct in PCTS.items():
    r_ = dfh[(dfh["name"] == tag) & (dfh["seed"] == 42)]
    if r_.empty:
        continue
    rows.append(dict(tag=tag, pct=pct, **{c: r_.iloc[0][c] for c in
        ["mAP50", "mAP5095", "recall", "f1", "recall_at_fpr1", "FPR25_hn", "FPR50_hn",
         "FPR50_novel", "FPR50_dfire"]}))
tab = pd.DataFrame(rows).set_index("tag")
header("دراسة النسب (seed 42)")
print(tab.to_string())
tab.to_csv(D_RESULTS/"hn_ratio_study.csv")

cand = tab.drop(index=[t for t in ["hn30rand"] if t in tab.index])
best_tag = cand["recall_at_fpr1"].astype(float).idxmax()
save_decision("best_ratio", str(best_tag))
save_decision("neg_budget", int(get_decision("ratio_counts", {}).get(best_tag, 0)) if best_tag != "hn00" else 0)
header("🏆 النسبة الفائزة: " + str(best_tag))
if "hn30rand" in tab.index and "hn30" in tab.index:
    print("  الضابط: hn30rand (عشوائي) FPR50_hn=%s%% مقابل hn30 (صعب) %s%% — دليل أهمية الصعوبة"
          % (tab.loc["hn30rand", "FPR50_hn"], tab.loc["hn30", "FPR50_hn"]))
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    t2 = tab.drop(index=[t for t in ["hn30rand"] if t in tab.index])
    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(t2["pct"], t2["FPR50_hn"], "o-", color="crimson", label="FPR@0.5 (hard negatives)")
    ax1.set_xlabel("Hard-negative ratio (%)"); ax1.set_ylabel("FPR@0.5 (%)", color="crimson")
    ax2 = ax1.twinx()
    ax2.plot(t2["pct"], t2["mAP50"], "s--", color="navy", label="mAP50")
    ax2.set_ylabel("mAP50", color="navy")
    ax1.grid(alpha=.3); fig.suptitle("Effect of hard-negative ratio")
    fig.savefig(D_FIGS/"hn_ratio_curve.png", dpi=200, bbox_inches="tight")
    print("🖼", D_FIGS/"hn_ratio_curve.png")
except Exception as e:
    pass


  دراسة النسب (seed 42)
          pct   mAP50  mAP5095  recall      f1  recall_at_fpr1  FPR25_hn  FPR50_hn  FPR50_novel  FPR50_dfire
tag                                                                                                         
hn00        0  0.9049   0.5837  0.8651  0.8905          0.0000      35.6      27.6        18.37        10.80
hn15       15  0.9141   0.5934  0.8556  0.8942          0.7500       2.2       2.0         5.61        11.40
hn30       30  0.8991   0.5857  0.8387  0.8751          0.8191       1.8       1.0         4.59        10.87
hn45       45  0.8991   0.5857  0.8387  0.8751          0.8191       1.8       1.0         4.59        10.87
hn30rand   30  0.8865   0.5740  0.8345  0.8754          0.0000      11.2       9.4         5.10         4.67
  📌 قرار محفوظ: best_ratio = hn30
  📌 قرار محفوظ: neg_budget = 1638

  🏆 النسبة الفائزة: hn30
  الضابط: hn30rand (عشوائي) FPR50_hn=9.4% مقابل hn30 (صعب) 1.0% — دليل أهمية الصعوبة
🖼 /content/drive/MyDrive/SAFE_FIR

### C3 — التحقق عبر البذور

إعادة النسبة الفائزة مع seed 123 و2024 وحساب mean±std.


In [ ]:
assert "DATA_YAMLS" in globals(), "شغّلي C1 أولاً بنفس الجلسة"
WINNER = get_decision("winner"); BEST = get_decision("best_ratio")
assert WINNER and BEST, "شغّلي B3 ثم C2 ثم C4 أولاً"
assert BEST != "hn00", "النسبة الفائزة 0%؟ — ناقشيني قبل الاستمرار (غير متوقع)"
for seed in (123, 2024):
    train_and_eval(BEST, WINNER, seed, group="hnratio", data_yaml=DATA_YAMLS[BEST],
                   notes="best-ratio multi-seed")
sub = master_df("hnratio"); sub = sub[sub["name"] == BEST]
header("النسبة الفائزة %s عبر الـ seeds" % BEST)
print(sub[["seed", "mAP50", "mAP5095", "recall_at_fpr1", "FPR50_hn"]].to_string(index=False))
print("mean±std: mAP50=%.4f±%.4f | R@FPR1=%.4f±%.4f | FPR50_hn=%.2f±%.2f" % (
    sub["mAP50"].mean(), sub["mAP50"].std(), sub["recall_at_fpr1"].mean(),
    sub["recall_at_fpr1"].std(), sub["FPR50_hn"].mean(), sub["FPR50_hn"].std()))


######################################################################
  RUN: hnratio__hn30__s123
######################################################################
New https://pypi.org/project/ultralytics/8.4.95 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_hn30/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, 

## D — SAFE-Mine: تعدين السوالب

استخراج السوالب الصعبة، عنقدتها، وبناء عينات التدريب والتقييم.



### D1 — بناء مكتبة التعدين

دمج Open Images وPlaces365 بعد ترشيح محتوى النار.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("D1) بناء مكتبة التعدين (~76k صورة)")

FIRE_BLOCK = {"fire", "flame", "smoke", "explosion", "fireplace", "campfire",
              "bonfire", "wildfire", "firefighter", "fire engine"}
ALLOW = {"fire hydrant", "fire escape", "fire extinguisher", "firefly"}
def is_fire_label(lbl):
    l = str(lbl).lower().strip()
    if l in ALLOW:
        return False
    return (l in FIRE_BLOCK) or ("fire" in l) or ("flame" in l) or ("smoke" in l)

CORPUS = []

pipq("fiftyone")
import fiftyone as fo
import fiftyone.zoo as foz

oi_manifest = D_SM/"manifests"/"oi_corpus_ids.json"
dsn = "oi_corpus_" + datetime.now().strftime("%H%M%S")
if oi_manifest.exists():
    ids = json.loads(oi_manifest.read_text())
    oi = foz.load_zoo_dataset("open-images-v7", split="test",
                              label_types=["classifications"],
                              image_ids=ids, dataset_name=dsn)
else:
    oi = foz.load_zoo_dataset("open-images-v7", split="test",
                              label_types=["classifications"],
                              max_samples=40000, shuffle=True, seed=51,
                              dataset_name=dsn)
kept_ids = []
for s in oi.select_fields(["filepath", "positive_labels"]):
    labs = set()
    try:
        pl = s["positive_labels"]
        if pl is not None:
            labs = {c.label for c in pl.classifications}
    except Exception:
        pass
    if not any(is_fire_label(x) for x in labs):
        p = Path(s.filepath)
        CORPUS.append(p)
        kept_ids.append(p.stem)
if not oi_manifest.exists():
    oi_manifest.write_text(json.dumps(kept_ids))
print("  Open Images بعد فلتر النار:", len(kept_ids))

try:
    import tarfile, urllib.request
    proot = L_CORP/"places"
    if count_imgs(proot) < 30000:
        print("  تنزيل Places365 val_large (~2.1GB) ...")
        tar_p = L_CORP/"val_large.tar"
        urllib.request.urlretrieve("http://data.csail.mit.edu/places/places365/val_large.tar", tar_p)
        with tarfile.open(tar_p) as t:
            t.extractall(L_CORP)
        if proot.exists():
            shutil.rmtree(proot)
        (L_CORP/"val_large").rename(proot)
        tar_p.unlink()
        urllib.request.urlretrieve(
            "http://data.csail.mit.edu/places/places365/filelist_places365-standard.tar",
            L_CORP/"filelist.tar")
        with tarfile.open(L_CORP/"filelist.tar") as t:
            t.extractall(L_CORP)
    cats = {}
    for line in (L_CORP/"categories_places365.txt").read_text().splitlines():
        nm, idx = line.rsplit(" ", 1)
        cats[int(idx)] = nm
    def block_place(nm):
        n = nm.lower()
        if "fire_escape" in n:
            return False
        return any(w in n for w in ("volcano", "campsite", "fire"))
    val_lab = {}
    for line in (L_CORP/"places365_val.txt").read_text().splitlines():
        fn, idx = line.split()
        val_lab[fn] = int(idx)
    pk = [proot/fn for fn, i in val_lab.items()
          if not block_place(cats[i]) and (proot/fn).exists()]
    CORPUS += pk
    print("  Places365 بعد الفلتر:", len(pk))
except Exception as e:
    pass

print("  ✅ إجمالي المكتبة:", len(CORPUS), "صورة خالية من نار حقيقية")


  D1) بناء مكتبة التعدين (~76k صورة)


/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.zoo.datasets:Downloading split 'test' to '/root/fiftyone/open-images-v7/test' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv' to '/root/fiftyone/open-images-v7/test/metadata/image_ids.csv'


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv' to '/root/fiftyone/open-images-v7/test/metadata/classes.csv'


...

Dataset 'oi_corpus_061817' created


INFO:fiftyone.zoo.datasets:Dataset 'oi_corpus_061817' created


  Open Images بعد فلتر النار: 39983
  تنزيل Places365 val_large (~2.1GB) ...


/tmp/ipykernel_2391/1301199803.py:61: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(L_CORP)
/tmp/ipykernel_2391/1301199803.py:70: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(L_CORP)


  Places365 بعد الفلتر: 36200
  ✅ إجمالي المكتبة: 76183 صورة خالية من نار حقيقية


### D2 — تعدين السوالب الصعبة

مسح المكتبة بأوزان الفائز في الجولة صفر.


In [ ]:
assert "CORPUS" in globals() and len(CORPUS) > 10000, "شغّلي D1 أولاً بنفس الجلسة"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3 أولاً"

ROUND = 0
TAU_MINE = 0.10
MINE_W = D_WEIGHTS/("tournament__%s__s42__best.pt" % WINNER)
assert MINE_W.exists(), "أوزان الفائز مفقودة: " + str(MINE_W)
header("D2) التعدين round %d بأوزان %s" % (ROUND, MINE_W.name))

hits_csv = D_SM/"mining"/("round%d_hits.csv" % ROUND)
hits_zip = D_SM/"mining"/("round%d_hits.zip" % ROUND)
if hits_csv.exists() and hits_zip.exists():
    pass
else:
    import csv as _csv
    model = YOLO(str(MINE_W))
    rows, t0, B = [], time.time(), 64
    for i in range(0, len(CORPUS), B):
        chunk = [str(p) for p in CORPUS[i:i+B]]
        res = model.predict(chunk, conf=TAU_MINE, imgsz=640, verbose=False, save=False)
        for src, r in zip(chunk, res):
            mc = float(r.boxes.conf.max().item()) if len(r.boxes) else 0.0
            if mc >= TAU_MINE:
                rows.append((src, mc, len(r.boxes)))
        if (i//B) % 100 == 0:
            print("  %d/%d  hits=%d  (%.1f د)" % (i, len(CORPUS), len(rows), (time.time()-t0)/60))
    print("  انتهى المسح: %d hit من %d" % (len(rows), len(CORPUS)))
    with open(hits_csv, "w", newline="") as f_:
        w = _csv.writer(f_)
        w.writerow(["path", "name", "max_conf", "n_boxes"])
        for p, mc, nb in rows:
            w.writerow([p, Path(p).name, "%.4f" % mc, nb])
    yld = {("yield@%s" % t): round(100*sum(mc >= t for _, mc, _ in rows)/max(1, len(CORPUS)), 3)
           for t in (0.10, 0.25, 0.50)}
    yld.update(corpus=len(CORPUS), hits=len(rows), round=ROUND, weights=str(MINE_W),
               ts=datetime.now().isoformat(timespec="seconds"))
    (D_SM/"mining"/("round%d_yield.json" % ROUND)).write_text(json.dumps(yld, indent=2))
    print("  YIELD:", yld)
    seen = set()
    tmpz = L_TMP/hits_zip.name
    with zipfile.ZipFile(tmpz, "w", zipfile.ZIP_STORED) as z:
        for p, mc, nb in rows:
            n = Path(p).name
            if n in seen or not Path(p).exists():
                continue
            seen.add(n)
            z.write(p, arcname=n)
    shutil.move(str(tmpz), str(hits_zip))
    print("  ✅ محفوظ:", hits_zip, "(%.2f GB)" % (hits_zip.stat().st_size/1e9))


  D2) التعدين round 0 بأوزان tournament__yolo26n__s42__best.pt
  0/76183  hits=8  (0.0 د)
  6400/76183  hits=1048  (2.1 د)
  12800/76183  hits=2042  (4.1 د)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (102539736 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  size = 1 if ndim == 1 else shape[1], shape[0]


  19200/76183  hits=3099  (6.1 د)
  25600/76183  hits=4081  (8.1 د)
  32000/76183  hits=5162  (10.1 د)
  38400/76183  hits=6205  (12.1 د)
  44800/76183  hits=7184  (13.3 د)
  51200/76183  hits=8186  (14.4 د)
  57600/76183  hits=9191  (15.4 د)
  64000/76183  hits=10197  (16.5 د)
  70400/76183  hits=11135  (17.5 د)
  انتهى المسح: 12052 hit من 76183
  YIELD: {'yield@0.1': 15.82, 'yield@0.25': 10.493, 'yield@0.5': 6.154, 'corpus': 76183, 'hits': 12052, 'round': 0, 'weights': '/content/drive/MyDrive/SAFE_FIRE/weights/tournament__yolo26n__s42__best.pt', 'ts': '2026-07-15T07:16:46'}
  ✅ محفوظ: /content/drive/MyDrive/SAFE_FIRE/safemine_v2/mining/round0_hits.zip (2.37 GB)


### D3 — عنقدة CLIP وفحص التلوث

اختيار K بـsilhouette، تسمية العناقيد وفحص تلوث النار.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
ROUND = 0
header("D3) عنقدة CLIP — round %d" % ROUND)

hits_csv = D_SM/"mining"/("round%d_hits.csv" % ROUND)
hits_zip = D_SM/"mining"/("round%d_hits.zip" % ROUND)
assert hits_csv.exists() and hits_zip.exists(), "شغّلي D2 أولاً"

hit_dir = L_HITS/("round%d" % ROUND)
if count_imgs(hit_dir) == 0:
    robust_extract(hits_zip, hit_dir)
hrows = pd.read_csv(hits_csv)
recs = [(r["name"], float(r["max_conf"])) for _, r in hrows.iterrows()
        if (hit_dir/str(r["name"])).exists()]
print("  hits متاحة:", len(recs))

pipq("transformers", "scikit-learn")
import torch as _t
from PIL import Image as PImage
from transformers import CLIPModel, CLIPProcessor
dev = "cuda" if _t.cuda.is_available() else "cpu"
cm = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(dev).eval()
cp = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@_t.no_grad()
def img_embed(pil_list):
    inp = cp(images=pil_list, return_tensors="pt").to(dev)
    out = cm.get_image_features(**inp)
    if not _t.is_tensor(out):
        vo = cm.vision_model(pixel_values=inp["pixel_values"])
        pooled = getattr(vo, "pooler_output", None)
        if pooled is None:
            pooled = vo.last_hidden_state[:, 0]
        out = cm.visual_projection(pooled)
    return out

@_t.no_grad()
def txt_embed(prompts):
    ti = cp(text=prompts, return_tensors="pt", padding=True).to(dev)
    out = cm.get_text_features(**ti)
    if not _t.is_tensor(out):
        to = cm.text_model(input_ids=ti["input_ids"], attention_mask=ti.get("attention_mask"))
        pooled = getattr(to, "pooler_output", None)
        if pooled is None:
            pooled = to.last_hidden_state[:, 0]
        out = cm.text_projection(pooled)
    return out

cache = D_SM/"mining"/("round%d_clip.npz" % ROUND)
if cache.exists():
    z_ = np.load(cache, allow_pickle=True)
    feats, names = z_["feats"], [str(x) for x in z_["names"]]
else:
    feats, names, B = [], [], 128
    for i in range(0, len(recs), B):
        batch = recs[i:i+B]
        ims = [PImage.open(hit_dir/nm).convert("RGB") for nm, _ in batch]
        f_ = img_embed(ims)
        f_ = f_/f_.norm(dim=-1, keepdim=True)
        feats.append(f_.cpu().numpy().astype("float16"))
        names += [nm for nm, _ in batch]
        if (i//B) % 20 == 0:
            print("  CLIP %d/%d" % (i, len(recs)))
    feats = np.concatenate(feats)
    np.savez_compressed(cache, feats=feats, names=np.array(names, dtype=object))
confmap = dict(recs)
conf = np.array([confmap.get(n, 0.0) for n in names], dtype="float32")
print("  embeddings:", feats.shape)

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
Ff = feats.astype("float32")
sub = np.random.RandomState(42).choice(len(Ff), min(4000, len(Ff)), replace=False)
krows = []
for K in (6, 8, 10, 12, 16, 20):
    km_ = KMeans(n_clusters=K, random_state=42, n_init=5).fit(Ff[sub])
    sil = silhouette_score(Ff[sub], km_.labels_, sample_size=min(2000, len(sub)),
                           random_state=42)
    krows.append({"K": K, "silhouette": round(float(sil), 4)})
    print("  K=%-3d silhouette=%.4f" % (K, sil))
ktab = pd.DataFrame(krows)
ktab.to_csv(D_SM/("round%d_K_selection.csv" % ROUND), index=False)
K_BEST = int(ktab.loc[ktab["silhouette"].idxmax(), "K"])
save_decision("kmeans_K", K_BEST)
print("  🏆 K المختار:", K_BEST)

km = KMeans(n_clusters=K_BEST, random_state=42, n_init=10).fit(Ff)
lab = km.labels_

LABELS = ["lamp or light fixture", "light bulb", "lit candle", "chandelier",
          "sunset or orange sky", "fireplace", "neon or LED sign", "streetlight",
          "holiday string lights", "glowing screen or monitor",
          "bright window or indoor light", "reflection or glare",
          "orange or red clothing", "person", "red or orange flower",
          "autumn leaves", "food or fruit", "orange vehicle or object",
          "fireworks", "fog or steam"]
Lt = txt_embed(["a photo of " + x for x in LABELS]).cpu().numpy()
Lt = Lt/(np.linalg.norm(Lt, axis=1, keepdims=True) + 1e-9)
Fn = Ff/(np.linalg.norm(Ff, axis=1, keepdims=True) + 1e-9)
vote = (Fn @ Lt.T).argmax(1)
cname, cstats = {}, []
for c in range(K_BEST):
    ix = np.where(lab == c)[0]
    counts = np.bincount(vote[ix], minlength=len(LABELS))
    top = counts.argsort()[::-1][:3]
    cname[c] = LABELS[top[0]]
    mix = " / ".join("%s %d%%" % (LABELS[t], 100*counts[t]/len(ix)) for t in top if counts[t] > 0)
    cstats.append(dict(cluster=int(c), label=cname[c], n=int(len(ix)), mix=mix))
    print("  عنقود %2d | n=%4d | %s" % (c, len(ix), mix))
(D_SM/"mining"/("round%d_clusters.json" % ROUND)).write_text(
    json.dumps(cstats, indent=2, ensure_ascii=False))

FIRE_P = ["a real fire with large flames", "a house or building on fire",
          "a wildfire in a forest", "intense flames burning", "a car on fire"]
SAFE_P = ["a lamp", "a lit candle", "a sunset sky", "an orange object", "a person",
          "indoor lighting", "a glowing screen", "a plate of food", "a red flower",
          "a fireplace"]
Ft = txt_embed(["a photo of " + x for x in FIRE_P]).cpu().numpy()
Ft = Ft/(np.linalg.norm(Ft, axis=1, keepdims=True) + 1e-9)
St = txt_embed(["a photo of " + x for x in SAFE_P]).cpu().numpy()
St = St/(np.linalg.norm(St, axis=1, keepdims=True) + 1e-9)
fire_s = (Fn @ Ft.T).max(1)
safe_s = (Fn @ St.T).max(1)
margin = fire_s - safe_s
susp = [names[i] for i in np.where((margin > 0.05) & (fire_s > 0.25))[0]]
(D_SM/"manifests"/"fire_block.json").write_text(json.dumps(sorted(susp)))
print("  مشتبه نار حقيقية (محظورة من العينات):", len(susp))

gal = D_SM/"figures"/("round%d_clusters" % ROUND)
gal.mkdir(parents=True, exist_ok=True)
def sheet(files, out, cols=4, cell=200):
    n = len(files)
    s_ = PImage.new("RGB", (cols*cell, max(1, math.ceil(n/cols))*cell), "white")
    for i, f_ in enumerate(files):
        im = PImage.open(f_).convert("RGB")
        im.thumbnail((cell, cell))
        s_.paste(im, ((i % cols)*cell, (i//cols)*cell))
    s_.save(out)
for c in range(K_BEST):
    ix = np.where(lab == c)[0]
    top = ix[np.argsort(-conf[ix])][:12]
    sheet([hit_dir/names[i] for i in top],
          gal/("c%02d__%s__n%d.jpg" % (c, cname[c].replace(" ", "_"), len(ix))))
print("  🖼 معارض العناقيد:", gal, "— عايني بعينج، أي عنقود نار حقيقية بلّغيني")

np.save(L_TMP/"d3_lab.npy", lab)
(L_TMP/"d3_names.json").write_text(json.dumps(names))
np.save(L_TMP/"d3_conf.npy", conf)
print("✅ D3 تمّت — التالي D4")


  D3) عنقدة CLIP — round 0
  hits متاحة: 12052


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

  embeddings: (12052, 512)
  K=6   silhouette=0.0422
  K=8   silhouette=0.0395
  K=10  silhouette=0.0428
  K=12  silhouette=0.0464
  K=16  silhouette=0.0476
  K=20  silhouette=0.0457
  📌 قرار محفوظ: kmeans_K = 16
  🏆 K المختار: 16
  عنقود  0 | n= 775 | reflection or glare 32% / person 29% / orange or red clothing 17%
  عنقود  1 | n=1108 | reflection or glare 33% / orange or red clothing 15% / fog or steam 14%
  عنقود  2 | n= 681 | orange vehicle or object 31% / reflection or glare 20% / fireplace 11%
  عنقود  3 | n= 626 | bright window or indoor light 31% / fireplace 19% / lamp or light fixture 17%
  عنقود  4 | n= 624 | red or orange flower 55% / food or fruit 15% / reflection or glare 6%
  عنقود  5 | n= 871 | person 40% / orange or red clothing 16% / glowing screen or monitor 8%
  عنقود  6 | n= 664 | person 63% / reflection or glare 15% / orange or red clothing 7%
  عنقود  7 | n= 693 | person 36% / orange or red clothing 21% / reflection or glare 16%
  عنقود  8 | n=1092 | glowing scre

### D4 — بناء عينات التعدين

إنشاء dev_neg وعينتي coverage وtop-confidence وداتاسيتاتهما.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
ROUND = 0
import collections
lab = np.load(L_TMP/"d3_lab.npy")
names = json.loads((L_TMP/"d3_names.json").read_text())
conf = np.load(L_TMP/"d3_conf.npy")
hit_dir = L_HITS/("round%d" % ROUND)
assert count_imgs(hit_dir) > 100, "شغّلي D3 أولاً بنفس الجلسة"
K_BEST = int(get_decision("kmeans_K", 12))
fire_block = set(json.loads((D_SM/"manifests"/"fire_block.json").read_text()))
header("D4) العينات — round %d (K=%d)" % (ROUND, K_BEST))

dev_manifest = D_SM/"manifests"/"dev_neg.json"
if not dev_manifest.exists():
    rng = np.random.RandomState(42)
    devl = []
    per = max(1, 400//K_BEST)
    for c in range(K_BEST):
        pool_ = [names[i] for i in np.where(lab == c)[0] if names[i] not in fire_block]
        rng.shuffle(pool_)
        devl += pool_[:per]
    devl = devl[:400]
    dev_manifest.write_text(json.dumps(devl))
dev_set = set(json.loads(dev_manifest.read_text()))
if not NEG_ZIPS["mined_dev"].exists():
    zip_flat([hit_dir/n for n in sorted(dev_set) if (hit_dir/n).exists()],
             NEG_ZIPS["mined_dev"])

    man = json.loads((D_SUITE/"manifest.json").read_text())
    man["negatives"]["mined_dev"] = {"zip": NEG_ZIPS["mined_dev"].name,
        "n": zip_count(NEG_ZIPS["mined_dev"]),
        "size_MB": round(NEG_ZIPS["mined_dev"].stat().st_size/1e6, 1),
        "md5": file_md5(NEG_ZIPS["mined_dev"])}
    (D_SUITE/"manifest.json").write_text(json.dumps(man, indent=2, ensure_ascii=False))
print("  dev_neg:", len(dev_set), "صورة (مضافة للـ Suite)")


BUDGET = int(get_decision("neg_budget", 0))
assert BUDGET > 0, "شغّلي C4 أولاً (يحفظ neg_budget)"
print("  الميزانية:", BUDGET)

used = set(dev_set) | fire_block
elig = [i for i in range(len(names)) if names[i] not in used]
by_c = collections.defaultdict(list)
for i in elig:
    by_c[int(lab[i])].append(i)
wts = {c: math.sqrt(len(ix))*float(np.mean(conf[ix])) for c, ix in by_c.items()}
tot = sum(wts.values()) + 1e-9
rng = np.random.RandomState(42)
sel = []
for c, ix in by_c.items():
    q = min(len(ix), max(20, round(BUDGET*wts[c]/tot)))
    sel += list(rng.choice(ix, size=q, replace=False))
if len(sel) > BUDGET:
    rng.shuffle(sel); sel = sel[:BUDGET]
elif len(sel) < BUDGET:
    left = sorted(set(elig) - set(sel), key=lambda i: -conf[i])
    sel += left[:BUDGET - len(sel)]
cov_names = [names[i] for i in sel]
top_names = [names[i] for i in sorted(elig, key=lambda i: -conf[i])[:BUDGET]]
(D_SM/"manifests"/("round%d_cov.json" % ROUND)).write_text(json.dumps(cov_names))
(D_SM/"manifests"/("round%d_top.json" % ROUND)).write_text(json.dumps(top_names))
print("  coverage=%d | top-conf=%d (متقاطعات: %d)" %
      (len(cov_names), len(top_names), len(set(cov_names) & set(top_names))))

SM_YAMLS = {
    "sm_cov": build_neg_dataset("sm_cov", [(hit_dir/n, "") for n in cov_names]),
    "sm_top": build_neg_dataset("sm_top", [(hit_dir/n, "") for n in top_names]),
}
print("✅ D4 تمّت — التالي D5 (backfill) ثم المرحلة E")


  D4) العينات — round 0 (K=16)
  dev_neg: 400 صورة (مضافة للـ Suite)
  الميزانية: 1638
  coverage=1638 | top-conf=1638 (متقاطعات: 236)
  فك home_fire.zip ...
  ✅ داتاسيت sm_cov: 1638 صورة إضافية → /content/sf/datasets/ds_sm_cov/data.yaml
  ✅ داتاسيت sm_top: 1638 صورة إضافية → /content/sf/datasets/ds_sm_top/data.yaml
✅ D4 تمّت — التالي D5 (backfill) ثم المرحلة E


### D5 — استكمال تقييم mined_dev

تقييم الرنّات السابقة على mined_dev دون تدريب.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
assert NEG_ZIPS["mined_dev"].exists(), "شغّلي D4 أولاً"
header("D5) backfill mined_dev")
suite = ensure_suite_local(strict=True)
md_dir = suite["mined_dev"]
df = master_df()
done_n = todo_n = 0
for _, r_ in df.iterrows():
    run_id = r_["run_id"]
    w = Path(str(r_["weights"]))
    sc = D_SCORES/(run_id + "__mined_dev.csv")
    if sc.exists():
        done_n += 1
        continue
    if not w.exists():

        continue
    print("  مسح:", run_id)
    scan_max_conf(YOLO(str(w)), md_dir, sc)
    todo_n += 1
print("✅ D5: مكتمل مسبقاً=%d | مُسح الآن=%d" % (done_n, todo_n))
print("   (تقارير المرحلة G تحسب FPR مباشرة من ملفات الـ scores)")


  D5) backfill mined_dev
  مسح: tournament__yolov8n__s42
  مسح: tournament__yolo11n__s42
  مسح: tournament__yolo26n__s42
  مسح: tournament__yolov8s__s42
  مسح: tournament__yolo11s__s42
  مسح: tournament__yolo26s__s42
  مسح: tournament__yolov8n__s123
  مسح: tournament__yolo11n__s123
  مسح: tournament__yolo26n__s123
  مسح: tournament__yolov8s__s123
  مسح: tournament__yolo11s__s123
  مسح: tournament__yolo26s__s123
  مسح: tournament__yolov8n__s2024
  مسح: tournament__yolo11n__s2024
  مسح: tournament__yolo26n__s2024
  مسح: tournament__yolov8s__s2024
  مسح: tournament__yolo11s__s2024
  مسح: tournament__yolo26s__s2024
  مسح: hnratio__hn15__s42
  مسح: hnratio__hn30__s42
  مسح: hnratio__hn45__s42
  مسح: hnratio__hn30rand__s42
  مسح: hnratio__hn30__s123
  مسح: hnratio__hn30__s2024
✅ D5: مكتمل مسبقاً=0 | مُسح الآن=24
   (تقارير المرحلة G تحسب FPR مباشرة من ملفات الـ scores)


## E — تجارب SAFE-Mine

مقارنة استراتيجيات أخذ العينات، خسارة الخلفية، والإعداد النهائي متعدد البذور.



### E1 — Coverage مقابل Top-confidence

مقارنة استراتيجيتَي أخذ العينات دون خسارة إضافية.


In [ ]:
assert "SM_YAMLS" in globals(), "شغّلي D4 أولاً بنفس الجلسة (أو أعيدي D3+D4 لإعادة البناء)"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"
for tag in ["sm_cov", "sm_top"]:
    train_and_eval(tag, WINNER, 42, group="safemine", data_yaml=SM_YAMLS[tag],
                   notes="mined negatives; sampling=" + tag)
sub = master_df("safemine")
cols = ["name", "seed", "mAP50", "recall_at_fpr1", "FPR50_hn", "FPR50_novel", "FPR50_mined"]
header("coverage ضد top-conf (وقارنيهم مع النسبة اليدوية الفائزة من C)")
print(sub[sub["seed"] == 42][cols].to_string(index=False))

✅ safemine__sm_cov__s42 منجز — تخطّي

######################################################################
  RUN: safemine__sm_top__s42
######################################################################
New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_sm_top/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, h

### E2 — خسارة كبح الخلفية

فحص sanity ثم دراسة λ بقيم 0.5 و1 و2.


In [ ]:
assert "SM_YAMLS" in globals(), "شغّلي D4 أولاً بنفس الجلسة"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"

if not get_decision("lbg_sanity_ok", False):
    print("🧪 SANITY: تدريب 2-epoch مع λ=1 للتأكد من ربط الخسارة ...")
    train_and_eval("lbg_sanity", WINNER, 42, group="_sanity",
                   data_yaml=SM_YAMLS["sm_cov"], lam=1.0, sanity=True)
    save_decision("lbg_sanity_ok", True)
    print("✅ الربط سليم (شفتي قيم L_bg تنطبع وتنزل) — نكمل بالتدريبات الكاملة")

for lamv, tag in [(0.5, "lbg05"), (1.0, "lbg10"), (2.0, "lbg20")]:
    train_and_eval(tag, WINNER, 42, group="safemine", data_yaml=SM_YAMLS["sm_cov"],
                   lam=lamv, notes="L_bg ablation lam=%s (gamma=2)" % lamv)
sub = master_df("safemine"); sub = sub[sub["seed"] == 42]
header("جدول λ (الصف lam=0 هو sm_cov)")
print(sub[["name", "lam", "mAP50", "recall_at_fpr1", "FPR50_hn", "FPR50_mined"]]
      .to_string(index=False))
print("القرار: إذا ماكو تحسن واضح بالخسارة → data-only أبسط وأفضل (نتيجة تُنشر بالحالتين)")

🧪 SANITY: تدريب 2-epoch مع λ=1 للتأكد من ربط الخسارة ...

######################################################################
  SANITY: _sanity__lbg_sanity__s42
######################################################################
New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_sm_cov/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

### E3 — الحسم متعدد البذور والمعماريات

تأكيد المرشحين المتقاربين وحسم الأفضل بالمتوسط.


In [ ]:
assert "SM_YAMLS" in globals(), "شغّلي D3 ثم D4 أولاً بنفس الجلسة"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"

sub = master_df("safemine"); sub = sub[sub["seed"] == 42]
assert not sub.empty, "شغّلي E1/E2 أولاً"
top = sub.sort_values("recall_at_fpr1", ascending=False)
gap = float(top.iloc[0]["recall_at_fpr1"]) - float(top.iloc[1]["recall_at_fpr1"])
fin = top.head(2) if gap < 0.03 else top.head(1)
print("فجوة الصدارة = %.4f → %s" % (gap, "مرشحان (تعادل إحصائي)" if len(fin) > 1 else "مرشح واحد"))
print(fin[["name", "lam", "mAP50", "recall_at_fpr1", "FPR50_hn", "FPR50_mined"]].to_string(index=False))

FINALISTS = []
for _, r_ in fin.iterrows():
    lam_v = pd.to_numeric(r_["lam"], errors="coerce")
    lam_ = float(lam_v) if pd.notna(lam_v) else 0.0
    ds_ = "sm_top" if str(r_["name"]) == "sm_top" else "sm_cov"
    FINALISTS.append((str(r_["name"]), ds_, lam_))

for name, ds_, lam_ in FINALISTS:
    for seed in (123, 2024):
        train_and_eval(name, WINNER, seed, group="safemine",
                       data_yaml=SM_YAMLS[ds_], lam=lam_, notes="finalist multi-seed")

sub = master_df("safemine")
cand_names = [f[0] for f in FINALISTS]
means = (sub[sub["name"].isin(cand_names)]
         .groupby("name")[["mAP50", "recall_at_fpr1", "FPR50_hn", "FPR50_mined"]]
         .agg(["mean", "std"]).round(4))
header("حسم المرشحين (متوسط الـ seeds)")
print(means.to_string())
m_r = sub[sub["name"].isin(cand_names)].groupby("name")["recall_at_fpr1"].mean()
m_a = sub[sub["name"].isin(cand_names)].groupby("name")["mAP50"].mean()
best_name = max(cand_names, key=lambda n: (round(float(m_r[n]), 3), round(float(m_a[n]), 4)))
BEST = {n: (d, l) for n, d, l in FINALISTS}[best_name]
save_decision("best_sm", best_name)
header("🏆 إعداد SAFE-Mine الفائز: %s (dataset=%s, lam=%s)" % (best_name, BEST[0], BEST[1]))


others = [m for m in ["yolov8n", "yolo11n", "yolo26n"] if m != WINNER][:2]
for arch in others:
    train_and_eval(best_name + "_" + arch, arch, 42, group="safemine",
                   data_yaml=SM_YAMLS[BEST[0]], lam=BEST[1], notes="cross-architecture")

sub = master_df("safemine")
header("SAFE-Mine النهائي (كل الصفوف)")
print(sub[["name", "model", "seed", "lam", "mAP50", "recall_at_fpr1",
           "FPR50_hn", "FPR50_novel", "FPR50_mined"]].to_string(index=False))

فجوة الصدارة = 0.0281 → مرشحان (تعادل إحصائي)
  name  lam  mAP50  recall_at_fpr1  FPR50_hn  FPR50_mined
sm_top  0.0 0.8871          0.7516       1.4         1.50
 lbg20  2.0 0.9110          0.7235       1.6         0.25
✅ safemine__sm_top__s123 منجز — تخطّي
✅ safemine__sm_top__s2024 منجز — تخطّي
✅ safemine__lbg20__s123 منجز — تخطّي

######################################################################
  RUN: safemine__lbg20__s2024
######################################################################
New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/dat

### E4 — جولة تعدين ثانية (اختيارية)

تُشغَّل في جلسة مستقلة وفق قاعدة التوقف المعلنة.


In [ ]:
assert "CORPUS" in globals() and len(CORPUS) > 10000, "هذي الخلية تحتاج D1 بنفس الجلسة"
WINNER = get_decision("winner"); BEST_SM = get_decision("best_sm")
assert BEST_SM, "شغّلي E3 أولاً"
ROUND = 1
MINE_W = D_WEIGHTS/("safemine__%s__s42__best.pt" % BEST_SM)
assert MINE_W.exists(), "أوزان best_sm مفقودة: " + str(MINE_W)
header("E4) جولة تعدين %d بأوزان %s" % (ROUND, MINE_W.name))

## F — تحسين الدقة واختيار البطل

اختيار البطل أولاً، ثم تقييم الكاسكيد ووسائل تحسين الاستدلال.



### F1′ — منحنى الدقة

مقارنة دقات الإدخال 640 و768 و960.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3 أولاً"

train_and_eval("res768", WINNER, 42, group="accuracy", imgsz=768, batch=24,
               notes="resolution ablation @768")
train_and_eval("res960", WINNER, 42, group="accuracy", imgsz=960, batch=16,
               notes="resolution ablation @960")

COLS = ["mAP50", "mAP5095", "recall", "flame_AP50", "smoke_AP50",
        "recall_at_fpr1", "FPR50_hn", "GFLOPs"]
dft = master_df("tournament")
dfa = master_df("accuracy")
rows = []
b = dft[(dft["model"] == WINNER) & (dft["seed"] == 42)]
if not b.empty:
    rows.append(dict(res=640, **{c: pd.to_numeric(b.iloc[0][c], errors="coerce") for c in COLS}))
for tag, r_ in [("res768", 768), ("res960", 960)]:
    rr = dfa[(dfa["name"] == tag) & (dfa["seed"] == 42)]
    if not rr.empty:
        rows.append(dict(res=r_, **{c: pd.to_numeric(rr.iloc[0][c], errors="coerce") for c in COLS}))
tab = pd.DataFrame(rows).set_index("res")
header("منحنى الدقّة — " + str(WINNER))
print(tab.to_string())
tab.to_csv(D_RESULTS/"resolution_curve.csv")

best_res = int(tab["recall_at_fpr1"].astype(float).idxmax())
save_decision("best_res", best_res)
print()
print("🏆 أفضل دقّة تشغيلية (بقاعدة Recall@FPR≤1%):", best_res)
print("   ملاحظة: الكاسكيد (F2') يستخدم 640 كمسار سريع و960 كمسار تحقق بغض النظر")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(tab.index, tab["mAP50"], "o-", color="navy", label="mAP50")
    ax1.set_xlabel("Input resolution"); ax1.set_ylabel("mAP50", color="navy")
    ax2 = ax1.twinx()
    ax2.plot(tab.index, tab["GFLOPs"], "s--", color="gray", label="GFLOPs")
    ax2.set_ylabel("GFLOPs", color="gray")
    ax1.grid(alpha=.3); fig.suptitle("Accuracy vs resolution (" + str(WINNER) + ")")
    fig.savefig(D_FIGS/"resolution_curve.png", dpi=200, bbox_inches="tight")
    print("🖼", D_FIGS/"resolution_curve.png")
except Exception as e:
    pass


######################################################################
  RUN: accuracy__res768__s42
######################################################################
  فك home_fire.zip ...
New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/home_fire/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_

### F5′ — اختيار البطل النهائي

مقارنة hn30 وsm_top واتحادهما combo بقواعد حسم معلنة.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"
header("F5' v3 — الاتحاد ثم الحسم")

pool_dir = L_DS/"hn_pool"
if count_imgs(pool_dir) == 0:
    robust_extract(D_TRAINNEG/"hn_train_pool.zip", pool_dir)
hit_dir = L_HITS/"round0"
if count_imgs(hit_dir) == 0:
    hz = D_SM/"mining"/"round0_hits.zip"
    assert hz.exists(), "round0_hits.zip مفقود"
    robust_extract(hz, hit_dir)
hn_names = json.loads((D_SM/"manifests"/"ratio_hn30.json").read_text())
top_names = json.loads((D_SM/"manifests"/"round0_top.json").read_text())
items, seen = [], set()
for p in [pool_dir/n for n in hn_names] + [hit_dir/n for n in top_names]:
    if p.name in seen:
        continue
    seen.add(p.name)
    items.append((p, ""))
combo_yaml = build_neg_dataset("combo", items)
print("سوالب الاتحاد:", len(items), "(يدوي %d + منقّب %d قبل الدمج)" % (len(hn_names), len(top_names)))


train_and_eval("combo", WINNER, 42, group="hero", data_yaml=combo_yaml,
               notes="union(hn30, sm_top) negatives=%d" % len(items))

df = master_df()
def rows_of(g, n):
    return df[(df["group"] == g) & (df["name"] == n) & (df["model"] == WINNER)]
CANDS = {"hn30": ("hnratio", rows_of("hnratio", "hn30")),
         "sm_top": ("safemine", rows_of("safemine", "sm_top")),
         "combo": ("hero", rows_of("hero", "combo"))}
summ = []
for n, (g_, rr) in CANDS.items():
    summ.append(dict(name=n, seeds=len(rr),
                     R_at_FPR1=round(rr["recall_at_fpr1"].mean(), 4),
                     mAP50=round(rr["mAP50"].mean(), 4),
                     FPR50_hn=round(rr["FPR50_hn"].mean(), 2),
                     FPR50_novel=round(rr["FPR50_novel"].mean(), 2),
                     FPR50_mined=round(rr["FPR50_mined"].mean(), 2)))
st = pd.DataFrame(summ).set_index("name")
header("الحسم (متوسط الـ seeds المتوفرة)")
print(st.to_string())

best_r = float(st["R_at_FPR1"].max())
finalists = [n for n in st.index if best_r - float(st.loc[n, "R_at_FPR1"]) <= 0.05]
print("متعادلون على R@FPR1 (ضمن 0.05):", finalists)
if len(finalists) > 1:
    fin2 = st.loc[finalists].sort_values(["FPR50_mined", "mAP50"], ascending=[True, False])
    HERO_NAME = str(fin2.index[0])
    print("الحسم بالعالم المفتوح (FPR50_mined) ثم mAP50 →", HERO_NAME)
else:
    HERO_NAME = finalists[0]
HERO_GRP = CANDS[HERO_NAME][0]
header("🏆 البطل: " + HERO_NAME)

if HERO_NAME == "combo":
    for seed in (123, 2024):
        train_and_eval("combo", WINNER, seed, group="hero", data_yaml=combo_yaml,
                       notes="hero multi-seed")
save_decision("hero", {"group": HERO_GRP, "name": HERO_NAME, "model": WINNER, "imgsz": 640})

df = master_df()
hero_rows = df[(df["group"] == HERO_GRP) & (df["name"] == HERO_NAME) & (df["model"] == WINNER)]
header("🏁 البطل النهائي (mean±std)")
cols = ["seed", "mAP50", "mAP5095", "recall", "recall_at_fpr1",
        "FPR50_hn", "FPR50_novel", "FPR50_mined"]
print(hero_rows[cols].to_string(index=False))
print()
print("mAP50 = %.4f ± %.4f | R@FPR1 = %.4f ± %.4f | FPR_hn = %.2f%% | FPR_mined = %.2f%%"
      % (hero_rows["mAP50"].mean(), hero_rows["mAP50"].std(),
         hero_rows["recall_at_fpr1"].mean(), hero_rows["recall_at_fpr1"].std(),
         hero_rows["FPR50_hn"].mean(), hero_rows["FPR50_mined"].mean()))


  F5' v3 — الاتحاد ثم الحسم
  فك home_fire.zip ...
  ✅ داتاسيت combo: 3276 صورة إضافية → /content/sf/datasets/ds_combo/data.yaml
سوالب الاتحاد: 3276 (يدوي 1638 + منقّب 1638 قبل الدمج)

######################################################################
  RUN: hero__combo__s42
######################################################################
New https://pypi.org/project/ultralytics/8.4.98 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_combo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=Non

### F2′a — ضبط بوابة الكاسكيد

ضبط مسار 640 السريع ومسار 768 للتحقق على val/dev.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
hero = get_decision("hero"); assert hero, "شغّلي F5' أولاً"
HW = D_WEIGHTS/("%s__%s__s42__best.pt" % (hero["group"], hero["name"]))
assert HW.exists(), "أوزان البطل مفقودة: " + str(HW)
print("أوزان البطل:", HW.name, "| سريع @640 | تحقق @768 (نفس الأوزان — استدلال فقط)")

data_yaml = ensure_home_fire_local(); hf = Path(data_yaml).parent
suite = ensure_suite_local(strict=True); assert "mined_dev" in suite
m = YOLO(str(HW))
val_dir = hf/"val"/"images"
tag = hero["name"]
fv = scan_max_conf(m, val_dir, D_SCORES/("casc3__%s__f__val.csv" % tag), imgsz=640)
av = scan_max_conf(m, val_dir, D_SCORES/("casc3__%s__a__val.csv" % tag), imgsz=768)
fd = scan_max_conf(m, suite["mined_dev"], D_SCORES/("casc3__%s__f__dev.csv" % tag), imgsz=640)
ad = scan_max_conf(m, suite["mined_dev"], D_SCORES/("casc3__%s__a__dev.csv" % tag), imgsz=768)

def cascade_scores(f, a2, lo, hi):
    band = (f > lo) & (f < hi)
    out = f.copy(); out[band] = a2[band]
    return out, float(band.mean())

def r_at_fpr1_img(pos, neg):
    best = 0.0
    for t in np.linspace(0.01, 0.95, 189):
        if 100*float((neg >= t).mean()) <= 1.0:
            best = max(best, float((pos >= t).mean()))
    return round(best, 4)

base_fast = r_at_fpr1_img(fv, fd)
base_acc = r_at_fpr1_img(av, ad)
rows = []
for lo in (0.05, 0.10, 0.15):
    for hi in (0.40, 0.50, 0.60, 0.70):
        pv, ev = cascade_scores(fv, av, lo, hi)
        nv, en = cascade_scores(fd, ad, lo, hi)
        rows.append({"t_lo": lo, "t_hi": hi, "R@FPR1": r_at_fpr1_img(pv, nv),
                     "esc_pos%": round(100*ev, 1), "esc_neg%": round(100*en, 1)})
tab = pd.DataFrame(rows)
header("ضبط البوابة (val ضد mined_dev) — مرجع: fast=%s | acc768=%s" % (base_fast, base_acc))
print(tab.to_string(index=False))
tab.to_csv(D_RESULTS/"cascade_gate_tuning.csv", index=False)
best = tab.sort_values(["R@FPR1", "esc_neg%"], ascending=[False, True]).iloc[0]
gain = float(best["R@FPR1"]) - base_fast
print()
if gain < 0.005:
    save_decision("cascade_ok", False)
    header("❌ الكاسكيد ما يضيف (%.4f) — نتيجة سلبية موثقة، لا تشغلين F2'b" % gain)
else:
    save_decision("cascade_ok", True)
    save_decision("cascade_gate", [float(best["t_lo"]), float(best["t_hi"])])
    save_decision("cascade_fast", str(HW))
    save_decision("cascade_acc", str(HW))
    header("✅ الكاسكيد يضيف +%.4f — البوابة [%s, %s] → كملي F2'b" % (gain, best["t_lo"], best["t_hi"]))

أوزان البطل: hero__combo__s42__best.pt | سريع @640 | تحقق @768 (نفس الأوزان — استدلال فقط)

  ضبط البوابة (val ضد mined_dev) — مرجع: fast=0.9146 | acc768=0.8877
 t_lo  t_hi  R@FPR1  esc_pos%  esc_neg%
 0.05   0.4  0.8785       3.6       5.0
 0.05   0.5  0.8885       4.9       5.2
 0.05   0.6  0.9015       6.8       5.5
 0.05   0.7  0.9331      11.4       5.8
 0.10   0.4  0.8777       3.3       3.0
 0.10   0.5  0.8877       4.6       3.2
 0.10   0.6  0.9008       6.5       3.5
 0.10   0.7  0.9315      11.1       3.8
 0.15   0.4  0.8762       2.5       2.8
 0.15   0.5  0.8862       3.8       3.0
 0.15   0.6  0.8992       5.8       3.2
 0.15   0.7  0.9300      10.3       3.5

  📌 قرار محفوظ: cascade_ok = True
  📌 قرار محفوظ: cascade_gate = [0.05, 0.7]
  📌 قرار محفوظ: cascade_fast = /content/drive/MyDrive/SAFE_FIRE/weights/hero__combo__s42__best.pt
  📌 قرار محفوظ: cascade_acc = /content/drive/MyDrive/SAFE_FIRE/weights/hero__combo__s42__best.pt

  ✅ الكاسكيد يضيف +0.0185 — البوابة [0.05, 0.

### F2′b — تقييم الكاسكيد

التقييم النهائي على حزمة الاختبار ورسم المقايضة.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
gate = get_decision("cascade_gate"); assert gate, "شغّلي F2'a أولاً"
lo, hi = float(gate[0]), float(gate[1])
FAST_W = Path(get_decision("cascade_fast")); ACC_W = Path(get_decision("cascade_acc"))
data_yaml = ensure_home_fire_local(); hf = Path(data_yaml).parent
suite = ensure_suite_local(strict=True)
mfast, macc = YOLO(str(FAST_W)), YOLO(str(ACC_W))

def cascade_scores(f, a2):
    band = (f > lo) & (f < hi)
    out = f.copy(); out[band] = a2[band]
    return out, float(band.mean())

test_dir = hf/"test"/"images"
S = {}
S["test"] = (scan_max_conf(mfast, test_dir, D_SCORES/"cascade__fast__test_pos.csv", imgsz=640),
             scan_max_conf(macc,  test_dir, D_SCORES/"cascade__acc__test_pos.csv",  imgsz=960))
for name, d in suite.items():
    S[name] = (scan_max_conf(mfast, d, D_SCORES/("cascade__fast__" + name + ".csv"), imgsz=640),
               scan_max_conf(macc,  d, D_SCORES/("cascade__acc__" + name + ".csv"),  imgsz=960))

def r_at_fpr1_img(pos, neg):
    best = 0.0
    for t in np.linspace(0.01, 0.95, 189):
        if 100*float((neg >= t).mean()) <= 1.0:
            best = max(best, float((pos >= t).mean()))
    return round(best, 4)

systems = {
    "fast_only": {k: v[0] for k, v in S.items()},
    "acc_only":  {k: v[1] for k, v in S.items()},
}
casc, esc = {}, {}
for k, (f, a2) in S.items():
    casc[k], esc[k] = cascade_scores(f, a2)
systems["cascade"] = casc

rows = []
for sysname, sc in systems.items():
    row = {"system": sysname,
           "ImgRecall@FPR1_hn": r_at_fpr1_img(sc["test"], sc["hn_holdout"])}
    for neg in ["hn_holdout", "novel", "dfire_neg", "mined_dev"]:
        if neg in sc:
            row["FPR50_" + neg] = round(100*float((sc[neg] >= 0.5).mean()), 2)
    if sysname == "cascade":
        row["esc_test%"] = round(100*esc["test"], 1)
        row["esc_hn%"] = round(100*esc["hn_holdout"], 1)
    rows.append(row)
tab = pd.DataFrame(rows)
header("الكاسكيد على الـ Suite (مستوى الصورة/الإنذار) — البوابة [%.2f, %.2f]" % (lo, hi))
print(tab.to_string(index=False))
tab.to_csv(D_RESULTS/"cascade_final.csv", index=False)

ratio = (960.0/640.0)**2
avg_cost = 1 + esc["test"]*(ratio - 1)
print()
print("تقدير كلفة الكاسكيد ≈ %.2f× كلفة 640 (نسبة تصعيد إيجابيات %.1f%%؛ على مشاهد فارغة واقعية أوطأ بكثير)"
      % (avg_cost, 100*esc["test"]))
print("للرسالة: صناديق الإطارات المُصعَّدة = مخرجات موديل 960 (جودتها بجدول F1')")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    for sysname, sc in systems.items():
        xs, ys = [], []
        for t in np.linspace(0.01, 0.95, 100):
            xs.append(float((sc["test"] >= t).mean()))
            ys.append(100*float((sc["hn_holdout"] >= t).mean()))
        pts = sorted(zip(xs, ys))
        ax.plot([p[0] for p in pts], [p[1] for p in pts], label=sysname)
    ax.set_xlabel("Image-level recall (test)")
    ax.set_ylabel("FPR on hn_holdout (%)")
    ax.set_ylim(-1, 30); ax.legend(); ax.grid(alpha=.3)
    ax.set_title("Confidence-Gated Dual-Resolution Cascade")
    fig.savefig(D_FIGS/"cascade_pareto.png", dpi=200, bbox_inches="tight")
    print("🖼", D_FIGS/"cascade_pareto.png")
except Exception as e:
    pass

  فك home_fire.zip ...

  الكاسكيد على الـ Suite (مستوى الصورة/الإنذار) — البوابة [0.05, 0.70]
   system  ImgRecall@FPR1_hn  FPR50_hn_holdout  FPR50_novel  FPR50_dfire_neg  FPR50_mined_dev  esc_test%  esc_hn%
fast_only             0.8846               0.8         2.04             3.60             1.25        NaN      NaN
 acc_only             0.9108               1.0         1.02             4.47             1.25        NaN      NaN
  cascade             0.9277               0.6         1.02             4.13             1.00       19.6      3.0

تقدير كلفة الكاسكيد ≈ 1.25× كلفة 640 (نسبة تصعيد إيجابيات 19.6%؛ على مشاهد فارغة واقعية أوطأ بكثير)
للرسالة: صناديق الإطارات المُصعَّدة = مخرجات موديل 960 (جودتها بجدول F1')
🖼 /content/drive/MyDrive/SAFE_FIRE/figures_v2/cascade_pareto.png


### F4′ — حساء النماذج

متوسط أوزان البذور الثلاث دون كلفة تدريب إضافية.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3 أولاً"

def make_soup(group, name, model_name, out_name, imgsz=640):
    ids = [group + "__" + name + "__s" + str(s) for s in SEEDS]
    ws = [D_WEIGHTS/(i + "__best.pt") for i in ids]
    missing = [w.name for w in ws if not w.exists()]
    if missing:

        return None
    out_id = "accuracy__" + out_name + "__s42"
    out_w = D_WEIGHTS/(out_id + "__best.pt")
    if not out_w.exists():
        cks = [torch.load(str(w), map_location="cpu", weights_only=False) for w in ws]
        models = [ck["model"].float() for ck in cks]
        sds = [m.state_dict() for m in models]
        avg = {}
        for k, v in sds[0].items():
            if v.dtype.is_floating_point:
                avg[k] = sum(sd[k].float() for sd in sds)/len(sds)
            else:
                avg[k] = v
        models[0].load_state_dict(avg)
        cks[0]["model"] = models[0].half()
        torch.save(cks[0], str(out_w))
        print("✅ حساء محفوظ:", out_w.name)

    return train_and_eval(out_name, model_name, 42, group="accuracy", imgsz=imgsz,
                          notes="uniform model soup of seeds " + str(SEEDS))

make_soup("tournament", WINNER, WINNER, "soup640", imgsz=640)

✅ حساء محفوظ: accuracy__soup640__s42__best.pt

######################################################################
  RUN: accuracy__soup640__s42
######################################################################
  ⏭ الأوزان موجودة — تقييم فقط
  📊 التقييم الكامل ...
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2126.3±750.5 MB/s, size: 252.0 KB)
val: Scanning /content/sf/datasets/home_fire/test/labels... 1300 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1300/1300 1.4Kit/s 0.9s
val: New cache created: /content/sf/datasets/home_fire/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 82/82 19.6it/s 4.2s
                   all       1300       1586          0          0          0          0
Speed: 0.5ms preprocess, 1.6ms inference, 0

### TTA — تقييم اختياري

وضع الدقة القصوى باستدلال أبطأ مع تعزيز اختباري.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
hero = get_decision("hero"); assert hero, "شغّلي F5' أولاً"
data_yaml = ensure_home_fire_local()
rows = []
for seed in SEEDS:
    w = D_WEIGHTS/("%s__%s__s%d__best.pt" % (hero["group"], hero["name"], seed))
    if not w.exists():
        continue
    v = YOLO(str(w)).val(data=data_yaml, split="test", imgsz=640, augment=True,
                         verbose=False, plots=False, project=str(L_TMP), name="v", exist_ok=True)
    rows.append({"seed": seed, "mAP50_TTA": round(float(v.box.map50), 4),
                 "mAP5095_TTA": round(float(v.box.map), 4)})
t = pd.DataFrame(rows)
header("البطل + TTA (test)")
print(t.to_string(index=False))
print("mean: mAP50=%.4f | mAP50-95=%.4f — صف اختياري بجدول الدقة (مو للنشر على الجهاز)"
      % (t["mAP50_TTA"].mean(), t["mAP5095_TTA"].mean()))
t.to_csv(D_RESULTS/"hero_tta.csv", index=False)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2446.9±1116.2 MB/s, size: 148.6 KB)
val: Scanning /content/sf/datasets/home_fire/test/labels.cache... 1300 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1300/1300 454.4Mit/s 0.0s
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1% ──────────── 1/82 4.7s/it 1.4s<6:24WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
WARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 4% ╸─────────── 4/82 6.7it/

## X — نماذج الدقة العليا

تجارب النماذج الأكبر والدقة الأعلى، ثم نقل المعرفة إلى نموذج خفيف.



### X1 — نموذج الدقة القصوى

تدريب yolo26s@960 مع سوالب الاتحاد.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
header("X1 — SAFE-Fire-X")

pool_dir = L_DS/"hn_pool"
if count_imgs(pool_dir) == 0:
    robust_extract(D_TRAINNEG/"hn_train_pool.zip", pool_dir)
hit_dir = L_HITS/"round0"
if count_imgs(hit_dir) == 0:
    hz = D_SM/"mining"/"round0_hits.zip"
    assert hz.exists(), "round0_hits.zip مفقود"
    robust_extract(hz, hit_dir)
hn_names = json.loads((D_SM/"manifests"/"ratio_hn30.json").read_text())
top_names = json.loads((D_SM/"manifests"/"round0_top.json").read_text())
items, seen = [], set()
for p in [pool_dir/n for n in hn_names] + [hit_dir/n for n in top_names]:
    if p.name in seen:
        continue
    seen.add(p.name)
    items.append((p, ""))
combo_yaml = build_neg_dataset("combo", items)
print("سوالب الاتحاد:", len(items))

train_and_eval("xs960", "yolo26s", 42, group="showcase", data_yaml=combo_yaml,
               imgsz=960, batch=12,
               notes="SAFE-Fire-X pilot: yolo26s@960 + combo negatives")

df = master_df("showcase")
df = df[df["seed"] == 42]
header("استطلاع الـ showcase (seed 42)")
print(df[["name", "model", "imgsz", "mAP50", "mAP5095", "recall",
          "recall_at_fpr1", "FPR50_hn", "FPR50_mined"]].to_string(index=False))
best = df.loc[pd.to_numeric(df["mAP50"], errors="coerce").idxmax()]
save_decision("showcase", {"name": str(best["name"]), "model": str(best["model"]),
                           "imgsz": int(best["imgsz"]), "batch": 12})
header("🏆 الـ showcase المختار: %s (%s @%s) — mAP50=%s (الهدف: تجاوز 0.932)"
       % (best["name"], best["model"], best["imgsz"], best["mAP50"]))


  X1 — SAFE-Fire-X
  فك home_fire.zip ...
  ✅ داتاسيت combo: 3276 صورة إضافية → /content/sf/datasets/ds_combo/data.yaml
سوالب الاتحاد: 3276
✅ showcase__xs960__s42 منجز — تخطّي

  استطلاع الـ showcase (seed 42)
 name   model  imgsz  mAP50  mAP5095  recall  recall_at_fpr1  FPR50_hn  FPR50_mined
xs960 yolo26s    960 0.9346   0.6205  0.8824          0.8824       0.2          1.5
  📌 قرار محفوظ: showcase = {'name': 'xs960', 'model': 'yolo26s', 'imgsz': 960, 'batch': 12}

  🏆 الـ showcase المختار: xs960 (yolo26s @960) — mAP50=0.9346 (الهدف: تجاوز 0.932)


### X2 — البذور والحساء وTTA

تأكيد نموذج الدقة القصوى واستخراج أفضل نتيجة عرض.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
sc = get_decision("showcase"); assert sc, "شغّلي X1 أولاً"
header("X2 — %s (%s @%d)" % (sc["name"], sc["model"], int(sc["imgsz"])))

pool_dir = L_DS/"hn_pool"
if count_imgs(pool_dir) == 0:
    robust_extract(D_TRAINNEG/"hn_train_pool.zip", pool_dir)
hit_dir = L_HITS/"round0"
if count_imgs(hit_dir) == 0:
    robust_extract(D_SM/"mining"/"round0_hits.zip", hit_dir)
hn_names = json.loads((D_SM/"manifests"/"ratio_hn30.json").read_text())
top_names = json.loads((D_SM/"manifests"/"round0_top.json").read_text())
items, seen = [], set()
for p in [pool_dir/n for n in hn_names] + [hit_dir/n for n in top_names]:
    if p.name in seen:
        continue
    seen.add(p.name)
    items.append((p, ""))
combo_yaml = build_neg_dataset("combo", items)

for seed in (123, 2024):
    train_and_eval(sc["name"], sc["model"], seed, group="showcase",
                   data_yaml=combo_yaml, imgsz=int(sc["imgsz"]), batch=int(sc["batch"]),
                   notes="showcase multi-seed")

ids = ["showcase__%s__s%d" % (sc["name"], s) for s in SEEDS]
ws = [D_WEIGHTS/(i + "__best.pt") for i in ids]
if all(w.exists() for w in ws):
    out_w = D_WEIGHTS/"showcase__soup_x__s42__best.pt"
    if not out_w.exists():
        cks = [torch.load(str(w), map_location="cpu", weights_only=False) for w in ws]
        models = [ck["model"].float() for ck in cks]
        sds = [m.state_dict() for m in models]
        avg = {}
        for k, v in sds[0].items():
            if v.dtype.is_floating_point:
                avg[k] = sum(sd[k].float() for sd in sds)/len(sds)
            else:
                avg[k] = v
        models[0].load_state_dict(avg)
        cks[0]["model"] = models[0].half()
        torch.save(cks[0], str(out_w))
        print("✅ حساء X محفوظ")
    train_and_eval("soup_x", sc["model"], 42, group="showcase",
                   imgsz=int(sc["imgsz"]), notes="uniform soup of showcase seeds")

data_yaml = ensure_home_fire_local()
df = master_df("showcase")
sub = df[df["name"] == sc["name"]]
best_seed = int(sub.loc[pd.to_numeric(sub["mAP50"], errors="coerce").idxmax(), "seed"])
tta_rows = []
for tag, rid in [("best_seed_s%d" % best_seed,
                  "showcase__%s__s%d" % (sc["name"], best_seed)),
                 ("soup_x", "showcase__soup_x__s42")]:
    w = D_WEIGHTS/(rid + "__best.pt")
    if not w.exists():
        continue
    v = YOLO(str(w)).val(data=data_yaml, split="test", imgsz=int(sc["imgsz"]),
                         augment=True, verbose=False, plots=False,
                         project=str(L_TMP), name="v", exist_ok=True)
    tta_rows.append({"config": tag + " + TTA",
                     "mAP50": round(float(v.box.map50), 4),
                     "mAP5095": round(float(v.box.map), 4)})
tta = pd.DataFrame(tta_rows)
tta.to_csv(D_RESULTS/"showcase_tta.csv", index=False)

header("SAFE-Fire-X — الحصيلة النهائية (المرجع: YOLO-HF = 0.932)")
print("الـ seeds:")
print(sub[["seed", "mAP50", "mAP5095", "recall", "recall_at_fpr1",
           "FPR50_hn", "FPR50_mined"]].to_string(index=False))
print()
print("mean: mAP50=%.4f±%.4f | mAP50-95=%.4f | R@FPR1=%.4f"
      % (sub["mAP50"].mean(), sub["mAP50"].std(), sub["mAP5095"].mean(),
         sub["recall_at_fpr1"].mean()))
srow = df[df["name"] == "soup_x"]
if not srow.empty:
    print("الحساء: mAP50=%s | mAP50-95=%s" % (srow.iloc[0]["mAP50"], srow.iloc[0]["mAP5095"]))
if not tta.empty:
    print(tta.to_string(index=False))
bestnum = max([sub["mAP50"].max()] +
              ([float(srow.iloc[0]["mAP50"])] if not srow.empty else []) +
              ([tta["mAP50"].max()] if not tta.empty else []))
print()
print("🎯 أعلى رقم محقق: %.4f — %s هدف 0.932"
      % (bestnum, "✅ تجاوزنا" if bestnum > 0.932 else "⚠ ما وصلنا بعد"))


  X2 — xs960 (yolo26s @960)
  فك home_fire.zip ...
  ✅ داتاسيت combo: 3276 صورة إضافية → /content/sf/datasets/ds_combo/data.yaml
✅ showcase__xs960__s123 منجز — تخطّي

######################################################################
  RUN: showcase__xs960__s2024
######################################################################
New https://pypi.org/project/ultralytics/8.4.102 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_combo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=

### X3 — مقارنة yolo26m@960

تجربة نموذج متوسط بالحجم نفسه للمقارنة العادلة.


In [ ]:
try:
    train_and_eval
except NameError:
    raise RuntimeError("شغّلي S1 (يربط الدرايف) ثم B1 (المحرك) قبل هذه الخلية")
header("X3 — SAFE-Fire-X (m): yolo26m @960")


pool_dir = L_DS/"hn_pool"
if count_imgs(pool_dir) == 0:
    robust_extract(D_TRAINNEG/"hn_train_pool.zip", pool_dir)
hit_dir = L_HITS/"round0"
if count_imgs(hit_dir) == 0:
    robust_extract(D_SM/"mining"/"round0_hits.zip", hit_dir)
hn_names = json.loads((D_SM/"manifests"/"ratio_hn30.json").read_text())
top_names = json.loads((D_SM/"manifests"/"round0_top.json").read_text())
items, seen = [], set()
for p in [pool_dir/n for n in hn_names] + [hit_dir/n for n in top_names]:
    if p.name in seen:
        continue
    seen.add(p.name)
    items.append((p, ""))
combo_yaml = build_neg_dataset("combo", items)
print("سوالب الاتحاد:", len(items))

train_and_eval("xm960", "yolo26m", 42, group="showcase", data_yaml=combo_yaml,
               imgsz=960, batch=8,
               notes="SAFE-Fire-X (m): yolo26m@960 + combo negatives, single seed")


df = master_df("showcase")
df = df[pd.to_numeric(df["mAP50"], errors="coerce").fillna(0) > 0].copy()
keep = ["name", "model", "seed", "imgsz", "mAP50", "mAP5095", "recall",
        "recall_at_fpr1", "FPR50_hn", "FPR50_mined"]
header("مقارنة SAFE-Fire-X (المرجع YOLO-HF = 0.932)")
print(df[keep].sort_values("mAP50", ascending=False).to_string(index=False))
best = df.loc[pd.to_numeric(df["mAP50"], errors="coerce").idxmax()]
print()
print("🎯 أعلى mAP50 = %.4f  (%s %s @%s)  —  %s هدف 0.932"
      % (float(best["mAP50"]), best["name"], best["model"], best["imgsz"],
         "✅ تجاوز" if float(best["mAP50"]) > 0.932 else "دون"))


  X3 — SAFE-Fire-X (m): yolo26m @960
  فك home_fire.zip ...
  ✅ داتاسيت combo: 3276 صورة إضافية → /content/sf/datasets/ds_combo/data.yaml
سوالب الاتحاد: 3276

######################################################################
  RUN: showcase__xm960__s42
######################################################################
New https://pypi.org/project/ultralytics/8.4.102 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sf/datasets/ds_combo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=Fal

### X4 — تقطير المعرفة

نقل معرفة yolo26s@960 إلى yolo26n@640.


In [ ]:
try:
    train_and_eval
except NameError:
    raise RuntimeError("شغّلي S1 (يربط الدرايف ويجهّز المسارات) ثم B1 (المحرك) قبل هذه الخلية")
header("X4 — التقطير: معلّم yolo26s@960 → طالب yolo26n@640")

TEACHER = D_WEIGHTS/"showcase__xs960__s42__best.pt"
assert TEACHER.exists(), "أوزان المعلّم مفقودة: " + str(TEACHER)
print("المعلّم:", TEACHER.name)

pool_dir = L_DS/"hn_pool"
if count_imgs(pool_dir) == 0:
    robust_extract(D_TRAINNEG/"hn_train_pool.zip", pool_dir)
hit_dir = L_HITS/"round0"
if count_imgs(hit_dir) == 0:
    robust_extract(D_SM/"mining"/"round0_hits.zip", hit_dir)
hn_names = json.loads((D_SM/"manifests"/"ratio_hn30.json").read_text())
top_names = json.loads((D_SM/"manifests"/"round0_top.json").read_text())
items, seen = [], set()
for p in [pool_dir/n for n in hn_names] + [hit_dir/n for n in top_names]:
    if p.name in seen:
        continue
    seen.add(p.name)
    items.append((p, ""))
combo_yaml = build_neg_dataset("combo", items)
print("داتاسيت التدريب:", combo_yaml, "| سوالب:", len(items))

header("SANITY: رنّة تقطير 2-epoch للتحقق من الـAPI", "-")
kd_ok = True
try:
    train_and_eval("kd_sanity", "yolo26n", 42, group="kd", data_yaml=combo_yaml,
                   imgsz=640, sanity=True,
                   extra_args={"distill_model": str(TEACHER), "dis": 6.0},
                   notes="KD API sanity check")
    print("✅ التقطير مدعوم ويشتغل — نكمل للتدريب الكامل")
except Exception as e:
    kd_ok = False

DIS_VALUES = [6.0, 12.0]
if kd_ok:
    for d in DIS_VALUES:
        nm = "kd%g" % d
        try:
            train_and_eval(nm, "yolo26n", 42, group="kd", data_yaml=combo_yaml,
                           imgsz=640,
                           extra_args={"distill_model": str(TEACHER), "dis": float(d)},
                           notes="KD from showcase__xs960__s42 (teacher s@960), dis=%g" % d)
        except Exception as e:
            pass


    allrows = master_df()
    base = allrows[(allrows["group"] == "hero") & (allrows["name"] == "combo") &
                   (allrows["seed"] == 42)]
    kd = allrows[allrows["group"] == "kd"]
    kd = kd[~kd["name"].astype(str).str.contains("sanity")]
    kd = kd[pd.to_numeric(kd["mAP50"], errors="coerce").fillna(0) > 0]
    cols = ["name", "model", "imgsz", "mAP50", "mAP5095", "recall",
            "recall_at_fpr1", "FPR50_hn", "FPR50_novel", "FPR50_mined"]
    header("التقطير مقابل الموديل المنشور الحالي (seed 42)")
    cmp_df = pd.concat([base.assign(name="hero_combo (الأساس)")[cols], kd[cols]])
    print(cmp_df.to_string(index=False))
    if not base.empty and not kd.empty:
        b = float(base.iloc[0]["mAP50"])
        bestkd = kd.loc[pd.to_numeric(kd["mAP50"], errors="coerce").idxmax()]
        g = float(bestkd["mAP50"]) - b
        print()
        print("الأساس mAP50=%.4f  →  أفضل تقطير %s = %.4f  (الفرق %+.4f)"
              % (b, bestkd["name"], float(bestkd["mAP50"]), g))
        print("تشتّت الـseeds عندك ≈ ±0.004 — أي مكسب أصغر منه = ضمن الضوضاء، وثّقيه كنتيجة سلبية.")

  فك home_fire.zip ...

  X4 — التقطير: معلّم yolo26s@960 → طالب yolo26n@640
المعلّم: showcase__xs960__s42__best.pt
  ✅ داتاسيت combo: 3276 صورة إضافية → /content/sf/datasets/ds_combo/data.yaml
داتاسيت التدريب: /content/sf/datasets/ds_combo/data.yaml | سوالب: 3276

----------------------------------------------------------------------
  SANITY: رنّة تقطير 2-epoch للتحقق من الـAPI
----------------------------------------------------------------------

######################################################################
  SANITY: kd__kd_sanity__s42
######################################################################
New https://pypi.org/project/ultralytics/8.4.103 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, classe

## التحقق الخارجي — D-Fire

اختبار التعميم على بيانات موجبة مستقلة لم تدخل التدريب.



### XD — التقييم المتقاطع بصناديق الكشف

قياس التعميم على إيجابيات D-Fire التي لم تدخل التدريب.


In [ ]:
import shutil, json
import pandas as pd
from ultralytics import YOLO

try:
    D_RESULTS, robust_extract, header
except NameError:
    raise RuntimeError("شغّلي S1 قبل هذه الخلية")

header("XD — تقييم متقاطع: إيجابيات D-Fire")

raw = L_DS / "dfire_raw"
if not (raw / "data" / "test" / "labels").exists():
    z = D_DATA / "dfire_raw.zip"
    assert z.exists(), "dfire_raw.zip مفقود من الدرايف: " + str(z)
    print("فك الأرشيف (3 غيغا، ياخذ دقيقة)...")
    robust_extract(z, raw)
src_img = raw / "data" / "test" / "images"
src_lbl = raw / "data" / "test" / "labels"
assert src_lbl.exists(), "بنية D-Fire غير متوقعة: " + str(src_lbl)


DS = L_DS / "ds_dfire_pos"
di, dl = DS / "test" / "images", DS / "test" / "labels"
REMAP = {0: 1, 1: 0}

if not dl.exists() or len(list(dl.glob("*.txt"))) == 0:
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    kept = skipped = 0
    for lp in sorted(src_lbl.glob("*.txt")):
        txt = lp.read_text().strip()
        if not txt:
            continue
        out = []
        for line in txt.splitlines():
            p = line.split()
            if len(p) < 5:
                continue
            try:
                c = int(p[0])
            except ValueError:
                continue
            if c not in REMAP:
                continue
            out.append(" ".join([str(REMAP[c])] + p[1:5]))
        if not out:
            skipped += 1
            continue
        ip = None
        for ext in (".jpg", ".jpeg", ".png", ".JPG"):
            cand = src_img / (lp.stem + ext)
            if cand.exists():
                ip = cand
                break
        if ip is None:
            skipped += 1
            continue
        (dl / lp.name).write_text("\n".join(out) + "\n")
        dst = di / ip.name
        if not dst.exists():
            try:
                dst.symlink_to(ip)
            except Exception:
                shutil.copy2(ip, dst)
        kept += 1
    print("إيجابيات مبنية: %d   |   متخطاة: %d" % (kept, skipped))
else:
    pass

yml = DS / "data.yaml"
yml.write_text(
    "path: %s\ntrain: test/images\nval: test/images\ntest: test/images\n"
    "nc: 2\nnames: ['flame', 'smoke']\n" % DS
)

MODELS = [
    ("tournament__yolo26n__s42", 640, "أساس بلا سوالب"),
    ("hero__combo__s42",         640, "البطل (سوالب) بذرة 42"),
    ("hero__combo__s123",        640, "البطل بذرة 123"),
    ("hero__combo__s2024",       640, "البطل بذرة 2024"),
    ("kd__kd6__s42",             640, "المنشور (تقطير dis=6)"),
    ("kd__kd12__s42",            640, "تقطير dis=12"),
    ("showcase__xs960__s42",     960, "الواجهة yolo26s@960"),
]

OUT = D_RESULTS / "cross_dataset_dfire.csv"
done = set()
if OUT.exists():
    done = set(pd.read_csv(OUT)["run_id"].astype(str))

rows = []
for run_id, imsz, note in MODELS:
    if run_id in done:
        print("تم مسبقاً — تخطي:", run_id)
        continue
    w = D_WEIGHTS / (run_id + "__best.pt")
    if not w.exists():

        continue

    v = YOLO(str(w)).val(data=str(yml), split="test", imgsz=imsz,
                         verbose=False, plots=False,
                         project=str(L_TMP), name="xd", exist_ok=True)
    p, r = float(v.box.mp), float(v.box.mr)
    ap50, ap = v.box.ap50, v.box.ap
    rows.append({
        "run_id": run_id, "note": note, "imgsz": imsz,
        "mAP50": round(float(v.box.map50), 4),
        "mAP5095": round(float(v.box.map), 4),
        "precision": round(p, 4), "recall": round(r, 4),
        "f1": round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0,
        "flame_AP50": round(float(ap50[0]), 4) if len(ap50) > 0 else "",
        "smoke_AP50": round(float(ap50[1]), 4) if len(ap50) > 1 else "",
    })

if rows:
    df = pd.DataFrame(rows)
    if OUT.exists():
        df = pd.concat([pd.read_csv(OUT), df], ignore_index=True)
    df.to_csv(OUT, index=False)

HF = {
    "tournament__yolo26n__s42": (0.9049, 0.8651),
    "hero__combo__s42":         (0.8987, 0.8346),
    "hero__combo__s123":        (0.8977, 0.8372),
    "hero__combo__s2024":       (0.9054, 0.8381),
    "kd__kd6__s42":             (0.9325, 0.8819),
    "kd__kd12__s42":            (0.9242, 0.8578),
    "showcase__xs960__s42":     (0.9346, 0.8824),
}
if OUT.exists():
    d = pd.read_csv(OUT)
    header("النتيجة: التعميم عبر الداتاسيتات", "-")
    print("%-26s | %-16s | %-16s | %s" %
          ("run", "home_fire mAP50", "D-Fire mAP50", "recall D-Fire"))
    print("-" * 82)
    for _, x in d.iterrows():
        hf = HF.get(x["run_id"], ("", ""))
        print("%-26s | %-16s | %-16s | %s" %
              (x["run_id"], hf[0], x["mAP50"], x["recall"]))
    print("\nملاحظة للكتابة: الأرقام على D-Fire ستكون أوطأ، وهذا متوقّع —")
    print("داتاسيت مختلفة، تأشير مختلف، ومشاهد خارجية أكثر. السؤال ليس")
    print("مقدار الانخفاض، بل: هل السوالب الصعبة كلّفتنا تعميماً؟ قارني")
    print("صف الأساس (tournament) بصف البطل (hero) وصف المنشور (kd6).")


  XD — تقييم متقاطع: إيجابيات D-Fire
فك الأرشيف (3 غيغا، ياخذ دقيقة)...
إيجابيات مبنية: 2301   |   متخطاة: 0
تم مسبقاً — تخطي: tournament__yolo26n__s42
تم مسبقاً — تخطي: hero__combo__s42
تم مسبقاً — تخطي: hero__combo__s123
تم مسبقاً — تخطي: hero__combo__s2024
تم مسبقاً — تخطي: kd__kd6__s42
تم مسبقاً — تخطي: kd__kd12__s42
تم مسبقاً — تخطي: showcase__xs960__s42

----------------------------------------------------------------------
  النتيجة: التعميم عبر الداتاسيتات
----------------------------------------------------------------------
run                        | home_fire mAP50  | D-Fire mAP50     | recall D-Fire
----------------------------------------------------------------------------------
tournament__yolo26n__s42   | 0.9049           | 0.226            | 0.2573
hero__combo__s42           | 0.8987           | 0.1505           | 0.1682
hero__combo__s123          | 0.8977           | 0.1426           | 0.2585
hero__combo__s2024         | 0.9054           | 0.1541           | 0.2206

### XD2 — التقييم على مستوى الصورة

قياس نسبة صور الحريق التي تُنتج كشفاً واحداً على الأقل.


In [ ]:
import numpy as np, pandas as pd
from ultralytics import YOLO

try:
    scan_max_conf
except NameError:
    raise RuntimeError("شغّلي B1 قبل هذه الخلية (تحتاج scan_max_conf)")

IMGS = L_DS / "ds_dfire_pos" / "test" / "images"
assert IMGS.exists(), "شغّلي XD أولاً لبناء ds_dfire_pos"
n_imgs = len(list(IMGS.glob("*")))
header("XD2 — كشف على مستوى الصورة | %d صورة حريق من D-Fire" % n_imgs)

MODELS = [
    ("tournament__yolo26n__s42", 640, "أساس بلا سوالب"),
    ("hero__combo__s42",         640, "البطل بذرة 42"),
    ("hero__combo__s123",        640, "البطل بذرة 123"),
    ("hero__combo__s2024",       640, "البطل بذرة 2024"),
    ("kd__kd6__s42",             640, "المنشور (dis=6)"),
    ("kd__kd12__s42",            640, "dis=12"),
    ("showcase__xs960__s42",     960, "الواجهة yolo26s@960"),
]
THR = [0.25, 0.50]

OUT = D_RESULTS / "cross_dataset_dfire_imagelevel.csv"
done = set(pd.read_csv(OUT)["run_id"].astype(str)) if OUT.exists() else set()

rows = []
for run_id, imsz, note in MODELS:
    if run_id in done:
        continue
    w = D_WEIGHTS / (run_id + "__best.pt")
    if not w.exists():
        continue
    print("فحص %-26s @%d ..." % (run_id, imsz))
    sc = D_SCORES / (run_id + "__dfire_pos_maxconf.csv")
    mx = scan_max_conf(YOLO(str(w)), IMGS, sc, imgsz=imsz)
    rec = {"run_id": run_id, "note": note, "imgsz": imsz, "n_images": int(len(mx))}
    for t in THR:
        rec["IDR@%.2f" % t] = round(float((mx >= t).mean()), 4)
    rec["median_conf"] = round(float(np.median(mx)), 4)
    rows.append(rec)

if rows:
    df = pd.DataFrame(rows)
    if OUT.exists():
        df = pd.concat([pd.read_csv(OUT), df], ignore_index=True)
    df.to_csv(OUT, index=False)
    print("\nمحفوظ:", OUT)

FPR = {
    "tournament__yolo26n__s42": (27.6, 32.4),
    "hero__combo__s42":         (0.4,  1.00),
    "hero__combo__s123":        (0.4,  1.00),
    "hero__combo__s2024":       (0.4,  1.00),
    "kd__kd6__s42":             (1.2,  2.75),
    "kd__kd12__s42":            (1.0,  1.50),
    "showcase__xs960__s42":     (0.6,  1.50),
}
if OUT.exists():
    d = pd.read_csv(OUT)
    header("المقايضة الحقيقية: كشف خارج المجال مقابل الإنذار الكاذب", "-")
    print("%-26s | %-12s | %-12s | %s" %
          ("run", "IDR@0.50", "IDR@0.25", "FPR_hn %"))
    print("-" * 74)
    for _, x in d.iterrows():
        f = FPR.get(x["run_id"], ("", ""))
        print("%-26s | %-12s | %-12s | %s" %
              (x["run_id"], x["IDR@0.50"], x["IDR@0.25"], f[0]))
    print("\nIDR = نسبة صور الحريق التي أنتج فيها الموديل كشفاً واحداً على الأقل.")
    print("نفس تعريف FPR بالضبط، لكن على صور موجبة — فالعمودان قابلان للمقارنة.")


  XD2 — كشف على مستوى الصورة | 2301 صورة حريق من D-Fire
فحص tournament__yolo26n__s42   @640 ...
فحص hero__combo__s42           @640 ...
فحص hero__combo__s123          @640 ...
فحص hero__combo__s2024         @640 ...
فحص kd__kd6__s42               @640 ...
فحص kd__kd12__s42              @640 ...
فحص showcase__xs960__s42       @960 ...

محفوظ: /content/drive/MyDrive/SAFE_FIRE/results/cross_dataset_dfire_imagelevel.csv

----------------------------------------------------------------------
  المقايضة الحقيقية: كشف خارج المجال مقابل الإنذار الكاذب
----------------------------------------------------------------------
run                        | IDR@0.50     | IDR@0.25     | FPR_hn %
--------------------------------------------------------------------------
tournament__yolo26n__s42   | 0.4081       | 0.5267       | 27.6
hero__combo__s42           | 0.2312       | 0.3303       | 0.4
hero__combo__s123          | 0.189        | 0.2595       | 0.4
hero__combo__s2024         | 0.1908       | 0

## G — النتائج النهائية

تحديث المقاييس وإنتاج الجداول والأشكال من مصدر نتائج موحّد.



### G0 — استكمال mined_dev وحسم مصدر السوالب

إكمال ملفات القياس واختيار مصدر سوالب البطل.


In [ ]:
assert "train_and_eval" in globals(), "شغّلي S1 ثم B1 أولاً"
suite = ensure_suite_local(strict=True)
assert "mined_dev" in suite, "شغّلي المرحلة D أولاً"
WINNER = get_decision("winner"); assert WINNER, "شغّلي B3"

df = master_df()
full = pd.read_csv(MASTER_CSV).drop_duplicates(subset="run_id", keep="last")
changed = 0
for _, r_ in df.iterrows():
    rid = str(r_["run_id"])
    sc = D_SCORES/(rid + "__mined_dev.csv")
    if not sc.exists():
        w = Path(str(r_["weights"]))
        if not w.exists():
            continue
        sz = pd.to_numeric(r_["imgsz"], errors="coerce")
        sz = int(sz) if pd.notna(sz) else 640

        scan_max_conf(YOLO(str(w)), suite["mined_dev"], sc, imgsz=sz)
    mx = pd.read_csv(sc)["max_conf"].to_numpy()
    full.loc[full["run_id"] == rid, "FPR25_mined"] = round(100*float((mx >= 0.25).mean()), 2)
    full.loc[full["run_id"] == rid, "FPR50_mined"] = round(100*float((mx >= 0.50).mean()), 2)
    changed += 1
tmp = MASTER_CSV.with_suffix(".csv.tmp")
full.to_csv(tmp, index=False)
tmp.replace(MASTER_CSV)
print("✅ حُدّثت أعمدة mined لـ %d صف" % changed)

df = master_df()
cand = df[((df["group"] == "hnratio") & (df["name"] == "hn30")) |
          ((df["group"] == "safemine") & (df["name"] == "sm_top") & (df["model"] == WINNER))]
g = (cand.groupby("name")[["mAP50", "recall_at_fpr1", "FPR50_hn", "FPR50_novel", "FPR50_mined"]]
     .agg(["mean", "std"]).round(4))
header("قرار مصدر سوالب البطل (متوسط 3 seeds)")
print(g.to_string())
m_ = cand.groupby("name")["recall_at_fpr1"].mean()
a_ = cand.groupby("name")["mAP50"].mean()
NEG_SRC = max(list(m_.index), key=lambda n: (round(float(m_[n]), 3), round(float(a_[n]), 4)))
save_decision("hero_neg_source", NEG_SRC)
header("🏆 مصدر سوالب البطل: " + NEG_SRC)
print("(المنافس الآخر يبقى بجداول الرسالة كمقارنة يدوي-ضد-منقّب)")

✅ حُدّثت أعمدة mined لـ 37 صف

  قرار مصدر سوالب البطل (متوسط 3 seeds)
         mAP50         recall_at_fpr1         FPR50_hn         FPR50_novel         FPR50_mined        
          mean     std           mean     std     mean     std        mean     std        mean     std
name                                                                                                  
hn30    0.9025  0.0045         0.8000  0.0499   1.0667  0.3055        3.06  1.3493     12.0000  0.4330
sm_top  0.8987  0.0156         0.7903  0.0393   1.0000  0.5292        1.36  0.2944      1.4167  0.1443
  📌 قرار محفوظ: hero_neg_source = hn30

  🏆 مصدر سوالب البطل: hn30
(المنافس الآخر يبقى بجداول الرسالة كمقارنة يدوي-ضد-منقّب)


### G1 — الجداول النهائية

تجميع mean±std وإعادة حساب FPR من ملفات scores.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
header("G1) الجداول النهائية")
df = master_df()

n0 = int((pd.to_numeric(df["mAP50"], errors="coerce").fillna(0) <= 0).sum())
if n0:
    print("⚠ استُبعدت %d رنّة فاشلة (mAP50=0) من الجداول — وثّقيها كنتيجة سلبية بالنص" % n0)
df = df[pd.to_numeric(df["mAP50"], errors="coerce").fillna(0) > 0].copy()
TABDIR = D_RESULTS/"thesis_tables"; TABDIR.mkdir(exist_ok=True)

def fpr_from_scores(run_id, sname, tau):
    sc = D_SCORES/(run_id + "__" + sname + ".csv")
    if not sc.exists():
        return None
    mx = pd.read_csv(sc)["max_conf"].to_numpy()
    return round(100*float((mx >= tau).mean()), 2)

for _, r_ in df.iterrows():
    v = fpr_from_scores(r_["run_id"], "mined_dev", 0.5)
    if v is not None:
        df.loc[df["run_id"] == r_["run_id"], "FPR50_mined"] = v

for grp in ["tournament", "hnratio", "safemine", "accuracy", "hero", "showcase"]:
    sub = df[df["group"] == grp]
    if sub.empty:
        continue
    g = sub.groupby("name")
    tab = pd.DataFrame({
        "model": g["model"].first(), "runs": g.size(),
        "mAP50": g["mAP50"].mean().round(4), "mAP50_std": g["mAP50"].std().round(4),
        "mAP5095": g["mAP5095"].mean().round(4),
        "recall": g["recall"].mean().round(4), "f1": g["f1"].mean().round(4),
        "R_at_FPR1": g["recall_at_fpr1"].mean().round(4),
        "FPR50_hn": g["FPR50_hn"].mean().round(2),
        "FPR50_novel": g["FPR50_novel"].mean().round(2),
        "FPR50_dfire": g["FPR50_dfire"].mean().round(2),
        "FPR50_mined": g["FPR50_mined"].mean().round(2),
    }).sort_values("R_at_FPR1", ascending=False)
    header("جدول المجموعة: " + grp, "-")
    print(tab.to_string())
    tab.to_csv(TABDIR/("table_" + grp + ".csv"))
print(); print("💾 كل الجداول بـ:", TABDIR)


  G1) الجداول النهائية
⚠ استُبعدت 2 رنّة فاشلة (mAP50=0) من الجداول — وثّقيها كنتيجة سلبية بالنص

----------------------------------------------------------------------
  جدول المجموعة: tournament
----------------------------------------------------------------------
           model  runs   mAP50  mAP50_std  mAP5095  recall      f1  R_at_FPR1  FPR50_hn  FPR50_novel  FPR50_dfire  FPR50_mined
name                                                                                                                          
yolov8s  yolov8s     3  0.9145     0.0028   0.5893  0.8650  0.8925     0.1230     30.53        26.53        15.00        29.92
yolo11n  yolo11n     3  0.9017     0.0107   0.5824  0.8466  0.8805     0.0954     32.53        25.34        14.36        29.08
yolov8n  yolov8n     3  0.9075     0.0045   0.5797  0.8565  0.8838     0.0936     32.13        25.51        15.62        29.58
yolo11s  yolo11s     3  0.9056     0.0121   0.5891  0.8605  0.8919     0.0272     33.73        2

### G2 — الأشكال النهائية

إنشاء Pareto ومقارنات الأساس والبطل.


In [ ]:
assert "ROOT" in globals(), "شغّلي S1 أولاً"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
header("G2) الأشكال")
df = master_df()
WINNER = get_decision("winner")
hero = get_decision("hero", {})

base_id = "tournament__%s__s42" % WINNER
hero_id = "%s__%s__s42" % (hero.get("group", "safemine"), hero.get("name", "sm_cov"))
pairs = [("Baseline (no negatives)", base_id, "tab:blue"),
         ("SAFE-Fire (ours)", hero_id, "tab:orange")]

fig, ax = plt.subplots(figsize=(6.5, 5.2))
ok = False
for label, rid, color in pairs:
    sw_p = D_SWEEPS/(rid + "__test_sweep.csv")
    sc_p = D_SCORES/(rid + "__hn_holdout.csv")
    if not (sw_p.exists() and sc_p.exists()):

        continue
    sw = pd.read_csv(sw_p)
    mx = pd.read_csv(sc_p)["max_conf"].to_numpy()
    xs = sw["recall"].tolist()
    ys = [100*float((mx >= t).mean()) for t in sw["conf"]]
    pts = sorted(zip(xs, ys))
    ax.plot([p[0] for p in pts], [p[1] for p in pts], "o-", color=color, label=label)
    ok = True
if ok:
    ax.set_xlabel("Recall (home_fire test)")
    ax.set_ylabel("FPR on hard-negative holdout (%)")
    ax.set_title("FPR-Recall operating curve (lower-right is better)")
    ax.legend(); ax.grid(alpha=.3)
    fig.savefig(D_FIGS/"pareto_final.png", dpi=200, bbox_inches="tight")
    print("🖼", D_FIGS/"pareto_final.png")


key = df[(df["seed"] == 42)].copy()
key = key.sort_values("recall_at_fpr1", ascending=False).head(10)
fig2, ax2 = plt.subplots(figsize=(8, 4.5))
ax2.barh(key["group"] + "/" + key["name"], key["recall_at_fpr1"], color="tab:green")
ax2.set_xlabel("Recall @ FPR<=1% (hn holdout)")
ax2.invert_yaxis(); ax2.grid(alpha=.3, axis="x")
fig2.savefig(D_FIGS/"recall_at_fpr1_bar.png", dpi=200, bbox_inches="tight")
print("🖼", D_FIGS/"recall_at_fpr1_bar.png")
print("✅ G2 تمّت")


  G2) الأشكال
🖼 /content/drive/MyDrive/SAFE_FIRE/figures_v2/pareto_final.png
🖼 /content/drive/MyDrive/SAFE_FIRE/figures_v2/recall_at_fpr1_bar.png
✅ G2 تمّت


## Export — التصدير إلى Jetson

تصدير النموذج إلى `ONNX` وتجهيزه للتوافق مع `TensorRT 8.2`.


### JETSON-A1 — التصدير إلى ONNX

تصدير kd6 وفحص توافق TensorRT 8.2.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import subprocess, sys, os, shutil
subprocess.run([sys.executable,"-m","pip","install","-q","onnx","onnxslim"])
from ultralytics import YOLO
import onnx

W = "/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.pt"
assert os.path.exists(W), "أوزان kd6 مفقودة: " + W
m = YOLO(W)

path = m.export(format="onnx", opset=12, simplify=True, dynamic=False,
                imgsz=640, half=False, device="cpu")
print("ONNX:", path)

g = onnx.load(path).graph
ops = {}
for n in g.node: ops[n.op_type] = ops.get(n.op_type, 0) + 1
print("\nأنواع العوامل:", dict(sorted(ops.items())))
nmod = ops.get("Mod", 0)
print("عدد Mod =", nmod, "→", "⚠ راح يكسر TensorRT 8.2 — عندي حل" if nmod else "✅ نظيف — نكمل مباشرة")

out = "/content/kd6_640_op12.onnx"; shutil.copy(path, out)
print("\n📥 نزّلي هذا وأرسليه لي:", out, "(%.1f MB)" % (os.path.getsize(out)/1e6))
from google.colab import files; files.download(out)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 211ms
Prepared 1 package in 336ms
Installed 1 package in 9ms
 + onnxruntime==1.27.0

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requireme

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 12 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 5.5s, saved as '/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx' (9.4 MB)

Export complete (6.2s)
Results saved to /content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx imgsz=640 data=/content/sf/datasets/ds_combo/data.yaml  
Visualize:       https://netron.app
ONNX: /content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx

أنواع العوامل: {'Add': 21, 'Cast': 1, 'Concat': 24, 'Conv': 102, 'Div': 1, 'Flatten': 2, 'Gather': 1, 'GatherElements': 2, 'MatMul': 4, 'MaxPool': 3, 'Mod': 1, 'Mul': 90, 'ReduceMax': 1, 'Reshape': 12, 'Resize': 2, 'Sigmoid': 88, 'Slice': 2, 'Softmax': 2, 'Split': 12, 'Sub': 1, 'Tile': 2, 'TopK': 2, 'Transpose': 5, 'Unsqueeze': 3}
عدد Mod = 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### JETSON-A2 — توافق TensorRT 8.2

استبدال عامل Mod بصيغة متوافقة مع TensorRT.


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable,"-m","pip","install","-q","onnx","onnx_graphsurgeon",
                "--extra-index-url","https://pypi.ngc.nvidia.com"])
import onnx, onnx_graphsurgeon as gs

SRC = "/content/drive/MyDrive/SAFE_FIRE/weights/kd__kd6__s42__best.onnx"
graph = gs.import_onnx(onnx.load(SRC))
mods = [n for n in graph.nodes if n.op == "Mod"]
print("عوامل Mod الموجودة:", len(mods))
for mod in mods:
    a, b = mod.inputs[0], mod.inputs[1]
    out = mod.outputs[0]
    div_o = gs.Variable(name=mod.name+"_div_out")
    mul_o = gs.Variable(name=mod.name+"_mul_out")
    div = gs.Node(op="Div", name=mod.name+"_div", inputs=[a, b], outputs=[div_o])
    mul = gs.Node(op="Mul", name=mod.name+"_mul", inputs=[div_o, b], outputs=[mul_o])
    sub = gs.Node(op="Sub", name=mod.name+"_sub", inputs=[a, mul_o], outputs=[out])
    mod.outputs.clear(); graph.nodes.remove(mod); graph.nodes += [div, mul, sub]
graph.cleanup().toposort()
DST = "/content/kd6_nomod.onnx"
onnx.save(gs.export_onnx(graph), DST)

ops = {}
for n in gs.import_onnx(onnx.load(DST)).nodes: ops[n.op] = ops.get(n.op,0)+1
print("Mod بعد الإصلاح =", ops.get("Mod",0), "→", "✅ نظيف" if not ops.get("Mod") else "لسه موجود")
print("📥 نزّلي وأرسليه لي:", DST, "(%.1f MB)" % (os.path.getsize(DST)/1e6))
from google.colab import files; files.download(DST)

عوامل Mod الموجودة: 1
Mod بعد الإصلاح = 0 → ✅ نظيف
📥 نزّلي وأرسليه لي: /content/kd6_nomod.onnx (9.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# الجزء الثاني — النظام المنشور

كل ما سبق جرى على L4 في Colab. ما يلي جرى على عتاد حقيقي: عقدة ESP32 وJetson
Nano 4GB وهاتف أندرويد. لا يُشغَّل أي من الخلايا التالية هنا — لا كاميرا ولا
منفذ تسلسلي ولا محرك TensorRT في آلة Colab. هي المصدر المنشور نفسه، مُدرَجاً
حتى يكون هذا الدفتر السجل الكامل للنظام لا لنصفه.

**سلسلة القياس كاملة:**

```
ESP32 (أربع قنوات) ──USB──► Jetson Nano ──HTTPS──► Firestore ──trigger──► FCM ──► الهاتف
                                   │
                                   └──MJPEG على الشبكة المحلية──► الهاتف
```


## H — عقدة الحسّاسات

أربع قنوات على ESP32 واحد: DS18B20 لحرارة الهواء، MLX90614 لحرارة السطح بلا
تلامس، MQ-2 للغاز، ووحدة أشعة تحت حمراء للهب.

**ثلاث قرارات قياس تستحق التوثيق:**

1. **لا وحدة ppm للغاز.** ورقة بيانات MQ-2 تُعرّف R0 بتركيز 1000 ppm من غاز
   LPG، بينما العقدة تأخذ خط أساسها في هواء نظيف. منحنيات ppm المنشورة لا
   تنطبق على هذه النسبة، فأي رقم ppm سيكون وحدة مُختلَقة. المقياس المنشور هو
   `Rs/R0` حيث 1.0 هواء نظيف وكلما قلّت زاد الغاز.

2. **خرج وحدة اللهب ينخفض عند رؤية الأشعة، لا يرتفع.** العتبة كُتبت في البداية
   كارتفاع، وهو ما كان سيُبقي القناة صامتة إلى الأبد.

3. **قناة لم تُقس تُرسَل `null` لا صفراً.** عقدة ميتة كانت ستبدو غرفة عند صفر
   درجة بلا غاز، وهو بالضبط الخلل الذي تتهم به هذه الرسالة الأدبيات.


In [ ]:
# ==========================================================================
# WIRING/firmware/safefire_node/safefire_node.ino
# --------------------------------------------------------------------------
# RUNS ON: ESP32-S3, uploaded with the Arduino IDE
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# The four channels, their baselines, and the node's own level.
# ==========================================================================

/*  SAFE-Fire sensor node - all four channels on one ESP32
 *  =====================================================================
 *  Merges the four bring-up sketches into the thing that actually ships.
 *  Every pin, divider and threshold below was established by the
 *  individual sketches; nothing here is new wiring, only integration.
 *
 *      MQ-2       AO -> 6.8k -> GPIO 34, 10k to GND      5 V   <-- only 5 V part
 *      IR flame   AO -> GPIO 35                          3.3 V
 *      DS18B20    DQ -> GPIO 4, 4.7k to 3.3 V            3.3 V
 *      MLX90614   SDA GPIO 21, SCL GPIO 22               3.3 V
 *
 *  WHY THE LOOP NEVER BLOCKS
 *      A DS18B20 conversion takes 750 ms at 12 bits. Calling the blocking
 *      read would stall the whole node for three quarters of a second,
 *      during which a flame flicker is missed entirely. Conversion is
 *      requested, the loop keeps sampling everything else, and the result
 *      is collected when it is ready.
 *
 *  BASELINES SURVIVE A REBOOT
 *      MQ-2 needs Vo0 and MLX90614 needs its clean-scene object
 *      temperature. Re-taking them on every boot would silently rebaseline
 *      in whatever conditions happened to exist at power-up, including a
 *      room that is already smoky. They are stored in NVS and reused, with
 *      the age of the calibration reported so a stale one is visible.
 *
 *  OUTPUT
 *      One JSON line per second on the serial port, for the Jetson to read.
 *      Human-readable lines are prefixed with '#' so a parser can skip them
 *      without needing to be clever.
 *
 *  Keys:  b re-baseline (clean air, nothing hot in view)   s status
 *         c clear stored baselines                        h help
 */

#include <OneWire.h>
#include <DallasTemperature.h>
#include <Wire.h>
#include <Preferences.h>

#define PIN_MQ2        34
#define PIN_FLAME      35
#define PIN_DS18B20     4
#define PIN_SDA        21
#define PIN_SCL        22

#define MQ2_GAIN     1.68f     // (6.8k + 10k) / 10k
#define MQ2_VCC      5.0f
#define MLX_ADDR     0x5A
#define I2C_HZ       100000L
#define AVG_N        32
#define EMIT_MS      1000
#define DS_CONV_MS   800       // 750 ms at 12 bit, plus margin

/* Alarm thresholds. The gas and thermal figures come from the measured
 * noise floors, not from round numbers: MQ-2 classifies a rise below 0.85
 * of baseline, and the MLX90614 noise floor was sigma = 0.163 C on this
 * rig, so 3 sigma is 0.49 C and a 2 C rise is a wide margin above it. */
#define GAS_RISE_ALARM   0.85f
#define SURF_RISE_ALARM  2.00f

/* The flame module's analogue output FALLS when it sees infrared. It rests near
 * the top of the range with no flame, which is why 2990 mV in a quiet room is
 * the correct idle reading and not a fault. Established in measure_flame.ino,
 * which reports the direction it measured rather than assuming one.
 *
 * Writing this threshold as a rise would leave the channel permanently silent:
 * the value can only go down, so a comparison against a rise never fires. */
#define FLAME_DROP_MV    150

OneWire oneWire(PIN_DS18B20);
DallasTemperature ds(&oneWire);
Preferences prefs;

static float    vo0 = NAN, objBase = NAN;
static uint32_t baseEpoch = 0;
static float    ambientC = NAN, surfaceC = NAN, mlxAmbC = NAN;
static float    gasRatio = NAN;
static uint16_t flameMv = 0, flameBaseMv = 0;
static bool     okDs = false, okMlx = false, okMq2 = false, okFlame = false;
static uint32_t dsAskedAt = 0;
static bool     dsPending = false;
static uint32_t lastEmit = 0;

/* ------------------------------------------------------------ MLX90614 */

static uint8_t crc8(const uint8_t *d, uint8_t n) {
  uint8_t c = 0;
  while (n--) {
    c ^= *d++;
    for (uint8_t i = 0; i < 8; i++)
      c = (c & 0x80) ? (uint8_t)((c << 1) ^ 0x07) : (uint8_t)(c << 1);
  }
  return c;
}

/* Returns false on a bus fault OR a bad checksum. The checksum matters:
 * without it a loose wire is read as a temperature, which is how the
 * DS18B20 bring-up went wrong twice. */
static bool mlxRead(uint8_t reg, float *out) {
  Wire.beginTransmission(MLX_ADDR);
  Wire.write(reg);
  if (Wire.endTransmission(false) != 0) return false;
  if (Wire.requestFrom((int)MLX_ADDR, 3) != 3) return false;
  uint8_t lo = Wire.read(), hi = Wire.read(), pec = Wire.read();
  uint8_t buf[5] = { (uint8_t)(MLX_ADDR << 1), reg,
                     (uint8_t)((MLX_ADDR << 1) | 1), lo, hi };
  if (crc8(buf, 5) != pec) return false;
  uint16_t v = ((uint16_t)hi << 8) | lo;
  if (v & 0x8000) return false;
  *out = v * 0.02f - 273.15f;
  return true;
}

/* --------------------------------------------------------------- MQ-2 */

static float mq2Volts() {
  uint32_t acc = 0;
  for (int i = 0; i < AVG_N; i++) { acc += analogReadMilliVolts(PIN_MQ2); delayMicroseconds(200); }
  return (acc / (float)AVG_N) / 1000.0f * MQ2_GAIN;
}

/* Rs/R0 with the module's load resistor cancelled out. RL is not printed
 * on the board and differs between makers, so it is eliminated rather than
 * guessed:  Rs/R0 = [(Vc-Vo)/Vo] * [Vo0/(Vc-Vo0)] */
static float mq2Ratio(float vo) {
  if (isnan(vo0) || vo <= 0.01f || vo >= MQ2_VCC - 0.01f) return NAN;
  return ((MQ2_VCC - vo) / vo) / ((MQ2_VCC - vo0) / vo0);
}

static uint16_t flameMillivolts() {
  uint32_t acc = 0;
  for (int i = 0; i < AVG_N; i++) { acc += analogReadMilliVolts(PIN_FLAME); delayMicroseconds(200); }
  return acc / AVG_N;
}

/* ------------------------------------------------- flame windowing
 *
 * With a lighter held near the node, the one-second reading alternated between
 * 231 mV and 2740 mV and the level fell back to NORMAL while the flame was
 * still burning. The cause was observed at the bench and is physical: room
 * airflow pushes the flame out of this module's narrow field of view and back
 * in. 2740 mV is the full idle value, not an intermediate one, which is what
 * losing sight of the source looks like rather than a sampling artefact.
 *
 * The MLX90614 in the same window stayed elevated between +0.4 and +3.6 C
 * without ever returning to baseline. Two channels watching one flame, and only
 * the narrow-field one loses it. That difference is field of view, and it is
 * worth recording.
 *
 * So the window is sampled continuously and reduced two ways:
 *
 *   minimum  - the detector. Any dip inside the window counts, because a flame
 *              that was briefly blown out of view is still a flame.
 *   dips     - how many times the source ENTERED view during the window. It was
 *              intended as a flicker discriminator and it is not one: measured
 *              with a lighter, the drop is 2500 mV against a 150 mV threshold, so
 *              flicker never comes close to crossing back and dips reads 0 for a
 *              steadily held flame. Kept because entry and exit events are useful,
 *              but it does not separate a flame from a steady source.
 *   p2p      - the actual flicker measure: peak to peak inside the window. A
 *              steady infrared source, sunlight on a wall or a hot lamp, holds the
 *              output at a level and gives a small p2p. A flame moves.
 */
static uint16_t winMin = 0xFFFF;
static uint16_t winMax = 0;
static uint16_t winDips = 0;
static bool     winBelow = false;
static uint16_t flameMinMv = 0;
static uint16_t flameDips = 0;
static uint16_t flameP2p = 0;

static void flameSample() {
  uint32_t acc = 0;
  for (int i = 0; i < 8; i++) { acc += analogReadMilliVolts(PIN_FLAME); delayMicroseconds(100); }
  uint16_t mv = acc / 8;
  flameMv = mv;
  if (mv < winMin) winMin = mv;
  if (mv > winMax) winMax = mv;
  if (flameBaseMv) {
    bool below = ((int)flameBaseMv - (int)mv) > FLAME_DROP_MV;
    if (below && !winBelow) winDips++;      // count crossings, not samples
    winBelow = below;
  }
}

static void flameCloseWindow() {
  flameMinMv = (winMin == 0xFFFF) ? flameMv : winMin;
  flameP2p = (winMax > winMin && winMin != 0xFFFF) ? (winMax - winMin) : 0;
  flameDips = winDips;
  winMin = 0xFFFF; winMax = 0; winDips = 0;
}

/* ------------------------------------------------------------ baseline */

static void saveBaselines() {
  prefs.begin("safefire", false);
  prefs.putFloat("vo0", vo0);
  prefs.putFloat("objBase", objBase);
  prefs.putUShort("flameBase", flameBaseMv);
  prefs.putUInt("epoch", baseEpoch);
  prefs.end();
}

static void loadBaselines() {
  prefs.begin("safefire", true);
  vo0        = prefs.getFloat("vo0", NAN);
  objBase    = prefs.getFloat("objBase", NAN);
  flameBaseMv = prefs.getUShort("flameBase", 0);
  baseEpoch  = prefs.getUInt("epoch", 0);
  prefs.end();
}

static void takeBaselines() {
  Serial.println(F("# BASELINE  clean air, nothing hot in the field. 6 s."));
  double sv = 0, so = 0, sf = 0;
  int nv = 0, no = 0, nf = 0;
  for (int i = 0; i < 30; i++) {
    float v = mq2Volts();
    if (v > 0.01f) { sv += v; nv++; }
    float o;
    if (mlxRead(0x07, &o)) { so += o; no++; }
    sf += flameMillivolts(); nf++;
    delay(200);
  }
  if (nv) vo0 = sv / nv;
  if (no >= 20) objBase = so / no;
  if (nf) flameBaseMv = (uint16_t)(sf / nf);
  baseEpoch = millis() / 1000;
  saveBaselines();
  Serial.print(F("# vo0 "));      Serial.print(vo0, 4);
  Serial.print(F(" V   objBase ")); Serial.print(objBase, 2);
  Serial.print(F(" C   flameBase ")); Serial.print(flameBaseMv);
  Serial.println(F(" mV   saved to NVS"));
  if (no < 20) Serial.println(F("# WARNING: MLX90614 gave too few clean reads"));
}

/* ------------------------------------------------------------- decision
 *
 * The node never declares an alarm. It reports WATCH when its own channels
 * are elevated and lets the Jetson fuse. That is the asymmetric policy from
 * the thesis: sensors may raise the level, never suppress a camera alarm,
 * because a fire far from the node must still alarm.
 */
static const char *nodeLevel() {
  bool gas   = okMq2  && !isnan(gasRatio) && gasRatio < GAS_RISE_ALARM;
  bool therm = okMlx  && !isnan(surfaceC) && !isnan(objBase)
                      && (surfaceC - objBase) > SURF_RISE_ALARM;
  bool ir    = okFlame && flameBaseMv
                      && (int)flameBaseMv - (int)flameMinMv > FLAME_DROP_MV;
  int n = (gas ? 1 : 0) + (therm ? 1 : 0) + (ir ? 1 : 0);
  if (n == 0) return "NORMAL";
  return "WATCH";
}

/* ---------------------------------------------------------------- emit */

/* Two output modes on purpose.
 *
 * A human at the bench cannot read a JSON line per second, and the Jetson cannot
 * parse an aligned table. So the node prints the table by default and switches to
 * JSON when something asks it to with 'j'. The Jetson sends that on connect.
 *
 * A value with no baseline prints as "----", never as 0. Printing zero for
 * "unknown" is what made the first run look like a dead gas channel when the
 * only thing missing was a baseline. */
static bool jsonMode = false;
static uint8_t rowsSinceHeader = 250;

static void humanHeader() {
  Serial.println();
  Serial.println(F("    time |  air C | surf C |  dSurf | Rs/R0 | gas % | flame mV |  drop |  p2p | level"));
  Serial.println(F("---------+--------+--------+--------+-------+-------+----------+-------+------+--------"));
  rowsSinceHeader = 0;
}

static void emitHuman() {
  if (rowsSinceHeader >= 20) humanHeader();
  rowsSinceHeader++;

  char line[160], c1[9], c2[9], c3[9], c4[8], c5[8], c6[8];
  if (okDs && !isnan(ambientC)) snprintf(c1, sizeof(c1), "%6.2f", ambientC);
  else                          snprintf(c1, sizeof(c1), "  ----");
  if (okMlx && !isnan(surfaceC)) snprintf(c2, sizeof(c2), "%6.2f", surfaceC);
  else                           snprintf(c2, sizeof(c2), "  ----");
  if (okMlx && !isnan(surfaceC) && !isnan(objBase))
       snprintf(c3, sizeof(c3), "%+6.2f", surfaceC - objBase);
  else snprintf(c3, sizeof(c3), "  ----");
  if (okMq2 && !isnan(gasRatio)) snprintf(c4, sizeof(c4), "%5.3f", gasRatio);
  else                           snprintf(c4, sizeof(c4), " ----");
  if (okMq2 && !isnan(gasRatio)) snprintf(c5, sizeof(c5), "%5.1f", 100.0f * (1.0f - gasRatio));
  else                           snprintf(c5, sizeof(c5), " ----");
  if (okFlame && flameBaseMv) snprintf(c6, sizeof(c6), "%5d", (int)flameBaseMv - (int)flameMinMv);
  else                        snprintf(c6, sizeof(c6), " ----");

  snprintf(line, sizeof(line), " %6.1fs | %s | %s | %s | %s | %s | %8u | %s | %4u | %s",
           millis() / 1000.0f, c1, c2, c3, c4, c5,
           (unsigned)flameMinMv, c6, (unsigned)flameP2p, nodeLevel());
  Serial.print(line);

  if (isnan(vo0) || isnan(objBase)) Serial.print(F("   <- press b, no baseline"));
  else if (!okDs || !okMlx || !okMq2 || !okFlame) Serial.print(F("   <- a channel is down, press s"));
  Serial.println();
}

static void emitJson() {
  /* gasPpm is deliberately -1. The MQ-2 datasheet defines R0 as the
   * resistance in 1000 ppm LPG, while our Vo0 is taken in clean air, so the
   * published ppm curves do not apply to this ratio. Emitting a number
   * anyway would be a fabricated unit. gasRatio is the real measurement and
   * gasRisePct is the readable form of it. */
  /* null, not 0, for anything without a baseline or a healthy channel. A
   * consumer can tell "unknown" from "measured zero"; it cannot if both are 0. */
  Serial.print(F("{\"t\":"));            Serial.print(millis());
  Serial.print(F(",\"ambientC\":"));
  if (okDs && !isnan(ambientC)) Serial.print(ambientC, 2); else Serial.print(F("null"));
  Serial.print(F(",\"surfaceC\":"));
  if (okMlx && !isnan(surfaceC)) Serial.print(surfaceC, 2); else Serial.print(F("null"));
  Serial.print(F(",\"surfaceRiseC\":"));
  if (okMlx && !isnan(surfaceC) && !isnan(objBase)) Serial.print(surfaceC - objBase, 2);
  else Serial.print(F("null"));
  Serial.print(F(",\"gasRatio\":"));
  if (okMq2 && !isnan(gasRatio)) Serial.print(gasRatio, 4); else Serial.print(F("null"));
  Serial.print(F(",\"gasRisePct\":"));
  if (okMq2 && !isnan(gasRatio)) Serial.print(100.0f * (1.0f - gasRatio), 1);
  else Serial.print(F("null"));
  Serial.print(F(",\"gasPpm\":null"));
  /* flameIr is normalised 0..1 with 1 meaning strong infrared, because that is
   * what the app's gauge expects. The sensor moves the other way, so the sign is
   * flipped here rather than in the app. */
  float irNorm = flameBaseMv
      ? constrain((float)((int)flameBaseMv - (int)flameMinMv) / 1000.0f, 0.0f, 1.0f)
      : 0.0f;
  Serial.print(F(",\"flameMv\":"));      Serial.print(flameMv);
  Serial.print(F(",\"flameMinMv\":"));   Serial.print(flameMinMv);
  Serial.print(F(",\"flameDips\":"));    Serial.print(flameDips);
  Serial.print(F(",\"flameP2pMv\":"));   Serial.print(flameP2p);
  Serial.print(F(",\"flameDropMv\":")); Serial.print(flameBaseMv ?
      (int)flameBaseMv - (int)flameMinMv : 0);
  Serial.print(F(",\"flameIr\":"));      Serial.print(irNorm, 3);
  Serial.print(F(",\"level\":\""));      Serial.print(nodeLevel());
  Serial.print(F("\",\"ok\":{\"ds\":")); Serial.print(okDs ? 1 : 0);
  Serial.print(F(",\"mlx\":"));          Serial.print(okMlx ? 1 : 0);
  Serial.print(F(",\"mq2\":"));          Serial.print(okMq2 ? 1 : 0);
  Serial.print(F(",\"flame\":"));        Serial.print(okFlame ? 1 : 0);
  Serial.print(F("},\"calAgeS\":"));     Serial.print(baseEpoch ? (millis() / 1000 - baseEpoch) : 0);
  Serial.println(F("}"));
}

/* ---------------------------------------------------------- boot check
 *
 * Names which channel failed, at the moment power is applied, before any
 * measurement is attempted. Four sensors were wired one at a time for exactly
 * this reason: a single "sensor error" line would send you looking at all four.
 */
static void bootCheck() {
  Serial.println(F("# --------------------------------------------------"));
  Serial.println(F("# BOOT CHECK"));

  int n = ds.getDeviceCount();
  ds.requestTemperatures();
  delay(DS_CONV_MS);
  float t = ds.getTempCByIndex(0);
  okDs = (n > 0 && t > -100.0f && t < 120.0f);
  Serial.print(F("#  DS18B20  GPIO 4    "));
  if (okDs) { Serial.print(F("PASS   ")); Serial.print(t, 2); Serial.println(F(" C")); }
  else if (n == 0) Serial.println(F("FAIL   no device. 4.7k pull-up to 3.3 V missing, or S and middle swapped."));
  else             Serial.println(F("FAIL   device seen but reading is -127. Check the pull-up."));

  Wire.beginTransmission(MLX_ADDR);
  bool addr = (Wire.endTransmission() == 0);
  float o = NAN;
  okMlx = addr && mlxRead(0x07, &o);
  Serial.print(F("#  MLX90614 21/22    "));
  if (okMlx) { Serial.print(F("PASS   ")); Serial.print(o, 2); Serial.println(F(" C object")); }
  else if (!addr) Serial.println(F("FAIL   nothing at 0x5A. SDA/SCL swapped, no 3.3 V, or ground open."));
  else            Serial.println(F("FAIL   answers but the checksum is bad. Reseat SDA and SCL."));

  uint16_t fmv = flameMillivolts();
  okFlame = (fmv > 0 && fmv < 3200);
  Serial.print(F("#  IR flame GPIO 35   "));
  if (okFlame) { Serial.print(F("PASS   ")); Serial.print(fmv); Serial.println(F(" mV")); }
  else Serial.println(F("FAIL   pinned or dead. Pin order is AO / G / + / DO, supply is THIRD."));

  float v = mq2Volts();
  okMq2 = (v > 0.02f && v < MQ2_VCC);
  Serial.print(F("#  MQ-2    GPIO 34   "));
  if (okMq2) {
    Serial.print(F("PASS   ")); Serial.print(v, 4);
    Serial.print(F(" V at AO, ")); Serial.print(v / MQ2_GAIN, 3);
    Serial.println(F(" V at the pin"));
  } else if (v <= 0.02f) {
    Serial.println(F("FAIL   near zero. AO open, no 5 V, or the divider node is shorted to GND."));
  } else {
    Serial.println(F("FAIL   at the ceiling. Divider not built, or GPIO 34 is on AO directly."));
    Serial.println(F("#           DISCONNECT GPIO 34 NOW - on 5 V that pin is exposed."));
  }

  int pass = (okDs ? 1 : 0) + (okMlx ? 1 : 0) + (okFlame ? 1 : 0) + (okMq2 ? 1 : 0);
  Serial.print(F("#  ")); Serial.print(pass); Serial.println(F(" / 4 channels alive"));
  if (pass < 4)
    Serial.println(F("#  Fix the failing channel before trusting anything below."));
  Serial.println(F("# --------------------------------------------------"));
}

static void status() {
  Serial.println(F("# --------------------------------------------------"));
  Serial.print(F("# DS18B20  ")); Serial.print(okDs ? F("ok  ") : F("FAIL"));
  Serial.print(F("  ambient ")); Serial.print(ambientC, 2); Serial.println(F(" C"));
  Serial.print(F("# MLX90614 ")); Serial.print(okMlx ? F("ok  ") : F("FAIL"));
  Serial.print(F("  surface ")); Serial.print(surfaceC, 2);
  Serial.print(F(" C   base ")); Serial.print(objBase, 2); Serial.println(F(" C"));
  Serial.print(F("# MQ-2     ")); Serial.print(okMq2 ? F("ok  ") : F("FAIL"));
  Serial.print(F("  Rs/R0 ")); Serial.print(gasRatio, 3);
  Serial.print(F("   Vo0 ")); Serial.print(vo0, 4); Serial.println(F(" V"));
  Serial.print(F("# flame    ")); Serial.print(okFlame ? F("ok  ") : F("FAIL"));
  Serial.print(F("  now ")); Serial.print(flameMv);
  Serial.print(F(" mV   window min ")); Serial.print(flameMinMv);
  Serial.print(F(" mV   base ")); Serial.print(flameBaseMv);
  Serial.print(F(" mV   p2p ")); Serial.print(flameP2p);
  Serial.print(F(" mV   entries ")); Serial.println(flameDips);
  Serial.print(F("# level    ")); Serial.println(nodeLevel());
  Serial.println(F("# --------------------------------------------------"));
}

/* ---------------------------------------------------------------------- */

void setup() {
  Serial.begin(115200);
  delay(600);
  Serial.println();
  Serial.println(F("# =================================================="));
  Serial.println(F("#  SAFE-Fire sensor node"));
  Serial.println(F("#  MQ-2 34 | flame 35 | DS18B20 4 | MLX 21/22"));
  Serial.println(F("# =================================================="));

  analogSetPinAttenuation(PIN_MQ2, ADC_11db);
  analogSetPinAttenuation(PIN_FLAME, ADC_11db);

  ds.begin();
  ds.setResolution(12);
  Wire.begin(PIN_SDA, PIN_SCL, I2C_HZ);

  /* The boot check is allowed to block, because nothing is being watched yet
   * and naming the broken channel is worth 800 ms. Async mode is enabled only
   * afterwards, for the loop that has to keep up with a flame flicker. */
  bootCheck();
  ds.setWaitForConversion(false);

  loadBaselines();
  if (isnan(vo0) || isnan(objBase)) {
    Serial.println(F("# no stored baselines. Press b in clean air."));
  } else {
    Serial.print(F("# baselines restored from NVS: vo0 "));
    Serial.print(vo0, 4); Serial.print(F(" V   objBase "));
    Serial.print(objBase, 2); Serial.println(F(" C"));
  }
  Serial.println(F("# human table by default. j switches to JSON for the Jetson."));
  Serial.println(F("# b baseline | s status | j json/table | c clear | h help"));

  ds.requestTemperatures();
  dsAskedAt = millis();
  dsPending = true;
}

void loop() {
  if (Serial.available()) {
    char k = Serial.read();
    if (k == 'b') { takeBaselines(); return; }
    if (k == 's') { status(); return; }
    if (k == 'c') {
      prefs.begin("safefire", false); prefs.clear(); prefs.end();
      vo0 = objBase = NAN; flameBaseMv = 0; baseEpoch = 0;
      Serial.println(F("# baselines cleared"));
      return;
    }
    if (k == 'j') {
      jsonMode = !jsonMode;
      Serial.print(F("# output mode: "));
      Serial.println(jsonMode ? F("JSON, one line per second")
                              : F("human table"));
      if (!jsonMode) rowsSinceHeader = 250;
      return;
    }
    if (k == 'h') {
      Serial.println(F("# b baseline | s status | j json/table | c clear | h help"));
      return;
    }
  }

  /* DS18B20, collected only when the conversion has had time to finish */
  if (dsPending && millis() - dsAskedAt >= DS_CONV_MS) {
    float t = ds.getTempCByIndex(0);
    okDs = (t > -100.0f && t < 120.0f);
    if (okDs) ambientC = t;
    ds.requestTemperatures();
    dsAskedAt = millis();
  }

  float o;
  okMlx = mlxRead(0x07, &o);
  if (okMlx) surfaceC = o;
  float a;
  if (mlxRead(0x06, &a)) mlxAmbC = a;

  float v = mq2Volts();
  okMq2 = (v > 0.02f && v < MQ2_VCC);
  if (okMq2) gasRatio = mq2Ratio(v);

  flameSample();
  okFlame = (flameMv > 0);

  if (millis() - lastEmit >= EMIT_MS) {
    lastEmit = millis();
    flameCloseWindow();
    if (jsonMode) emitJson(); else emitHuman();
  }
  delay(5);
}


## I — الجيتسون: القراءة والدمج والنشر

`node_reader.py` يقرأ العقدة عبر USB في خيط منفصل ولا يُبطئ حلقة الاستدلال
أبداً، ثم يطبّق **سُلَّم الإنذار غير المتماثل**:

| الكاميرا | الحسّاسات | المستوى |
|---|---|---|
| تُطلق | هادئة | **ALERT** — إنذار على الرؤية وحدها |
| تُطلق | توافق | **CONFIRMED** — أعلى أولوية |
| هادئة | مرتفعة | **WATCH** — ليس إنذاراً |
| هادئة | هادئة | NORMAL |

**عدم التماثل مقصود ومُبرَّر بالقياس:** في أربع تجارب دخان مضبوطة، أخطأت قناة
الغاز واحدة تماماً وقرأت هواءً نظيفاً لمدة 42 ثانية بينما احترق مصدر على بعد
10 سم، لأن تيار هواء الغرفة حمل العمود بعيداً عنها. سياسة تشترط موافقة
الحسّاسات كانت ستكبت ذلك الإنذار.


In [ ]:
# ==========================================================================
# JETSON/scripts/node_reader.py
# --------------------------------------------------------------------------
# RUNS ON: Jetson Nano 4GB, Python 3.6
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# Serial reader, the fusion ladder, and the app field mapping.
# ==========================================================================

# -*- coding: utf-8 -*-
"""SAFE-Fire - read the sensor node over USB serial, and fuse with vision.

Runs on the Jetson. Two jobs:

    1. keep the latest sensor sample from the ESP32, without ever blocking the
       inference loop
    2. apply the alarm ladder, which is the policy the thesis argues for and the
       Android client already encodes

THE POLICY IS ASYMMETRIC ON PURPOSE
    Sensors may RAISE the level. They may never lower one the camera raised.
    A fire on the far side of a room reaches the camera long before the plume
    reaches a point sensor: in four controlled smoke trials the gas channel
    missed one entirely, reading clean air for 42 s while a source burned 10 cm
    away, because room airflow carried the plume past it. A policy that required
    sensor agreement would have suppressed that alarm.

    So:
        camera fires, sensors quiet   -> ALERT       (alarm, on vision alone)
        camera fires, sensors agree   -> CONFIRMED   (highest priority)
        camera quiet, sensors elevated-> WATCH       (NOT an alarm)
        neither                       -> NORMAL

STALE DATA IS NOT DATA
    If the node stops talking, the last sample is discarded rather than reused.
    A silent node must not keep voting with a reading from two minutes ago, in
    either direction. Past NODE_STALE_S the node contributes nothing and its
    absence is reported.

    Usage
        python3 node_reader.py --monitor              live, from the node
        python3 node_reader.py --replay log.txt       parse a captured log
"""
from __future__ import print_function

import argparse
import json
import sys
import threading
import time

BAUD = 115200
NODE_STALE_S = 10.0
PORT_CANDIDATES = ("/dev/ttyUSB0", "/dev/ttyUSB1", "/dev/ttyACM0", "/dev/ttyACM1")

LEVELS = ("NORMAL", "WATCH", "ALERT", "CONFIRMED", "OFFLINE")


class NodeReader(object):
    """Latest-sample-wins reader for the ESP32 sensor node."""

    def __init__(self, port=None, baud=BAUD, verbose=False):
        self.port = port
        self.baud = baud
        self.verbose = verbose
        self.sample = None            # last good parsed sample
        self.sample_at = 0.0          # monotonic seconds when it arrived
        self.stats = {"lines": 0, "parsed": 0, "bad": 0, "comments": 0}
        self._ser = None
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.Lock()

    # -- port ------------------------------------------------------------

    def open(self):
        try:
            import serial
        except ImportError:
            raise RuntimeError("pyserial is missing:  pip3 install pyserial")
        last = None
        for p in ([self.port] if self.port else PORT_CANDIDATES):
            if not p:
                continue
            try:
                # DTR and RTS must be deasserted BEFORE the port opens.
                #
                # On an ESP32 DevKit those two lines drive the auto-reset
                # circuit: DTR through a transistor to EN, RTS to GPIO0.
                # pyserial asserts both by default, which pulls GPIO0 low and
                # leaves the board sitting in the bootloader, saying nothing at
                # all. Measured on this rig with probe_serial.py:
                #
                #     dtr=0 rts=0   1553 bytes in 6 s   <- correct
                #     dtr=1 rts=0    528 bytes
                #     dtr=0 rts=1      0 bytes          <- held in bootloader
                #     dtr=1 rts=1   1228 bytes
                #
                # Setting the attributes before open() avoids emitting the
                # pulse at all, rather than resetting and hoping.
                s = serial.Serial()
                s.port = p
                s.baudrate = self.baud
                s.timeout = 1.0
                s.dtr = False
                s.rts = False
                s.open()
                self._ser = s
                self.port = p
                break
            except Exception as e:
                last = e
        if not self._ser:
            raise RuntimeError("no serial port. Tried %s. Last error: %s"
                               % (", ".join(PORT_CANDIDATES), last))
        # The node prints a human table by default. 'j' switches it to JSON.
        # Give it a moment: if the board did reset, a byte sent during the
        # reset is lost.
        time.sleep(2.0)
        self._ser.reset_input_buffer()
        self._ser.write(b"j")
        self._ser.flush()
        return self.port

    # -- parsing ---------------------------------------------------------

    def feed(self, line):
        """Consume one line. Returns the parsed sample, or None."""
        self.stats["lines"] += 1
        line = line.strip()
        if not line:
            return None
        if line.startswith("#"):
            self.stats["comments"] += 1
            if self.verbose:
                print(line)
            return None
        if not line.startswith("{"):
            # The human table, or a partial line from a mid-transmission open.
            return None
        try:
            s = json.loads(line)
        except ValueError:
            self.stats["bad"] += 1
            return None
        if "level" not in s or "ok" not in s:
            self.stats["bad"] += 1
            return None
        self.stats["parsed"] += 1
        with self._lock:
            self.sample = s
            self.sample_at = time.time()
        return s

    # -- background ------------------------------------------------------

    def start(self):
        if self._thread:
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop)
        self._thread.daemon = True
        self._thread.start()

    def _loop(self):
        buf = b""
        while not self._stop.is_set():
            try:
                chunk = self._ser.read(256)
            except Exception:
                time.sleep(0.5)
                continue
            if not chunk:
                continue
            buf += chunk
            while b"\n" in buf:
                raw, buf = buf.split(b"\n", 1)
                self.feed(raw.decode("utf-8", "replace"))

    def stop(self):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=2.0)
            self._thread = None
        if self._ser:
            try:
                self._ser.close()
            except Exception:
                pass

    # -- access ----------------------------------------------------------

    def latest(self):
        """The current sample, or None if there is none or it has gone stale."""
        with self._lock:
            s, at = self.sample, self.sample_at
        if s is None or (time.time() - at) > NODE_STALE_S:
            return None
        return s

    def age(self):
        with self._lock:
            return None if self.sample_at == 0 else time.time() - self.sample_at

    def health(self):
        """Per-channel health from the last sample, or all-unknown if stale."""
        s = self.latest()
        if not s:
            return {"ds": None, "mlx": None, "mq2": None, "flame": None}
        ok = s.get("ok") or {}
        return dict((k, bool(ok.get(k))) for k in ("ds", "mlx", "mq2", "flame"))


# ------------------------------------------------------------------- fusion

def to_app_sensors(sample, age_s=None):
    """Map a node sample onto the field names the Firestore schema expects.

    gasPpm stays null. The MQ-2 datasheet defines R0 in 1000 ppm LPG while our
    baseline is clean air, so the published ppm curves do not apply to this
    ratio and emitting a number would be a fabricated unit. gasRatio is the
    measurement; the client should render that.

    age_s is how long ago the sample arrived, from NodeReader.age(). It is a
    property of the link and not of the sample, so it has to be passed in. The
    app reads it as sampleAgeMs and treats a missing value as infinitely old,
    which meant every reading was flagged out of date while the node was
    perfectly healthy.
    """
    if not sample:
        # An absent sensors map made a dead node look like a room at 0 C with
        # no gas, because the client defaulted every missing number to zero.
        # Say "no reading" explicitly instead.
        return {"timestamp": int(time.time() * 1000), "nodeOnline": False,
                "sampleAgeMs": None, "ambientC": None, "surfaceC": None,
                "surfaceRiseC": None, "gasRatio": None, "gasRisePct": None,
                "flameIr": None, "flameDropMv": None, "flameMinMv": None,
                "nodeLevel": None, "calAgeS": None,
                "okDs": False, "okMlx": False, "okMq2": False, "okFlame": False}
    ok = sample.get("ok") or {}

    # The firmware computes millis()/1000 - baseEpoch in unsigned arithmetic. A
    # baseline captured a few seconds ahead of that clock underflows to about
    # 4.29e9 instead of going negative, which the app would render as a
    # calibration age of 136 years. Measured on the bench: calAgeS 4294967292,
    # which is 2**32 - 4. Anything past a plausible uptime is not an age.
    cal = sample.get("calAgeS")
    if cal is not None and (cal < 0 or cal > 365 * 24 * 3600):
        cal = None

    return {
        "timestamp": int(time.time() * 1000),
        "nodeOnline": True,
        # How old the reading is. Null here made the app call every sample
        # stale, since an unknown age cannot be assumed to be recent.
        "sampleAgeMs": None if age_s is None else int(age_s * 1000),
        "ambientC": sample.get("ambientC"),
        "surfaceC": sample.get("surfaceC"),
        "surfaceRiseC": sample.get("surfaceRiseC"),
        "gasRatio": sample.get("gasRatio"),
        "gasRisePct": sample.get("gasRisePct"),
        "flameIr": sample.get("flameIr"),
        "flameDropMv": sample.get("flameDropMv"),
        # The app shows the raw millivolts beside the normalised figure, and the
        # firmware has been sending this all along.
        "flameMinMv": sample.get("flameMinMv"),
        # The node's own verdict, kept as a cross check against the fusion done
        # here. The two are computed from the same thresholds, so a disagreement
        # means one of the two copies has drifted.
        "nodeLevel": sample.get("level"),
        # Seconds since the baselines were taken. The app uses it to say a
        # channel has never been baselined rather than showing a bare dash.
        "calAgeS": cal,
        "okDs": bool(ok.get("ds")),
        "okMlx": bool(ok.get("mlx")),
        "okMq2": bool(ok.get("mq2")),
        "okFlame": bool(ok.get("flame")),
    }


def fuse(vision_alarm, sample, vision_degraded=False):
    """Combine the camera decision with the node. Returns (level, rationale).

    vision_alarm     True once the temporal layer has raised an alarm
    sample           the node's latest sample, or None if silent or stale
    vision_degraded  the camera pipeline is not running
    """
    node_level = (sample or {}).get("level")
    sensors_up = node_level in ("WATCH", "ALERT", "CONFIRMED")

    if vision_degraded:
        level = "OFFLINE"
    elif vision_alarm and sensors_up:
        level = "CONFIRMED"
    elif vision_alarm:
        level = "ALERT"
    elif sensors_up:
        level = "WATCH"
    else:
        level = "NORMAL"

    if sample is None:
        said_s = "No sensor sample: the node is silent or its data is stale."
    else:
        bits = []
        if sample.get("surfaceRiseC") is not None:
            bits.append("surface %+.2f C above baseline" % sample["surfaceRiseC"])
        if sample.get("gasRatio") is not None:
            bits.append("Rs/R0 %.3f" % sample["gasRatio"])
        if sample.get("flameDropMv") is not None:
            bits.append("infrared drop %d mV" % sample["flameDropMv"])
        said_s = ("; ".join(bits) + ".") if bits else "Node reporting, no channel elevated."

    said_v = ("Detection sustained through the persistence rule."
              if vision_alarm else
              "No detection above the calibrated threshold." if not vision_degraded else
              "Camera pipeline is not running.")

    conclusion = {
        "NORMAL": "Nothing above threshold on either channel.",
        "WATCH": ("Sensors elevated while the camera is clear. Not an alarm: "
                  "sensors alone never raise one."),
        "ALERT": ("Camera detected fire. This alarms on its own, so a fire far "
                  "from the sensor node is not missed."),
        "CONFIRMED": "Both independent channels agree. Highest priority.",
        "OFFLINE": "Detection is degraded; the sensor channel is watching alone.",
    }[level]

    return level, {
        "visionSaid": said_v,
        "sensorsSaid": said_s,
        "conclusion": conclusion,
        "visionContributed": bool(vision_alarm),
        "sensorsContributed": bool(sensors_up),
    }


# -------------------------------------------------------------------- modes

def _show(sample, level, why, age):
    ok = sample.get("ok", {})
    print("  %-9s  air %-6s surf %-6s dSurf %-6s Rs/R0 %-6s drop %-5s "
          "p2p %-5s ok %d%d%d%d  age %.1fs"
          % (level,
             sample.get("ambientC"), sample.get("surfaceC"),
             sample.get("surfaceRiseC"), sample.get("gasRatio"),
             sample.get("flameDropMv"), sample.get("flameP2pMv"),
             ok.get("ds", 0), ok.get("mlx", 0), ok.get("mq2", 0), ok.get("flame", 0),
             age or 0.0))


def run_monitor(a):
    r = NodeReader(port=a.port, verbose=True)
    print("opening serial ...")
    print("port:", r.open())
    r.start()
    print("switched the node to JSON. Ctrl-C to stop.")
    print()
    last_seen = None
    try:
        while True:
            s = r.latest()
            if s is None:
                age = r.age()
                if age is None:
                    print("  waiting for the first sample ...")
                else:
                    print("  NODE STALE, last sample %.1f s ago" % age)
            elif s is not last_seen:
                last_seen = s
                level, why = fuse(False, s)
                _show(s, level, why, r.age())
            time.sleep(1.0)
    except KeyboardInterrupt:
        pass
    finally:
        r.stop()
        print()
        print("lines %d  parsed %d  comments %d  bad %d"
              % (r.stats["lines"], r.stats["parsed"],
                 r.stats["comments"], r.stats["bad"]))


def run_replay(a):
    """Parse a captured log. Verifies the parser with no hardware attached."""
    r = NodeReader()
    src = sys.stdin if a.replay == "-" else open(a.replay)
    n = 0
    for line in src:
        s = r.feed(line)
        if s:
            n += 1
            for va in (False, True):
                level, why = fuse(va, s)
                print("  vision=%-5s -> %-9s  %s"
                      % (va, level, why["conclusion"][:58]))
            if n >= a.limit:
                break
    print()
    print("lines %d  parsed %d  comments %d  bad %d"
          % (r.stats["lines"], r.stats["parsed"],
             r.stats["comments"], r.stats["bad"]))
    if r.stats["parsed"] == 0:
        print("nothing parsed. Was the node in JSON mode? Press j, or let this")
        print("script send it on connect.")


def run_send(a):
    """Send one command key and print what the node says back.

    Baselining is deliberate, never automatic. It must happen in clean air with
    nothing hot in the field, and doing it on a schedule would eventually
    rebaseline in a room that is already smoky.
    """
    r = NodeReader(port=a.port, verbose=True)
    print("port:", r.open())
    time.sleep(0.5)
    r._ser.reset_input_buffer()
    r._ser.write(a.send.encode()[:1])
    r._ser.flush()
    print("sent %r, listening 12 s" % a.send[:1])
    print()
    t0 = time.time()
    buf = b""
    while time.time() - t0 < 12.0:
        chunk = r._ser.read(256)
        if chunk:
            buf += chunk
    r.stop()
    for line in buf.decode("utf-8", "replace").splitlines():
        if line.strip():
            print("  " + line.rstrip())


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="SAFE-Fire node reader and fusion")
    ap.add_argument("--port", default=None, help="e.g. /dev/ttyUSB0")
    ap.add_argument("--limit", type=int, default=5, help="replay: samples to show")
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--monitor", action="store_true", help="live from the node")
    g.add_argument("--replay", metavar="FILE", help="parse a captured log, - for stdin")
    g.add_argument("--send", metavar="KEY",
                   help="send one command key to the node: b baseline, s status, "
                        "j json/table, c clear")
    a = ap.parse_args()
    if a.send:
        run_send(a)
    else:
        (run_monitor if a.monitor else run_replay)(a)


### النشر إلى Firestore

مكتبة قياسية فقط، بلا Firebase SDK: الجهاز يكتب عبر REST. كل شيء في مسار
الشبكة يجري على عامل بطابور بخانة واحدة، فالتعثّر يُسقط حالة قديمة بدلاً من
إسقاط إطارات.


In [ ]:
# ==========================================================================
# JETSON/scripts/publish.py
# --------------------------------------------------------------------------
# RUNS ON: Jetson Nano 4GB, Python 3.6
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# Throttled Firestore REST writer, off the hot path.
# ==========================================================================

# -*- coding: utf-8 -*-
"""SAFE-Fire - publish edge state to Firestore from the Jetson.

The piece that was missing. The Android client was verified against seeded
documents, so everything downstream of Firestore already works; nothing upstream
of it wrote anything. This module closes that gap.

WHY REST AND NOT THE SDK
    JetPack 4.6.1 ships Python 3.6. firebase-admin and google-cloud-firestore
    both want newer runtimes and pull a large dependency tree onto a 2 GB board
    that is already running a TensorRT engine. The REST API needs nothing but
    the standard library, so this file has no imports the Nano does not already
    have.

WHY WRITES ARE THROTTLED AND OFF THE HOT PATH
    The inference loop runs at 16.4 fps. Writing a document per frame would put
    a network round trip inside a 61 ms budget and would burn the free quota in
    an afternoon. State is written on a CHANGE of alarm level, or every
    STATE_PERIOD_S seconds, whichever comes first, and always from a background
    thread with a single-slot queue. If the network stalls, frames are dropped
    from the queue rather than blocking the detector. A fire detector that
    stutters because the cloud is slow is worse than one that publishes late.

HOW THE PHONE FINDS THE LAN VIDEO STREAM
    nodes/jetson already gets a heartbeat every NODE_PERIOD_S. The stream's
    address rides on it, so discovery costs no extra writes and a DHCP lease
    change repairs itself within one heartbeat instead of needing the app
    rebuilt. Not mDNS: multicast is unreliable on consumer access points that
    do client isolation, and it would add a failure mode this design does not
    have.

CREDENTIALS
    The Web API key is read from app/google-services.json, or from the
    SAFEFIRE_API_KEY environment variable. Nothing is hard-coded and nothing is
    printed. Pass --key-file on the command line if the JSON lives elsewhere.

    While the database is in test mode this key is enough to write. Once the
    rules are tightened, which must happen before deployment, this module needs
    an ID token instead; see set_id_token().
"""
from __future__ import print_function

import base64
import json
import os
import socket
import ssl
import sys
import threading
import time

try:                                     # py3
    from urllib.request import Request, urlopen
    from urllib.error import HTTPError, URLError
except ImportError:                      # py2, just in case
    from urllib2 import Request, urlopen, HTTPError, URLError

PROJECT_DEFAULT = "safefire-1"
STATE_PERIOD_S = 5.0                     # heartbeat for state/current
NODE_PERIOD_S = 10.0                     # heartbeat for nodes/jetson
HTTP_TIMEOUT_S = 8.0
FRAME_MAX_BYTES = 700 * 1024             # above this Firestore rejects the doc

LEVELS = ("NORMAL", "WATCH", "ALERT", "CONFIRMED", "OFFLINE")


# --------------------------------------------------------------- value mapping

def encode(v):
    """Python value -> Firestore typed value.

    Order matters: bool is a subclass of int in Python, so it must be tested
    first or every True becomes integerValue 1 and the app reads a number where
    it expects a boolean.
    """
    if v is None:
        return {"nullValue": None}
    if isinstance(v, bool):
        return {"booleanValue": v}
    if isinstance(v, int):
        return {"integerValue": str(v)}
    if isinstance(v, float):
        return {"doubleValue": v}
    if isinstance(v, str):
        return {"stringValue": v}
    if isinstance(v, (list, tuple)):
        return {"arrayValue": {"values": [encode(x) for x in v]}}
    if isinstance(v, dict):
        return {"mapValue": {"fields": dict((k, encode(x)) for k, x in v.items())}}
    raise TypeError("cannot encode %r" % type(v))


# ----------------------------------------------------------------- addressing

def _iface_ips():
    """Every non-loopback IPv4 address the kernel has, in interface order.

    socket.gethostbyname_ex reads /etc/hosts, which on a stock L4T image maps
    the hostname to 127.0.1.1, so it answers "loopback" for a board that is on
    Wi-Fi. Asking the kernel with SIOCGIFADDR is the version that is true.
    docker0 is skipped because it is up whenever Docker is installed and its
    address routes nowhere the phone can reach.
    """
    out = []
    try:
        import fcntl
        import struct
    except ImportError:                  # not Linux, so there is nothing to read
        return out
    try:
        with open("/proc/net/dev") as f:
            lines = f.readlines()[2:]    # two header rows
    except (IOError, OSError):
        return out
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        for line in lines:
            name = line.split(":", 1)[0].strip()
            if not name or name == "lo" or name.startswith("docker"):
                continue
            try:
                packed = fcntl.ioctl(
                    s.fileno(), 0x8915,          # SIOCGIFADDR
                    struct.pack("256s", name[:15].encode("ascii")))
                ip = socket.inet_ntoa(packed[20:24])
            except (IOError, OSError):
                continue                 # interface is down, or has no address
            if not ip.startswith("127.") and not ip.startswith("169.254."):
                out.append(ip)
    finally:
        s.close()
    return out


def lan_ip():
    """The address this board is reachable at on its own network, or None.

    Connecting a UDP socket sends no packet. It asks the kernel which interface
    would carry traffic to that destination and binds the socket to that
    interface's address, which is a read of the routing table and nothing more.
    The literal below is never contacted, so this works with the internet down,
    with DNS broken, and behind a router that drops everything outbound.

    It does need a default route, which a board on an isolated switch will not
    have. That is what the interface scan is for.
    """
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("8.8.8.8", 9))
        ip = s.getsockname()[0]
        if ip and not ip.startswith("127.") and not ip.startswith("169.254."):
            return ip
    except Exception:
        pass
    finally:
        s.close()
    found = _iface_ips()
    return found[0] if found else None


# ------------------------------------------------------------------ the client

class FirestorePublisher(object):

    def __init__(self, project=PROJECT_DEFAULT, api_key=None, key_file=None,
                 verbose=True):
        self.project = project
        self.verbose = verbose
        self.api_key = api_key or self._find_key(key_file)
        self.id_token = None
        self.base = ("https://firestore.googleapis.com/v1/projects/%s"
                     "/databases/(default)/documents" % project)
        self._ctx = ssl.create_default_context()
        self._lock = threading.Lock()
        self._pending = None                 # single slot; newest wins
        self._stop = threading.Event()
        self._worker = None
        self._last_state_write = 0.0
        self._last_level = None
        self._stream = None                  # set by set_stream, see heartbeat
        self.stats = {"writes": 0, "errors": 0, "dropped": 0,
                      "last_rtt_ms": None, "last_error": None}

    # -- credentials ------------------------------------------------------

    @staticmethod
    def _find_key(key_file=None):
        env = os.environ.get("SAFEFIRE_API_KEY")
        if env:
            return env
        candidates = [key_file] if key_file else []
        candidates += [
            os.path.expanduser("~/google-services.json"),
            "/home/aya5/google-services.json",
            os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "google-services.json"),
        ]
        for p in candidates:
            if p and os.path.isfile(p):
                with open(p) as f:
                    j = json.load(f)
                try:
                    return j["client"][0]["api_key"][0]["current_key"]
                except (KeyError, IndexError):
                    pass
        raise RuntimeError(
            "no API key. Set SAFEFIRE_API_KEY, or place google-services.json "
            "next to this script or in the home directory.")

    def set_id_token(self, token):
        """Use an authenticated identity instead of the bare API key.

        Required once the Firestore rules stop allowing open writes. The token
        is a Firebase ID token; obtain one with signInAnonymously against the
        Identity Toolkit REST endpoint and refresh it before it expires.
        """
        self.id_token = token

    # -- transport --------------------------------------------------------

    def _patch(self, path, doc):
        url = "%s/%s" % (self.base, path)
        url += "?key=%s" % self.api_key
        body = json.dumps({"fields": dict((k, encode(v))
                                          for k, v in doc.items())}).encode("utf-8")
        headers = {"Content-Type": "application/json"}
        if self.id_token:
            headers["Authorization"] = "Bearer %s" % self.id_token
        req = Request(url, data=body, headers=headers)
        req.get_method = lambda: "PATCH"
        t0 = time.time()
        try:
            r = urlopen(req, timeout=HTTP_TIMEOUT_S, context=self._ctx)
            r.read()
            rtt = (time.time() - t0) * 1000.0
            self.stats["writes"] += 1
            self.stats["last_rtt_ms"] = round(rtt, 1)
            return True, rtt
        except HTTPError as e:
            detail = e.read()[:200].decode("utf-8", "replace")
            self.stats["errors"] += 1
            self.stats["last_error"] = "HTTP %s %s" % (e.code, detail)
        except (URLError, ssl.SSLError, OSError) as e:
            self.stats["errors"] += 1
            self.stats["last_error"] = str(e)
        if self.verbose:
            sys.stderr.write("publish: %s -> %s\n" % (path, self.stats["last_error"]))
        return False, (time.time() - t0) * 1000.0

    # -- documents --------------------------------------------------------

    @staticmethod
    def frame_to_b64(jpeg_bytes):
        """JPEG bytes -> base64 string, or None if it would not fit.

        Firestore caps a document at 1 MB and base64 adds a third. Returning
        None rather than truncating matters: the app falls back to placeholder
        art when frameB64 is absent, and a truncated string would instead give
        it a corrupt image to decode.
        """
        if not jpeg_bytes:
            return None
        if len(jpeg_bytes) > FRAME_MAX_BYTES:
            return None
        return base64.b64encode(jpeg_bytes).decode("ascii")

    def build_state(self, level, detections=None, sensors=None, frame_b64=None,
                    fps=None, inference_ms=None, rationale=None, since_ms=None):
        assert level in LEVELS, "level must be one of %s" % (LEVELS,)
        doc = {
            "level": level,
            "since": int(since_ms if since_ms is not None else time.time() * 1000),
            "detections": detections or [],
        }
        # "acknowledged" is deliberately NOT written here. It is owned by the
        # client: the person acknowledges an alarm on the phone. Writing False
        # on every five-second heartbeat clobbered that within seconds, so the
        # acknowledgement never stuck. Only publish_event seeds it, once.
        if frame_b64:
            doc["frameB64"] = frame_b64
        if sensors:
            doc["sensors"] = sensors
        if fps is not None:
            doc["fps"] = float(fps)
        if inference_ms is not None:
            doc["inferenceMs"] = float(inference_ms)
        doc["rationale"] = rationale or self.explain(level, detections, sensors)
        return doc

    @staticmethod
    def explain(level, detections=None, sensors=None):
        """Fill the rationale from the actual decision, not a fixed template.

        The schema notes that this is the sentence a person reads at 3 a.m. to
        decide whether to get out of bed, so it should say what was actually
        observed. A constant string here would be a lie with a timestamp on it.
        """
        dets = detections or []
        s = sensors or {}
        vision = bool(dets)
        best = max([d.get("confidence", 0.0) for d in dets]) if dets else 0.0
        labels = ", ".join(sorted(set(d.get("label", "?") for d in dets))) or "nothing"
        sensor_hit = level in ("WATCH", "CONFIRMED")

        if vision:
            said_v = "%s detected, highest confidence %.2f." % (labels, best)
        else:
            said_v = "No detection above the calibrated threshold."
        if s:
            # dict.get returns None for a key that EXISTS holding None, so the
            # default never fires and "%.1f" % None raises. Every unhealthy
            # channel is published as None, so this was a live crash waiting
            # for one dead sensor. Gas is a ratio here, never a ppm figure:
            # the datasheet R0 is defined in 1000 ppm LPG and our baseline is
            # clean air, so a ppm number would be a fabricated unit.
            def _f(key, fmt, unit):
                v = s.get(key)
                return (fmt % v) + unit if isinstance(v, (int, float)) else None
            parts = [p for p in (_f("surfaceC", "surface %.1f", " C"),
                                 _f("ambientC", "ambient %.1f", " C"),
                                 _f("gasRatio", "Rs/R0 %.3f", ""),
                                 _f("flameIr", "infrared %.2f", "")) if p]
            said_s = (", ".join(parts) + ".") if parts else \
                "A sensor sample arrived but no channel reported a value."
        else:
            said_s = "No sensor sample in this window."

        conclusion = {
            "NORMAL": "Nothing above threshold on either channel.",
            "WATCH": ("Sensors elevated while the camera is clear. Not an alarm: "
                      "sensors alone never raise one."),
            "ALERT": ("Camera detected fire. This alarms on its own, so a fire "
                      "far from the sensor node is not missed."),
            "CONFIRMED": "Both independent channels agree. Highest priority.",
            "OFFLINE": "Detection is degraded; the sensor channel is watching alone.",
        }[level]

        return {"visionSaid": said_v, "sensorsSaid": said_s,
                "conclusion": conclusion,
                "visionContributed": vision,
                "sensorsContributed": bool(s) and sensor_hit}

    # -- public API -------------------------------------------------------

    def publish_state(self, doc, force=False):
        """Write state/current, throttled.

        Writes when the level changed, or when STATE_PERIOD_S has passed, or
        when force is set. Returns True if a write was actually attempted.
        """
        now = time.time()
        changed = doc.get("level") != self._last_level
        due = (now - self._last_state_write) >= STATE_PERIOD_S
        if not (force or changed or due):
            return False
        # Stamped at write time, not at capture time, because the question the
        # app asks of this field is whether the edge is still alive. Without it
        # a level published before a shutdown stays on the phone for ever with
        # nothing to mark it as history.
        doc = dict(doc)
        doc["updatedAt"] = int(now * 1000)
        ok, _ = self._patch("state/current", doc)
        if ok:
            self._last_state_write = now
            self._last_level = doc.get("level")
        return True

    def publish_event(self, doc, event_id=None):
        """Append to events/. The id is the document id so a retry overwrites."""
        eid = event_id or ("evt_%d" % int(time.time() * 1000))
        d = dict(doc)
        d["id"] = eid
        # Firestore orderBy also filters for existence of the field, so an
        # event without "timestamp" is dropped by the query itself, before
        # any parsing. Without this line the history collection is written
        # correctly and is invisible.
        d.setdefault("timestamp", d.get("since", int(time.time() * 1000)))
        d.setdefault("acknowledged", False)   # seeded once, then client-owned
        ok, rtt = self._patch("events/%s" % eid, d)
        return eid if ok else None

    def set_stream(self, port=None, path="/stream.mjpg", up=False):
        """Advertise a LAN video stream on every heartbeat from now on.

        Call with port=None to stop advertising. Nothing is written here: the
        fields go out with the heartbeat that already runs, which is what keeps
        discovery free.
        """
        if port is None:
            self._stream = None
        else:
            self._stream = {"port": int(port), "path": path, "up": bool(up)}

    def _stream_fields(self):
        """The four fields the app reads off nodes/jetson.

        The address is re-detected on every heartbeat rather than captured at
        startup, so a new DHCP lease is corrected within NODE_PERIOD_S.

        When the address cannot be determined the two string fields are written
        as null rather than left out. A PATCH merges, so leaving them out would
        keep whatever was published last and hand the app an address that no
        longer routes anywhere, which is worse than telling it we do not know.
        """
        if not self._stream:
            return {}
        ip = lan_ip()
        return {
            "streamUrl": ("http://%s:%d%s" % (ip, self._stream["port"],
                                              self._stream["path"])) if ip else None,
            "streamPort": int(self._stream["port"]),
            "lanIp": ip,
            "streamUp": bool(self._stream["up"] and ip),
        }

    def heartbeat(self, online=True, detail="YOLO26n | TensorRT FP16"):
        doc = {
            "name": "Jetson Nano",
            "online": bool(online),
            "lastSeen": int(time.time() * 1000),
            "detail": detail,
        }
        doc.update(self._stream_fields())
        return self._patch("nodes/jetson", doc)[0]

    def publish_reading(self, sample):
        rid = "rd_%d" % int(time.time() * 1000)
        return self._patch("readings/%s" % rid, sample)[0]

    # -- background mode --------------------------------------------------

    def start_async(self):
        """Publish from a worker thread with a single-slot queue.

        The detector calls submit() and returns immediately. If a write is still
        in flight when the next state arrives, the older one is discarded: for a
        live status document the newest value is the only one that matters, and
        a backlog would make the app show stale state after a network hiccup.
        """
        if self._worker:
            return
        self._stop.clear()
        self._worker = threading.Thread(target=self._loop)
        self._worker.daemon = True
        self._worker.start()

    def submit(self, doc, frame=None, size=(640, 360), quality=75):
        """Hand a document to the worker. Optionally hand it a raw frame too.

        Encoding is deferred on purpose. Measured on the Jetson, doing the
        resize, JPEG encode and base64 inline cost about 350 ms on each of the
        13 frames that published during a 651-frame run. The median and the 99th
        percentile were untouched at 61.1 and 63.2 ms, but the MEAN rose from
        61.0 to 67.9 and throughput fell from 16.4 to 14.7 fps. A handful of
        very slow frames is exactly the shape a detector should not have, so the
        work moved here, where being slow costs nothing.
        """
        with self._lock:
            if self._pending is not None:
                self.stats["dropped"] += 1
            self._pending = (doc, frame, size, quality)

    def _loop(self):
        last_hb = 0.0
        while not self._stop.is_set():
            item = None
            with self._lock:
                if self._pending is not None:
                    item, self._pending = self._pending, None
            if item is not None:
                doc, frame, size, quality = item
                if frame is not None and "frameB64" not in doc:
                    try:
                        import cv2
                        small = cv2.resize(frame, size)
                        ok, buf = cv2.imencode(
                            ".jpg", small, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
                        if ok:
                            b64 = self.frame_to_b64(buf.tobytes())
                            if b64:
                                doc["frameB64"] = b64
                    except Exception as e:
                        self.stats["last_error"] = "encode: %s" % e
                self.publish_state(doc)
            now = time.time()
            if now - last_hb >= NODE_PERIOD_S:
                self.heartbeat(True)
                last_hb = now
            time.sleep(0.25)

    def stop(self):
        self._stop.set()
        if self._worker:
            self._worker.join(timeout=3.0)
            self._worker = None
        self.heartbeat(False)


# ------------------------------------------------------------------- self test

def _selftest(project, key_file):
    """Write one document of every kind and report the round trip time.

    Run this on the Jetson before wiring the publisher into the detector. It
    answers the only question that matters at this stage: can this board reach
    Firestore, and how long does a write take from here?
    """
    p = FirestorePublisher(project=project, key_file=key_file)
    print("project      : %s" % p.project)
    print("api key      : loaded, %d chars, not printed" % len(p.api_key))
    print()

    rtts = []
    print("heartbeat    :", "ok" if p.heartbeat(True) else "FAILED")
    if p.stats["last_rtt_ms"]:
        rtts.append(p.stats["last_rtt_ms"])

    doc = p.build_state(
        "NORMAL",
        detections=[],
        sensors={"timestamp": int(time.time() * 1000), "ambientC": 24.6,
                 "surfaceC": 26.1, "gasPpm": 98, "flameIr": 0.03},
        fps=16.4, inference_ms=52.3)
    print("state/current:", "ok" if p.publish_state(doc, force=True) else "FAILED",
          " %.0f ms" % (p.stats["last_rtt_ms"] or 0))
    rtts.append(p.stats["last_rtt_ms"] or 0)

    print("reading      :", "ok" if p.publish_reading(
        {"timestamp": int(time.time() * 1000), "ambientC": 24.6,
         "surfaceC": 26.1, "gasPpm": 98, "flameIr": 0.03}) else "FAILED")
    rtts.append(p.stats["last_rtt_ms"] or 0)

    print()
    print("writes %d   errors %d" % (p.stats["writes"], p.stats["errors"]))
    if p.stats["last_error"]:
        print("last error: %s" % p.stats["last_error"])
    if rtts:
        rtts = [r for r in rtts if r]
        print("round trip : min %.0f  mean %.0f  max %.0f ms"
              % (min(rtts), sum(rtts) / len(rtts), max(rtts)))
    print()
    print("Open the app now. The Status screen should show NORMAL and the")
    print("Devices screen should show the Jetson online.")


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser(description="SAFE-Fire Firestore publisher")
    ap.add_argument("--project", default=PROJECT_DEFAULT)
    ap.add_argument("--key-file", default=None,
                    help="path to google-services.json")
    ap.add_argument("--selftest", action="store_true")
    a = ap.parse_args()
    if a.selftest:
        _selftest(a.project, a.key_file)
    else:
        ap.print_help()


### البث المباشر على الشبكة المحلية

الكاميرا تُفتح **مرة واحدة**، بواسطة خط الأنابيب. هذا الخادم يعيد نشر الإطارات
التي التقطها الكاشف أصلاً ولا يفتح الكاميرا بنفسه. الترميز يجري على خيطه الخاص
لأن ترميز JPEG داخل حلقة الإطار كان قد رفع المتوسط من 61.0 إلى 67.9 مللي ثانية
بينما بقي الوسيط وp99 كما هما.


In [ ]:
# ==========================================================================
# JETSON/scripts/stream_server.py
# --------------------------------------------------------------------------
# RUNS ON: Jetson Nano 4GB, Python 3.6
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# MJPEG over HTTP, one encode per frame however many watch.
# ==========================================================================

# -*- coding: utf-8 -*-
"""SAFE-Fire - the live LAN view, served from frames the detector already has.

The phone shows what the camera sees, over the local network, without a second
camera handle and without taking anything from the inference budget.

WHY THIS SERVER NEVER OPENS THE CAMERA
    nvarguscamerasrc takes an Argus capture session against sensor 0, and the
    Nano's Argus daemon grants one session per sensor. A second VideoCapture on
    the same pipeline fails at isOpened(). safefire_run.py owns the camera; this
    module is handed frames that were already captured and already run through
    the detector, and republishes them. There is no code path in this file that
    opens a device.

WHY THE ENCODE IS ON A THREAD AND NOT IN THE FRAME LOOP
    Measured on this board: doing the resize, JPEG encode and base64 inline on
    the publishing frames cost about 350 ms each. Median and p99 were untouched
    at 61.1 and 63.2 ms, but the MEAN rose from 61.0 to 67.9 and throughput fell
    from 16.4 to 14.7 fps. A handful of very slow frames is exactly the shape a
    fire detector must not have, so publish.py moved that work to a worker.
    This module makes the same trade for the same reason: offer() assigns a
    reference, one encoder thread does the pixels.

ONE ENCODE PER FRAME, WHATEVER THE CLIENT COUNT
    Client threads never encode. They wait on a single JPEG slot and write the
    newest bytes they have not sent yet. Two phones cost the Jetson exactly what
    one phone costs; the second one only costs Wi-Fi.

A SLOW CLIENT IS THE CLIENT'S PROBLEM, NEVER THE DETECTOR'S
    The slot holds one frame, newest wins, so a client that falls behind skips
    what it missed instead of accumulating a backlog. Writes happen outside the
    lock on immutable bytes, the socket has a send timeout, and a client that
    trips it is dropped. MAX_CLIENTS bounds threads and memory on a 2 GB board.
    When the last client leaves, offer() returns on a single integer compare and
    the encoder blocks, so an unwatched stream costs the board nothing.

    Usage
        python3 stream_server.py                  self test, synthetic frames
        python3 stream_server.py --port 8091      same, on another port
        python3 stream_server.py --boxes          burn the boxes in as well
"""
from __future__ import print_function

import argparse
import ctypes
import http.server
import json
import os
import socket
import socketserver
import sys
import threading
import time

import cv2
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from publish import lan_ip                                   # noqa: E402

BOUNDARY = "safefire"                    # fixed by the app contract, not a taste
PATH_STREAM = "/stream.mjpg"
PATH_SNAPSHOT = "/snapshot.jpg"
PATH_HEALTH = "/health"

PORT_DEFAULT = 8090
SIZE_DEFAULT = (480, 360)                # 4:3, the camera's own shape, see below
QUALITY_DEFAULT = 70
FPS_CAP_DEFAULT = 5.0

# 25 percent of one of four cores. The encoder measures itself and slows down
# rather than exceed this, so the worst case is a slower stream and never a
# slower detector.
CPU_BUDGET_MS_PER_S = 250.0

MAX_CLIENTS = 3                          # bounds threads and sockets on 2 GB
MAX_BOXES = 8                            # the publisher caps at 8 too
CLIENT_STACK_BYTES = 512 * 1024
SOCKET_TIMEOUT_S = 10.0                  # a stalled write must raise, not park
REQUEST_TIMEOUT_S = 15.0
CLIENT_IDLE_LIMIT_S = 20.0               # no frames for this long, drop the client

# The camera is 640x480. publish.py resizes to 640x360, which squashes the
# picture vertically by 0.75 and puts every normalised box in the wrong place.
# The stream keeps 4:3 so the overlay lands on the object.


def _renice(delta):
    """Lower this thread's scheduling priority. The detector always wins.

    A positive nice value needs no privilege. setpriority with PRIO_PROCESS and
    a thread id applies to that thread alone on Linux, where a thread is a task.
    The syscall number is the aarch64 SYS_gettid; if it is wrong on this L4T the
    call fails and we simply do not renice, which is why nothing is raised.
    """
    try:
        libc = ctypes.CDLL("libc.so.6", use_errno=True)
        tid = libc.syscall(178)
        os.setpriority(os.PRIO_PROCESS, tid, delta)
    except Exception:
        pass


def _dets_header(boxes):
    """label,conf,x,y,w,h per box, semicolon separated, or None when empty.

    Sent as an optional part header so the app can draw boxes that belong to the
    frame it is looking at, instead of to whatever Firestore last wrote. An
    HTTP parser that does not know the header reads past it, so this costs a
    client that ignores it nothing.
    """
    if not boxes:
        return None
    return ";".join("%s,%.3f,%.4f,%.4f,%.4f,%.4f" % b for b in boxes)


def _draw(img, boxes):
    """Burn the boxes into the streamed copy. Off by default.

    The app overlays its own boxes from the header, so drawing here as well
    would double them. This exists for curl, VLC and the browser page, where
    there is nothing to overlay with.
    """
    h, w = img.shape[:2]
    for label, conf, x, y, bw, bh in boxes:
        x1, y1 = int(x * w), int(y * h)
        x2, y2 = int((x + bw) * w), int((y + bh) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 255), 2)
        cv2.putText(img, "%s %.2f" % (label, conf), (x1, max(12, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 255), 1, cv2.LINE_AA)


# --------------------------------------------------------------------- the hub

class FrameHub(object):
    """One latest-frame slot, one encoder, one HTTP server, many readers."""

    def __init__(self, size=SIZE_DEFAULT, quality=QUALITY_DEFAULT,
                 fps_cap=FPS_CAP_DEFAULT, port=PORT_DEFAULT, bind="",
                 net=640, pad=0, src_size=(640, 480), names=None,
                 draw_boxes=False, max_clients=MAX_CLIENTS):
        self.size = (int(size[0]), int(size[1]))
        self.quality = int(quality)
        self.fps_cap = float(fps_cap)
        self.fps_effective = float(fps_cap)
        self.port = int(port)
        self.bind = bind
        self.path = PATH_STREAM
        self.host = None
        self.net = int(net)
        self.pad = int(pad)
        self.src_w, self.src_h = int(src_size[0]), int(src_size[1])
        self.names = list(names) if names else []
        self.draw_boxes = bool(draw_boxes)
        self.max_clients = int(max_clients)

        # Read lock free by offer() on the hot path. Under the GIL that read is
        # atomic; the only consequence of the race is noticing a connect or a
        # disconnect one frame late.
        self.clients = 0

        self._lock = threading.Lock()
        self._in_cond = threading.Condition(self._lock)    # producer -> encoder
        self._out_cond = threading.Condition(self._lock)   # encoder -> clients
        self._raw = None                 # newest frame offered, by reference
        self._raw_dets = None
        self._in_seq = 0
        self._jpeg = None                # newest encoded frame, immutable bytes
        self._jpeg_hdr = None
        self._out_seq = 0
        self._enc_at = []                # completion times, for the served rate

        self._last_offer = 0.0
        self._min_period = 1.0 / max(0.5, self.fps_cap)
        self._started_at = time.time()
        self._stop = threading.Event()
        self._enc = None
        self._srv = None
        self._srv_thread = None
        self.stats = {"accepted": 0, "encoded": 0, "served": 0, "contended": 0,
                      "errors": 0, "client_errors": 0, "rejected": 0,
                      "bytes": 0, "peak_clients": 0,
                      "encode_ms_ema": None, "encode_ms_max": 0.0,
                      "last_error": None}

    # -- lifecycle --------------------------------------------------------

    def start(self):
        """Bind the port and start the two threads.

        The server is constructed first because that is the step that can fail,
        and failing before any thread exists keeps the caller's error path
        trivial: catch, report, run without a stream.
        """
        if self._srv is not None:
            return
        self._stop.clear()
        self._srv = _Server((self.bind, self.port), self)
        self.host = lan_ip()
        self._enc = threading.Thread(target=self._encode_loop)
        self._enc.daemon = True
        self._enc.start()
        self._srv_thread = threading.Thread(
            target=self._srv.serve_forever, kwargs={"poll_interval": 0.3})
        self._srv_thread.daemon = True
        self._srv_thread.start()

    def stop(self):
        """Wake everything that is waiting, then close the listening socket."""
        self._stop.set()
        with self._in_cond:
            self._in_cond.notify_all()
        with self._out_cond:
            self._out_cond.notify_all()
        if self._srv is not None:
            try:
                self._srv.shutdown()          # returns once serve_forever exits
            except Exception:
                pass
            try:
                self._srv.server_close()
            except Exception:
                pass
            self._srv = None
        for t in (self._enc, self._srv_thread):
            if t is not None:
                t.join(timeout=2.0)
        self._enc = self._srv_thread = None
        # Client threads are daemons and poll stopping() once a second, so they
        # unwind on their own rather than being joined one by one here.

    def stopping(self):
        return self._stop.is_set()

    def url(self, host=None):
        return "http://%s:%d%s" % (host or self.host or lan_ip() or "127.0.0.1",
                                   self.port, self.path)

    # -- producer side ----------------------------------------------------

    def offer(self, frame, dets=None):
        """Hand over the frame the detector just finished with.

        This is the only function called from the frame loop, and it is written
        to be boring. Nobody watching: one attribute read and a compare, then
        gone. Somebody watching: a clock read, a compare, and on the frames that
        pass the rate gate a non blocking lock acquire plus three assignments.
        No encode, no image allocation, no socket, nothing that can block.

        The frame is shared by reference, not copied, which is safe because
        nothing downstream writes into it: preprocess() copies out of it into
        the preallocated canvas, and pub.submit() already shares the same
        reference with the publisher's worker. If cap.read() turns out to
        recycle its buffer the worst case is an occasional torn frame in the
        STREAM; the detector already consumed the pixels it needed.
        """
        if self.clients <= 0:
            return False
        now = time.time()
        if now - self._last_offer < self._min_period:
            return False
        # Try, never wait. The lock is only ever held for a few assignments, so
        # a miss means the encoder is mid handover, and dropping one frame from
        # the stream is always cheaper than making the detector queue for it.
        if not self._lock.acquire(False):
            self.stats["contended"] += 1
            return False
        try:
            self._last_offer = now
            self._raw = frame
            self._raw_dets = dets
            self._in_seq += 1
            self.stats["accepted"] += 1
            # We hold the lock this condition wraps, so notify() is legal here
            # even though the acquire did not go through the with statement.
            self._in_cond.notify()
        finally:
            self._lock.release()
        return True

    # -- encoder ----------------------------------------------------------

    def _encode_loop(self):
        _renice(10)
        last = 0
        while not self._stop.is_set():
            with self._in_cond:
                while self._in_seq == last and not self._stop.is_set():
                    self._in_cond.wait(0.5)
                if self._stop.is_set():
                    return
                frame, dets, last = self._raw, self._raw_dets, self._in_seq
                self._raw = None            # do not pin a capture buffer here
                self._raw_dets = None
            if frame is None:
                continue
            boxes = self._norm_boxes(dets)
            t0 = time.time()
            try:
                small = cv2.resize(frame, self.size, interpolation=cv2.INTER_AREA)
                if self.draw_boxes and boxes:
                    _draw(small, boxes)
                ok, buf = cv2.imencode(".jpg", small,
                                       [int(cv2.IMWRITE_JPEG_QUALITY), self.quality])
                if not ok:
                    continue
                jpeg = buf.tobytes()
            except Exception as e:
                self.stats["errors"] += 1
                self.stats["last_error"] = "encode: %s" % e
                continue
            ms = (time.time() - t0) * 1000.0
            hdr = _dets_header(boxes)
            now = time.time()
            with self._out_cond:
                self._jpeg = jpeg           # bytes, immutable, safe to hand out
                self._jpeg_hdr = hdr
                self._out_seq += 1
                self.stats["encoded"] += 1
                self.stats["bytes"] = len(jpeg)
                ema = self.stats["encode_ms_ema"]
                self.stats["encode_ms_ema"] = ms if ema is None else 0.9 * ema + 0.1 * ms
                if ms > self.stats["encode_ms_max"]:
                    self.stats["encode_ms_max"] = ms
                self._enc_at.append(now)
                if len(self._enc_at) > 30:
                    del self._enc_at[0]
                self._out_cond.notify_all()
            self._apply_budget()

    def _apply_budget(self):
        """Derive the frame period from what the encode actually costs.

        The configured cap binds while encoding is cheap. If it turns out
        expensive the budget binds instead and the stream quietly drops to a
        lower rate, which is the whole point: the failure mode is a slower
        video, never a slower detector. /health reports which one is binding.
        """
        ema = self.stats["encode_ms_ema"]
        fps = self.fps_cap
        if ema:
            fps = min(fps, CPU_BUDGET_MS_PER_S / ema)
        fps = max(0.5, fps)
        self.fps_effective = fps
        self._min_period = 1.0 / fps

    def _norm_boxes(self, dets):
        """Detector rows to boxes normalised onto the streamed image.

        Same letterbox correction the publisher uses, so a box sits in the same
        place in the stream as in the cloud snapshot. The canvas is NET wide and
        the frame occupies rows [pad, pad + src_h), so x divides by NET and y
        has the pad removed before dividing by the source height.
        """
        if dets is None or len(dets) == 0:
            return []
        out = []
        for r in dets[:MAX_BOXES]:
            cls = int(r[5])
            label = self.names[cls] if cls < len(self.names) else "obj"
            out.append((label, float(r[4]),
                        float(max(0.0, r[0])) / self.net,
                        float(max(0.0, r[1] - self.pad)) / self.src_h,
                        float(r[2] - r[0]) / self.net,
                        float(r[3] - r[1]) / self.src_h))
        return out

    # -- consumer side ----------------------------------------------------

    def add_client(self):
        with self._lock:
            if self.clients >= self.max_clients:
                self.stats["rejected"] += 1
                return False
            self.clients += 1
            if self.clients > self.stats["peak_clients"]:
                self.stats["peak_clients"] = self.clients
            return True

    def remove_client(self):
        with self._lock:
            if self.clients > 0:
                self.clients -= 1

    def out_seq(self):
        with self._lock:
            return self._out_seq

    def wait_frame(self, last_seq, timeout=1.0):
        """The newest frame this caller has not sent, or None on timeout.

        Latest wins. A client that fell behind gets the current frame and never
        learns about the ones in between, which is the same policy as the
        publisher's single slot queue and is there for the same reason: a
        backlog would show the phone a picture of the past while claiming to be
        live. The bytes are immutable, so the caller writes them to its socket
        with the lock released.
        """
        with self._out_cond:
            if self._out_seq == last_seq:
                self._out_cond.wait(timeout)
            if self._out_seq == last_seq or self._jpeg is None:
                return None
            return self._out_seq, self._jpeg, self._jpeg_hdr

    def count_served(self, n):
        with self._lock:
            self.stats["served"] += n

    def health(self):
        """The document GET /health returns.

        up, fps, w, h and clients are the fields the app reads. The rest is the
        affordability instrumentation: it is how the cost of this channel gets
        reported as its own line rather than disappearing into the detector's
        numbers.
        """
        with self._lock:
            clients = self.clients
            fps = self._fps_locked()
            s = dict(self.stats)
        ema = s["encode_ms_ema"]
        return {
            "up": not self._stop.is_set(),
            "fps": round(fps, 2),
            "w": self.size[0],
            "h": self.size[1],
            "clients": clients,
            "fpsCap": round(self.fps_cap, 2),
            "fpsEffective": round(self.fps_effective, 2),
            "quality": self.quality,
            "accepted": s["accepted"],
            "encoded": s["encoded"],
            "served": s["served"],
            "rejected": s["rejected"],
            "contended": s["contended"],
            "clientErrors": s["client_errors"],
            "errors": s["errors"],
            "bytes": s["bytes"],
            "encodeMsEma": None if ema is None else round(ema, 2),
            "encodeMsMax": round(s["encode_ms_max"], 2),
            "peakClients": s["peak_clients"],
            "uptimeS": int(time.time() - self._started_at),
        }

    def _fps_locked(self):
        """Delivered rate over the last 30 encodes. Zero when nothing is flowing."""
        t = self._enc_at
        if len(t) < 2 or (time.time() - t[-1]) > 3.0:
            return 0.0
        span = t[-1] - t[0]
        return (len(t) - 1) / span if span > 0 else 0.0

    def node_fields(self, up=True):
        """The nodes/jetson fields that advertise this stream.

        publish.py re-detects the address on every heartbeat, so this is only
        the shape. Kept here so the two files agree on the field names.
        """
        ip = lan_ip()
        return {"streamUrl": self.url(ip) if ip else None,
                "streamPort": self.port,
                "lanIp": ip,
                "streamUp": bool(up and ip)}


# ------------------------------------------------------------------- transport

class _Handler(http.server.BaseHTTPRequestHandler):

    protocol_version = "HTTP/1.1"
    server_version = "SafeFireStream/1.0"
    timeout = REQUEST_TIMEOUT_S

    # BaseHTTPRequestHandler writes a line to stderr for every request. That
    # console belongs to the detector, and a phone reconnecting on a weak link
    # would scroll its output away.
    def log_message(self, fmt, *args):
        pass

    def do_GET(self):
        path = self.path.split("?", 1)[0]
        if path == PATH_HEALTH:
            self._health()
        elif path == PATH_STREAM:
            self._stream()
        elif path == PATH_SNAPSHOT:
            self._snapshot()
        elif path in ("/", "/index.html"):
            self._index()
        else:
            self._plain(404, "not found\n")

    # -- replies ----------------------------------------------------------

    def _send(self, code, ctype, body):
        self.send_response(code)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store, no-cache, private")
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(body)

    def _plain(self, code, text):
        self._send(code, "text/plain; charset=utf-8", text.encode("utf-8"))

    def _health(self):
        body = json.dumps(self.server.hub.health()).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store, no-cache, private")
        # The app fetches this from a WebView-free client, but a browser tab on
        # the laptop is the fastest way to read it, so do not make it fight CORS.
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Connection", "close")
        self.end_headers()
        self.wfile.write(body)

    def _index(self):
        """A page with one img tag. This is the whole camera-off test rig."""
        hub = self.server.hub
        html = ("<!doctype html><meta charset=utf-8>"
                "<title>SAFE-Fire stream</title>"
                "<body style='margin:0;background:#0A0E14;color:#E8EDF5;"
                "font:14px system-ui,sans-serif;text-align:center'>"
                "<p>SAFE-Fire live view, %d x %d, cap %.1f fps"
                " &middot; <a style='color:#25C685' href='%s'>health</a></p>"
                "<img src='%s' style='max-width:100%%;image-rendering:auto'>"
                "</body>" % (hub.size[0], hub.size[1], hub.fps_cap,
                             PATH_HEALTH, PATH_STREAM))
        self._send(200, "text/html; charset=utf-8", html.encode("utf-8"))

    # -- the stream -------------------------------------------------------

    def _stream(self):
        hub = self.server.hub
        if not hub.add_client():
            self._plain(503, "too many clients\n")
            return
        try:
            # A write that stalls must raise so the finally below gives the
            # client slot back. Without this a phone that walked out of range
            # would hold a slot until the process ended.
            self.connection.settimeout(SOCKET_TIMEOUT_S)
            self.connection.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        except Exception:
            pass
        try:
            self.send_response(200)
            self.send_header("Content-Type",
                             "multipart/x-mixed-replace; boundary=%s" % BOUNDARY)
            self.send_header("Cache-Control", "no-store, no-cache, private")
            self.send_header("Pragma", "no-cache")
            # No Content-Length on the response: the body ends when the socket
            # does. Every PART carries one, which is what lets the client read a
            # header then readFully(n) instead of scanning for the boundary
            # inside JPEG entropy data.
            self.send_header("Connection", "close")
            self.end_headers()
            self.close_connection = True
            last = 0
            idle = 0.0
            while not hub.stopping():
                item = hub.wait_frame(last, timeout=1.0)
                if item is None:
                    idle += 1.0
                    if idle >= CLIENT_IDLE_LIMIT_S:
                        break        # detector stopped offering; let it reconnect
                    continue
                idle = 0.0
                last, jpeg, dets = item
                head = ["--%s" % BOUNDARY,
                        "Content-Type: image/jpeg",
                        "Content-Length: %d" % len(jpeg)]
                if dets:
                    head.append("X-SafeFire-Dets: %s" % dets)
                part = ("\r\n".join(head) + "\r\n\r\n").encode("ascii")
                # One write per frame: the part is a burst of segments and a
                # split write would let the tail wait on an ACK.
                self.wfile.write(part + jpeg + b"\r\n")
                hub.count_served(1)
        except Exception:
            # A client hanging up mid frame is normal operation, not an error
            # worth printing over the detector's output.
            hub.stats["client_errors"] += 1
        finally:
            hub.remove_client()

    def _snapshot(self):
        """One fresh frame, for curl. Registers as a client so one gets encoded."""
        hub = self.server.hub
        if not hub.add_client():
            self._plain(503, "too many clients\n")
            return
        try:
            item = None
            last = hub.out_seq()
            deadline = time.time() + 3.0
            while time.time() < deadline and not hub.stopping():
                item = hub.wait_frame(last, timeout=0.5)
                if item is not None:
                    break
            if item is None:
                self._plain(503, "no frame yet\n")
                return
            self._send(200, "image/jpeg", item[1])
        except Exception:
            hub.stats["client_errors"] += 1
        finally:
            hub.remove_client()


class _Server(socketserver.ThreadingMixIn, http.server.HTTPServer):
    """Python 3.6 has no ThreadingHTTPServer; this is what 3.7 later added."""

    daemon_threads = True
    allow_reuse_address = True

    def __init__(self, addr, hub):
        self.hub = hub
        http.server.HTTPServer.__init__(self, addr, _Handler)

    def process_request(self, request, client_address):
        # 512 KB is ample for a socket writer and keeps three client threads to
        # 1.5 MB of reserved stack. stack_size is process global, so it is put
        # back at once: the publisher's TLS path and the serial reader must keep
        # the default.
        prev = None
        try:
            prev = threading.stack_size(CLIENT_STACK_BYTES)
        except (ValueError, RuntimeError):
            prev = None
        try:
            socketserver.ThreadingMixIn.process_request(self, request, client_address)
        finally:
            if prev is not None:
                try:
                    threading.stack_size(prev)
                except (ValueError, RuntimeError):
                    pass

    def handle_error(self, request, client_address):
        # Same reason as log_message: a dropped phone is not an incident.
        pass


# ------------------------------------------------------------------- self test

def _pattern(w, h, i, bg):
    """A synthetic frame with something that moves.

    A still test card cannot tell a working stream from a frozen one, and a
    frozen stream is the exact failure this server has to be checked for.
    """
    img = bg.copy()
    t = i * 0.06
    cx = int(w * (0.5 + 0.38 * np.sin(t)))
    cy = int(h * (0.5 + 0.28 * np.cos(t * 0.7)))
    cv2.circle(img, (cx, cy), 46, (40, 90, 245), -1)
    cv2.circle(img, (cx, cy), 46, (255, 255, 255), 2)
    bar = int((i * 7) % w)
    cv2.line(img, (bar, 0), (bar, h), (255, 220, 120), 3)
    cv2.putText(img, "SAFE-Fire synthetic  frame %d" % i, (14, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, time.strftime("%H:%M:%S"), (14, h - 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return img, cx, cy


def _fake_dets(cx, cy, pad, i):
    """One box around the moving blob, in the detector's canvas coordinates.

    Rows are [x1, y1, x2, y2, confidence, class], letterboxed the same way the
    real pipeline letterboxes, so the normalisation in _norm_boxes is exercised
    rather than bypassed.
    """
    return np.array([[cx - 52, cy - 52 + pad, cx + 52, cy + 52 + pad,
                      0.55 + 0.4 * abs(np.sin(i * 0.05)), 0.0]], np.float32)


def _selftest(a):
    src_w, src_h, net = 640, 480, 640
    pad = (net - src_h) // 2
    bg = np.zeros((src_h, src_w, 3), np.uint8)
    bg[:, :, 0] = np.linspace(20, 90, src_w).astype(np.uint8)     # a blue ramp
    bg[:, :, 1] = 24
    bg[:, :, 2] = 30

    hub = FrameHub(size=(a.width, a.height), quality=a.quality, fps_cap=a.fps,
                   port=a.port, net=net, pad=pad, src_size=(src_w, src_h),
                   names=["FLAME", "SMOKE"], draw_boxes=a.boxes)
    hub.start()
    host = hub.host or "127.0.0.1"
    print("SAFE-Fire stream self test. No camera is opened by this script.")
    print()
    print("  stream    %s" % hub.url())
    print("  browser   http://%s:%d/" % (host, hub.port))
    print("  health    http://%s:%d%s" % (host, hub.port, PATH_HEALTH))
    print("  snapshot  http://%s:%d%s" % (host, hub.port, PATH_SNAPSHOT))
    print()
    print("  curl -s http://127.0.0.1:%d%s" % (hub.port, PATH_HEALTH))
    print("  curl -s http://127.0.0.1:%d%s -o /tmp/s.jpg" % (hub.port, PATH_SNAPSHOT))
    print()
    print("feeding %.1f synthetic fps, serving at most %.1f. Ctrl-C to stop."
          % (a.src_fps, a.fps))
    print()

    period = 1.0 / max(0.1, a.src_fps)
    i = 0
    t_start = time.time()
    try:
        while True:
            if a.seconds and time.time() - t_start > a.seconds:
                break
            t0 = time.time()
            frame, cx, cy = _pattern(src_w, src_h, i, bg)
            dets = None if a.no_dets else _fake_dets(cx, cy, pad, i)
            hub.offer(frame, dets)
            i += 1
            if i % int(max(1.0, a.src_fps) * 10) == 0:
                hl = hub.health()
                print("  clients %d  encoded %d  %.2f fps  %s bytes  encode %s ms"
                      % (hl["clients"], hl["encoded"], hl["fps"],
                         hl["bytes"], hl["encodeMsEma"]))
            rest = period - (time.time() - t0)
            if rest > 0:
                time.sleep(rest)
    except KeyboardInterrupt:
        print("\nstopping")
    finally:
        hub.stop()

    hl = hub.health()
    print()
    print("offered %d   accepted %d   encoded %d   served %d   contended %d"
          % (i, hl["accepted"], hl["encoded"], hl["served"], hl["contended"]))
    print("encode  ema %s ms   max %s ms   last frame %s bytes"
          % (hl["encodeMsEma"], hl["encodeMsMax"], hl["bytes"]))
    if hl["encoded"] and hl["encoded"] > hl["accepted"]:
        print("MORE ENCODES THAN FRAMES ACCEPTED, which must never happen")
    return 0


if __name__ == "__main__":
    ap = argparse.ArgumentParser(
        description="SAFE-Fire MJPEG stream server, self test with no camera")
    ap.add_argument("--port", type=int, default=PORT_DEFAULT)
    ap.add_argument("--width", type=int, default=SIZE_DEFAULT[0])
    ap.add_argument("--height", type=int, default=SIZE_DEFAULT[1])
    ap.add_argument("--quality", type=int, default=QUALITY_DEFAULT)
    ap.add_argument("--fps", type=float, default=FPS_CAP_DEFAULT,
                    help="cap on frames served per second")
    ap.add_argument("--src-fps", type=float, default=16.0,
                    help="rate the fake detector offers frames at")
    ap.add_argument("--seconds", type=float, default=None)
    ap.add_argument("--boxes", action="store_true",
                    help="burn the boxes into the streamed image")
    ap.add_argument("--no-dets", action="store_true",
                    help="do not attach synthetic detections")
    sys.exit(_selftest(ap.parse_args()))


### النظام كاملاً

الكاميرا ← TensorRT ← العتبة ← طبقة الاستمرارية K-of-N ← الدمج مع العقدة ←
Firestore ← الهاتف.


In [ ]:
# ==========================================================================
# JETSON/scripts/safefire_run.py
# --------------------------------------------------------------------------
# RUNS ON: Jetson Nano 4GB, Python 3.6
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# The whole system, running.
# ==========================================================================

# -*- coding: utf-8 -*-
"""SAFE-Fire - the whole system, running.

Camera -> TensorRT -> threshold -> persistence layer -> fusion with the sensor
node -> Firestore -> phone. Every piece was measured on its own; this is the
file that runs them together.

    camera + engine     from cam_infer2.py, the version that measured 16.39 fps
    threshold           the operating point C3 measured, or a conformal one
    persistence         K-of-N with IoU association, from temporal_alarm.py
    sensor node         node_reader.py over USB serial
    fusion              the asymmetric ladder: sensors raise, never suppress
    publish             publish.py, throttled and off the hot path

WHAT IS NOT ALLOWED TO SLOW THE DETECTOR
    Nothing in the network path runs inside the frame loop. Publishing happens
    on a worker with a single-slot queue, so a stall drops stale state instead
    of dropping frames. The serial reader is a second worker. The loop itself
    does capture, preprocess, infer, threshold, track: the same work that was
    benchmarked, plus a dictionary update.

THE OPERATING POINT IS MEASURED, NOT CHOSEN
    Default is confidence 0.50 with 3-of-5 and IoU 0.30, because that exact
    configuration has a measured false-alarm rate on 32.2 minutes of negative
    video: 31.7 alarms per hour pooled, 66.2 on FIRESENSE and 3.4 on KMU.
    --alpha switches to a conformal threshold instead, which carries a
    distribution-free guarantee but a different, separately measured recall.

    Usage
        python3 safefire_run.py                    run until stopped
        python3 safefire_run.py --seconds 120      run for two minutes
        python3 safefire_run.py --no-publish       local only, no cloud writes
        python3 safefire_run.py --alpha 0.05       use the conformal threshold
"""
from __future__ import print_function

import argparse
import ctypes
import os
import sys
import time

import cv2
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from node_reader import NodeReader, fuse, to_app_sensors      # noqa: E402
from publish import FirestorePublisher                        # noqa: E402
from stream_server import FrameHub                            # noqa: E402
from notify import FcmSender, from_rationale                  # noqa: E402

ENGINE = "/home/aya5/kd6_fp16.engine"
SRC_W, SRC_H = 1640, 1232
CAP_W, CAP_H = 640, 480
NET = 640
# These strings are parsed by DetectionClass.valueOf on the Android side,
# so they must match that enum exactly: FLAME and SMOKE, upper case. They
# were "fire" and "smoke", which threw inside a runCatching and made every
# box vanish with no error anywhere.
NAMES = ["FLAME", "SMOKE"]

# Measured operating point. See C3_TEMPORAL/E3_2_two_by_two.csv.
CONF_DEFAULT = 0.50
K, N, IOU_MIN = 3, 5, 0.30

# Conformal thresholds from C2_CONFORMAL/E2_2_conformal.csv, verified on a
# disjoint 250 negatives. Recall at each is 0.7582 and 0.8819.
CONFORMAL_TAU = {0.01: 0.6641, 0.05: 0.0842}

FRAME_W, FRAME_H, JPEG_Q = 640, 360, 75

cudart = ctypes.CDLL("libcudart.so")
cudart.cudaMalloc.argtypes = [ctypes.POINTER(ctypes.c_void_p), ctypes.c_size_t]
cudart.cudaMemcpy.argtypes = [ctypes.c_void_p, ctypes.c_void_p,
                              ctypes.c_size_t, ctypes.c_int]
H2D, D2H = 1, 2


def cuda_malloc(n):
    p = ctypes.c_void_p()
    if cudart.cudaMalloc(ctypes.byref(p), n):
        raise RuntimeError("cudaMalloc")
    return p


# ------------------------------------------------------ persistence layer

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0


class Track(object):
    __slots__ = ("box", "hits", "last", "cls", "alarmed")

    def __init__(self, box, frame, cls):
        self.box, self.cls = box, cls
        self.hits = [frame]
        self.last = frame
        self.alarmed = False


class TemporalAlarm(object):
    """K-of-N persistence with spatial association.

    Persistence asks how often. The IoU chain asks whether it stayed in the
    same place. A flickering reflection satisfies the first and fails the
    second, which is why both are required. Identical logic to
    C3_TEMPORAL/temporal_alarm.py, so the measured rates carry over.
    """

    def __init__(self, k=K, n=N, iou_min=IOU_MIN, conf=CONF_DEFAULT):
        self.k, self.n, self.iou_min, self.conf = k, n, iou_min, conf
        self.tracks = []

    def update(self, idx, dets):
        dets = [d for d in dets if d[4] >= self.conf]
        used = set()
        for t in self.tracks:
            best, bi = 0.0, -1
            for j, d in enumerate(dets):
                if j in used or int(d[5]) != t.cls:
                    continue
                v = iou(t.box, d[:4])
                if v > best:
                    best, bi = v, j
            if bi >= 0 and best >= self.iou_min:
                used.add(bi)
                t.box = dets[bi][:4]
                t.hits.append(idx)
                t.last = idx
        for j, d in enumerate(dets):
            if j not in used:
                self.tracks.append(Track(d[:4], idx, int(d[5])))
        self.tracks = [t for t in self.tracks if idx - t.last <= self.n]
        fired = []
        for t in self.tracks:
            t.hits = [h for h in t.hits if h > idx - self.n]
            if not t.alarmed and len(t.hits) >= self.k:
                t.alarmed = True
                fired.append(t)
        return fired

    def active(self):
        return any(t.alarmed for t in self.tracks)


# ------------------------------------------------------------------- main

def main(a):
    conf = a.conf
    if a.alpha is not None:
        if a.alpha not in CONFORMAL_TAU:
            print("alpha must be one of %s" % sorted(CONFORMAL_TAU))
            return 2
        conf = CONFORMAL_TAU[a.alpha]
        print("conformal threshold for alpha=%.2f : %.4f" % (a.alpha, conf))
    print("threshold %.4f   rule %d-of-%d   IoU >= %.2f" % (conf, a.k, a.n, IOU_MIN))

    import tensorrt as trt
    logger = trt.Logger(trt.Logger.ERROR)
    runtime = trt.Runtime(logger)
    engine = runtime.deserialize_cuda_engine(open(a.engine, "rb").read())
    ctx = engine.create_execution_context()
    inp = np.zeros(tuple(engine.get_binding_shape(0)), np.float32)
    out = np.zeros(tuple(engine.get_binding_shape(1)), np.float32)
    d_in, d_out = cuda_malloc(inp.nbytes), cuda_malloc(out.nbytes)
    bindings = [int(d_in.value), int(d_out.value)]

    pipe = ("nvarguscamerasrc sensor-id=0 ! "
            "video/x-raw(memory:NVMM),width=%d,height=%d,framerate=30/1 ! "
            "nvvidconv ! video/x-raw,width=%d,height=%d,format=BGRx ! "
            "videoconvert ! video/x-raw,format=BGR ! "
            "appsink drop=true max-buffers=1 sync=false"
            % (SRC_W, SRC_H, CAP_W, CAP_H))
    cap = cv2.VideoCapture(pipe, cv2.CAP_GSTREAMER)
    if not cap.isOpened():
        print("CAMERA FAILED TO OPEN")
        return 1

    pad = (NET - CAP_H) // 2
    canvas = np.full((NET, NET, 3), 114, np.uint8)   # allocated once

    def preprocess(frame):
        canvas[pad:pad + CAP_H] = frame              # only the centre changes
        return cv2.dnn.blobFromImage(canvas, 1.0 / 255.0, (NET, NET),
                                     swapRB=True, crop=False)

    node = None
    if not a.no_node:
        try:
            node = NodeReader(port=a.port)
            print("node on", node.open())
            node.start()
        except Exception as e:
            print("node unavailable: %s" % e)
            print("continuing on vision alone")
            node = None

    pub = None
    if not a.no_publish:
        pub = FirestorePublisher(verbose=True)
        pub.start_async()
        print("publishing to", pub.project)

    # The phone cannot discover the Jetson on its own, so the hub's address goes
    # out through Firestore, which is the only place the two ever meet.
    hub = None
    if not a.no_stream:
        try:
            # net, pad and src_size are what let the hub put a box in the same
            # place the cloud snapshot puts it. They are the letterbox geometry,
            # not a preference.
            hub = FrameHub(port=a.stream_port, fps_cap=a.stream_fps,
                           net=NET, pad=pad, src_size=(CAP_W, CAP_H),
                           names=NAMES, draw_boxes=True)
            hub.start()
            print("stream on", hub.url())
            if pub:
                # The publisher re-detects the address on every heartbeat, so a
                # DHCP lease change repairs itself without a restart.
                pub.set_stream(port=hub.port, path=hub.path, up=True)
        except Exception as e:
            print("stream unavailable: %s" % e)
            print("continuing without live video")
            hub = None

    # Push normally comes from the Cloud Function in CLOUD/functions, which
    # keeps the service account key off this device entirely. This path exists
    # so the two can be measured against each other: it sends at the instant the
    # event is written, without waiting for the write to commit and a trigger to
    # fire. Opt in, and do not run both at once or the phone buzzes twice.
    notifier = None
    if a.notify_edge:
        notifier = FcmSender(verbose=True)
        if notifier.available():
            print("push alerts via FCM topic", notifier.topic)
        else:
            print("push alerts off: set SAFEFIRE_FCM_KEY to enable")
            notifier = None

    layer = TemporalAlarm(a.k, a.n, IOU_MIN, conf)
    t_all = []
    frames = alarms = 0
    last_level = None
    last_report = time.time()
    t_start = time.time()

    print()
    print("running. Ctrl-C to stop.")
    print()
    try:
        while True:
            if a.seconds and time.time() - t_start > a.seconds:
                break
            t0 = time.time()
            ok, frame = cap.read()
            if not ok:
                print("grab failed")
                break
            x = preprocess(frame)
            cudart.cudaMemcpy(d_in, x.ctypes.data_as(ctypes.c_void_p), inp.nbytes, H2D)
            ctx.execute_v2(bindings)
            cudart.cudaMemcpy(out.ctypes.data_as(ctypes.c_void_p), d_out, out.nbytes, D2H)
            cudart.cudaDeviceSynchronize()
            keep = out[0][out[0][:, 4] >= conf]
            fired = layer.update(frames, keep)

            # Offered before the timing is taken, deliberately. Putting it after
            # would hide the stream's cost from the very numbers used to argue
            # the system runs in real time. It is measured or it is not claimed.
            # Same boxes the publisher uses, so the video and the event agree.
            if hub is not None:
                hub.offer(frame, keep if layer.active() else None)

            t_all.append(time.time() - t0)
            frames += 1
            if fired:
                alarms += 1

            sample = node.latest() if node else None
            level, why = fuse(layer.active(), sample)

            # Publish on a change, or on the publisher's own heartbeat. Nothing
            # here waits for the network.
            if pub and (level != last_level or (time.time() - last_report) >= 5.0):
                # Publish only what the persistence layer confirmed. Using the
                # raw per-frame array let a NORMAL document carry boxes while
                # its own rationale said nothing was above threshold, which is
                # a document that contradicts itself.
                dets = []
                for r in (keep[:8] if layer.active() else []):
                    dets.append({"label": NAMES[int(r[5])] if int(r[5]) < len(NAMES) else "obj",
                                 "confidence": round(float(r[4]), 4),
                                 "x": round(float(max(0, r[0])) / NET, 4),
                                 "y": round(float(max(0, r[1] - pad)) / CAP_H, 4),
                                 "w": round(float(r[2] - r[0]) / NET, 4),
                                 "h": round(float(r[3] - r[1]) / CAP_H, 4)})
                fps = 1.0 / np.mean(t_all[-60:]) if t_all else 0.0
                doc = pub.build_state(
                    level, detections=dets,
                    sensors=to_app_sensors(sample, node.age() if node else None),
                    frame_b64=None,          # the worker encodes it, see submit()
                    fps=round(fps, 2),
                    inference_ms=round(float(np.mean(t_all[-60:]) * 1000.0), 1),
                    rationale=why)
                # Hand over the frame, not a JPEG. Encoding here cost 350 ms on
                # every publishing frame and dropped throughput by 10 percent.
                if level != last_level:
                    # Encode for the event BEFORE handing the document to the
                    # worker. publish_event used to copy the dict while the
                    # worker was still about to inject frameB64 into it, so
                    # every stored alarm showed placeholder art, and the copy
                    # raced the worker. A transition is rare; paying the
                    # encode here costs nothing measurable.
                    ev = dict(doc)
                    okj, jbuf = cv2.imencode(
                        ".jpg", cv2.resize(frame, (FRAME_W, FRAME_H)),
                        [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_Q])
                    if okj:
                        b64 = pub.frame_to_b64(jbuf.tobytes())
                        if b64:
                            ev["frameB64"] = b64
                    eid = pub.publish_event(ev)
                    # Sent on the transition only, and with the id of the
                    # document that was actually stored, so tapping the
                    # notification opens that event and not a guess at it.
                    if notifier is not None and eid:
                        title, body = from_rationale(level, why)
                        notifier.notify(level, title, body, event_id=eid)
                pub.submit(doc, frame=frame, size=(FRAME_W, FRAME_H), quality=JPEG_Q)
                if level != last_level:
                    print("  [%7.1fs] %-9s -> %s" % (time.time() - t_start, level,
                                                     why["conclusion"][:52]))
                last_level = level
                last_report = time.time()

            if time.time() - last_report < 0.001 or frames % 100 == 0:
                pass
    except KeyboardInterrupt:
        print("\nstopping")
    finally:
        cap.release()
        if hub:
            # Before the publisher stops, so the last heartbeat can say the
            # stream is down instead of leaving the phone dialling a dead port.
            if pub:
                pub.set_stream(port=hub.port, path=hub.path, up=False)
            hub.stop()
        if node:
            node.stop()
        if pub:
            pub.stop()

    if t_all:
        arr = np.sort(np.array(t_all) * 1000.0)
        print()
        print("=" * 62)
        print("  frames %d   alarms raised %d   %.1f s" % (frames, alarms,
                                                           time.time() - t_start))
        print("  end to end  mean %.1f  median %.1f  p95 %.1f  p99 %.1f ms"
              % (arr.mean(), arr[len(arr) // 2],
                 arr[int(len(arr) * 0.95)], arr[int(len(arr) * 0.99)]))
        print("  throughput  %.2f FPS" % (1000.0 / arr.mean()))
        if notifier:
            print("  push        sent %s  errors %s  last %s"
                  % (notifier.stats["sent"], notifier.stats["errors"],
                     notifier.stats["last_error"]))
        if hub:
            # Reported as its own line so the stream's cost is visible next to
            # the detector's, instead of being absorbed into it silently.
            h = hub.health()
            print("  stream      accepted %s  encoded %s  served %s  "
                  "contended %s  peak clients %s"
                  % (h["accepted"], h["encoded"], h["served"],
                     h["contended"], h["peakClients"]))
            print("  stream cost encode mean %s ms  max %.1f ms  "
                  "delivered %.2f FPS"
                  % (h["encodeMsEma"], h["encodeMsMax"], h["fps"]))
        if pub:
            print("  publisher   writes %d  errors %d  dropped %d  last rtt %s ms"
                  % (pub.stats["writes"], pub.stats["errors"],
                     pub.stats["dropped"], pub.stats["last_rtt_ms"]))
        if node:
            print("  node        parsed %d  bad %d" % (node.stats["parsed"],
                                                       node.stats["bad"]))
        print("=" * 62)
    return 0


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="SAFE-Fire full system")
    ap.add_argument("--engine", default=ENGINE)
    ap.add_argument("--conf", type=float, default=CONF_DEFAULT)
    ap.add_argument("--alpha", type=float, default=None,
                    help="use the conformal threshold for this alpha: 0.01 or 0.05")
    ap.add_argument("--k", type=int, default=K)
    ap.add_argument("--n", type=int, default=N)
    ap.add_argument("--port", default=None, help="serial port of the sensor node")
    ap.add_argument("--seconds", type=float, default=None)
    ap.add_argument("--no-node", action="store_true")
    ap.add_argument("--no-publish", action="store_true")
    ap.add_argument("--notify-edge", action="store_true",
                    help="send push from here instead of the Cloud Function, "
                         "for measuring the two paths against each other")
    ap.add_argument("--no-stream", action="store_true",
                    help="do not serve MJPEG video on the local network")
    ap.add_argument("--stream-port", type=int, default=8090)
    ap.add_argument("--stream-fps", type=float, default=5.0,
                    help="cap on frames handed to the stream, not on inference")
    sys.exit(main(ap.parse_args()))


## J — طبقة السحابة والتطبيق

### الإشعارات

الواجهة القديمة `POST /fcm/send` بمفتاح خادم **أُغلقت في 22 تموز 2024**؛ أي
شرح يستعمل `Authorization: key=` لم يعد يعمل. البديل هو HTTP v1 بتوكن OAuth2
قصير العمر.

الإرسال يجري من دالة سحابية لا من الجيتسون، لأن الإرسال من الجهاز يتطلب مفتاح
حساب خدمة مخزوناً على جهاز داخل بيت، ضمن مستودع يُنشر بمعرّف DOI. هنا لا يوجد
اعتماد خارج بنية جوجل إطلاقاً.

**المنطقة ليست تفضيلاً:** قاعدة البيانات في `eur3`، فيُنشئ Eventarc المُشغِّل
هناك. النشر الافتراضي إلى `us-central1` رُفض بخطأ تحقّق يسمّي
`locations/eur3/triggers/...`. المنطقة `europe-west1` داخل `eur3`.


In [ ]:
# ==========================================================================
# CLOUD/functions/index.js
# --------------------------------------------------------------------------
# RUNS ON: Cloud Functions for Firebase, Node.js 22
# NOT RUNNABLE IN COLAB. Reproduced here so this notebook is the whole
# record of the system, including the half that never touched an L4.
# One Firestore trigger, fanning out to the alarms topic.
# ==========================================================================

/**
 * SAFE-Fire push fan-out.
 *
 * One Firestore trigger. When the edge device writes a new alarm into
 * `events/`, this sends a message to the `alarms` topic, which every installed
 * app subscribes to on start.
 *
 * WHY THE CLOUD AND NOT THE JETSON
 *   The Jetson can send these itself, and `JETSON/scripts/notify.py` still can,
 *   but doing so needs a service account private key sitting on a device in a
 *   flat in Thi Qar, in a repository that is published under a DOI. Here the
 *   credential never exists outside Google's infrastructure: the Admin SDK is
 *   authenticated by the runtime. That is the whole argument.
 *
 *   The cost is one hop. The edge path sends at the same instant it writes the
 *   document; this path waits for the write to commit and the trigger to fire,
 *   and a cold start adds more. Both are worth measuring, which is why the edge
 *   sender was kept behind a flag rather than deleted.
 *
 * WHY DATA ONLY, WITH NO notification BLOCK
 *   A `notification` block is rendered by the system, which bypasses
 *   onMessageReceived while the app is backgrounded. The app would lose its
 *   per-level channel, PRIORITY_MAX and CATEGORY_ALARM. Data only keeps the app
 *   in charge of how a fire alarm looks.
 *
 * WHY ONLY A LEVEL AND AN ID ARE SENT
 *   The sentence the edge writes into the event is English. Composing the text
 *   here would push English onto an Arabic phone. The app builds both the title
 *   and the body from its own string resources, so the notification arrives in
 *   the reader's language. The payload carries the two facts that cannot be
 *   derived: which level, and which event to open.
 */
const {onDocumentCreated} = require("firebase-functions/v2/firestore");
const {setGlobalOptions} = require("firebase-functions/v2");
const logger = require("firebase-functions/logger");
const {initializeApp} = require("firebase-admin/app");
const {getMessaging} = require("firebase-admin/messaging");

initializeApp();

// maxInstances is a cost fuse, not a performance setting. A runaway write loop
// on the edge must not be able to bill an unbounded number of invocations.
//
// The region is not a preference. This project's Firestore database lives in
// eur3, the European multi-region, so the Eventarc trigger is created there;
// deploying the function into us-central1 left the trigger and its target on
// opposite sides of the Atlantic. europe-west1 sits inside eur3, which removes
// that hop and is also far closer to Thi Qar than Iowa is.
setGlobalOptions({maxInstances: 5, region: "europe-west1"});

/** Must match ALARM_TOPIC in SafeFireApp.kt. */
const TOPIC = "alarms";

/**
 * WATCH is sensors elevated with the camera clear. The system deliberately does
 * not call that an alarm, so it does not wake anyone at three in the morning
 * either. Only these two are pushed.
 */
const ALARM_LEVELS = new Set(["ALERT", "CONFIRMED"]);

exports.notifyOnAlarm = onDocumentCreated("events/{eventId}", async (event) => {
  const snap = event.data;
  if (!snap) {
    return;
  }

  const doc = snap.data() || {};
  const level = String(doc.level || "");

  if (!ALARM_LEVELS.has(level)) {
    logger.info("no push for level", {level, eventId: event.params.eventId});
    return;
  }

  const eventId = String(doc.id || event.params.eventId);

  try {
    const id = await getMessaging().send({
      topic: TOPIC,
      data: {
        level: level,
        eventId: eventId,
      },
      android: {
        // Without high priority the message waits for the next Doze
        // maintenance window, which for a fire alarm is the same as not
        // sending it. ttl bounds how long a stale alarm may still arrive.
        priority: "high",
        ttl: 120000,
      },
    });
    logger.info("push sent", {level, eventId, messageId: id});
  } catch (err) {
    // Logged and swallowed. Throwing would retry the whole trigger, and a
    // duplicate fire alarm on the phone is worse than a missing log line.
    logger.error("push failed", {level, eventId, error: String(err)});
  }
});


### عقد البيانات مع التطبيق

يقرأ التطبيق 43 حقلاً، ويكتبها جميعاً الجهاز. جرى التحقق حقلاً بحقل، وكشف
الفحص ثلاثة حقول كان التطبيق يقرأها ولا يكتبها أحد:

| الحقل | الأثر قبل الإصلاح |
|---|---|
| `sampleAgeMs` | `stale` صحيح دائماً، فإشعار "القراءات قديمة" دائم حتى والعقدة سليمة |
| `nodeLevel` | حكم العقدة نفسها يُرمى، فيضيع تدقيق متقاطع |
| `calAgeS` | يستحيل قول "لم تُعايَر بعد" |

كان الفيرموير يرسل الثلاثة فعلاً؛ دالة التحويل كانت تُسقطها.

**قاعدة معمارية:** كل قناة في `SensorSample` قابلة للإبطال، و`null` تعني "لم
تُقَس" لا صفراً. الشرطة `—` في الواجهة قراءة صادقة، والصفر ادعاء كاذب.


## K — القياسات على العتاد

ثلاثة أشواط على الجيتسون في 12 آب 2026، أطولها 152 ثانية:

| المقياس | القيمة |
|---|---|
| الإطارات | 2,401 في 152.1 ثانية |
| **الإنتاجية** | **16.19 إطار/ثانية** |
| زمن الطرف إلى الطرف | متوسط 61.8 · وسيط 61.1 · p95 62.1 · **p99 63.3 مللي ثانية** |
| كتابات Firestore | 37، **صفر أخطاء** |
| عيّنات العقدة | 149 محلَّلة، **صفر تالفة** |
| البث | 55 مقبولة · 55 مرمّزة · **54 مُرسَلة لعميل حقيقي** · صفر تنازع قفل |
| كلفة ترميز البث | متوسط 11.74 · أقصى 17.9 مللي ثانية، على خيطه الخاص |

الأشواط الثلاثة: 15.33 و16.03 و16.19 إطار/ثانية — متسقة مع 15.94 المسجّلة
سابقاً.

**الاستهلاك يُبلَّغ في سطر مستقل عمداً.** لو دُمجت كلفة البث في أرقام الكاشف
لاختفت، والادّعاء بأن النظام يعمل في الزمن الحقيقي يجب أن يُقاس شاملاً لا
منتقى.
